# PC-Free-T4

Escritorio XFCE en Colab con GPU Tesla T4. Transmisión: Selkies 2.1, H.264/NVENC, WebSockets y túnel Cloudflare HTTP/2. Resolución automática desde 960×540, 60 FPS y CBR; cola de 120 fragmentos.

En el escritorio:

- **Chrome** como navegador predeterminado.
- **Descargar aplicación** — descarga e instala `.deb`, `.AppImage`, `.tar`, `.tar.gz` y `.rpm`.
- **ProtonUp-Qt** — descarga versiones de Proton para Steam.
- **Instalar DEB** — instala un `.deb` de Descargas.

Orden de ejecución: región, núcleo, preparación, inicio. Después: prioridades, captura, entorno de juegos, Chrome y celdas opcionales de Steam, Heroic, Proton, GE-Proton y pruebas.

Cliente: botón **Jugar**; salir de la captura con Escape sostenido. En Steam: `pcfree-nvidia-run %command%`.

Política de Colab: un escritorio remoto en el runtime gratuito sin saldo no está permitido. La celda de inicio incluye la confirmación.


In [ ]:


# @title Comprobar región de la VM antes de iniciar (México)
import json, urllib.request

PAISES_ACEPTADOS = ['US', 'MX', 'CA']

with urllib.request.urlopen('https://ipinfo.io/json', timeout=10) as r:
    info = json.load(r)
pais = info.get('country')
print('VM de Colab:', info.get('city'), '|', info.get('region'), '|', pais)

if pais not in PAISES_ACEPTADOS:
    raise RuntimeError(
        f'La VM está en {pais}, fuera de la región esperada. Cambiar en Entorno de ejecución → '
        'Desconectar y eliminar entorno, reconectar la T4 y repetir la comprobación.'
    )
print('Región aceptable [OK].')

In [ ]:
# @title Cargar núcleo interno PC-Free-T4 (ejecutar antes de iniciar)
# Carga las definiciones internas; no arranca servicios.
print("PC-Free-T4: iniciando carga de fuentes internas…", flush=True)
SOURCES = {'selkies-desktop.py': '"""Standalone Ubuntu 24.04/T4 desktop. Importing this module performs no setup.\n\nOnly bootstrap() installs/starts services. No notebook globals, Drive, keepalive,\ndriver installation, or automatic app windows. Logs stay on the ephemeral VM.\n"""\nimport base64\nimport contextlib\nimport fcntl\nimport hashlib\nimport json\nimport os\nimport platform\nimport pwd\nimport re\nimport secrets\nimport shlex\nimport shutil\nimport subprocess\nimport tarfile\nimport threading\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nfrom pathlib import Path\n\nBASE = Path(\'/tmp/pcfree-selkies\')\nUSER = \'pcfree-browser\'\nDISPLAY = \':98\'\nPORT = 8082\nSELKIES_VERSION = \'2.1.0\'\nSELKIES_DEB_VERSION = \'2.1.0-1~ubuntu24.04\'\nSELKIES_DEB_SHA256 = \'cedba7a9d8bb4d07fd2b2139a518dc2c03452e0f0254c5bb088426623e2027f3\'\nSELKIES_URL = (\'https://github.com/selkies-project/selkies/releases/download/\'\n               f\'{SELKIES_VERSION}/selkies-{SELKIES_VERSION}-ubuntu24.04-amd64.deb\')\nCLOUDFLARED_VERSION = \'2026.10.3\'\nCLOUDFLARED_SHA256 = \'053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0\'\nCLOUDFLARED_URL = (\'https://github.com/cloudflare/cloudflared/releases/download/\'\n                   f\'{CLOUDFLARED_VERSION}/cloudflared-linux-amd64\')\nCHROME_VERSION = \'156.0.8078.47-1\'\nCHROME_SHA256 = \'6c92cd22e90c6e3e255d782991543fa2091db8614e8536a971a7433a5c4d7ac8\'\nCHROME_URL = (\'https://dl.google.com/linux/chrome/deb/pool/main/g/google-chrome-stable/\'\n              f\'google-chrome-stable_{CHROME_VERSION}_amd64.deb\')\nPACKAGES = [\'xserver-xorg-core\', \'xfce4-session\', \'xfwm4\', \'xfce4-panel\',\n            \'xfdesktop4\', \'xfce4-settings\', \'thunar\', \'xfce4-terminal\',\n            \'dbus-x11\', \'mesa-utils\', \'vulkan-tools\', \'x11-utils\', \'x11-xserver-utils\', \'xauth\',\n            \'pulseaudio\', \'pulseaudio-utils\', \'ca-certificates\', \'python3.12\',\n            \'libgtk-3-0t64\', \'libdbus-glib-1-2\', \'libasound2t64\', \'zenity\', \'sudo\']\nOPTIONAL_PACKAGES = [\n    \'gstreamer1.0-libav\', \'gstreamer1.0-plugins-good\',\n    \'gstreamer1.0-plugins-bad\', \'gstreamer1.0-plugins-ugly\',\n    \'libavcodec-extra\', \'libva2\', \'libva-drm2\', \'libva-x11-2\', \'libvdpau1\',\n    \'libopenal1\', \'libsdl2-2.0-0\',\n    \'xdg-utils\', \'desktop-file-utils\', \'shared-mime-info\', \'fonts-liberation\', \'wget\',\n]\nTEST_PACKAGES = [\'gcc-multilib\', \'libc6-dev-i386\', \'libx11-dev\', \'libgl1-mesa-dev\',\n                 \'libx11-dev:i386\', \'libgl1-mesa-dev:i386\']\nI386_LIBRARIES = [\n    \'libgl1:i386\', \'libglx0:i386\', \'libglvnd0:i386\', \'libegl1:i386\',\n    \'libgles2:i386\', \'libvulkan1:i386\', \'libdrm2:i386\', \'libgbm1:i386\',\n    \'libx11-6:i386\', \'libx11-xcb1:i386\', \'libxext6:i386\', \'libxrandr2:i386\',\n    \'libxinerama1:i386\', \'libxcursor1:i386\', \'libxi6:i386\', \'libxss1:i386\',\n    \'libxtst6:i386\', \'libxfixes3:i386\', \'libxrender1:i386\', \'libxcb1:i386\',\n    \'libxcb-render0:i386\', \'libxcb-shm0:i386\', \'libxcb-dri3-0:i386\',\n    \'libxkbcommon0:i386\', \'libxkbcommon-x11-0:i386\', \'libwayland-client0:i386\',\n    \'libudev1:i386\', \'libdbus-1-3:i386\', \'libfontconfig1:i386\',\n    \'libfreetype6:i386\', \'libasound2t64:i386\', \'libpulse0:i386\',\n    \'libva2:i386\', \'libva-drm2:i386\', \'libva-x11-2:i386\', \'libvdpau1:i386\',\n    \'libopenal1:i386\', \'libsdl2-2.0-0:i386\', \'libstdc++6:i386\',\n]\nAUDIO_SINK_ARGS = [\'sink_name=output\', \'rate=48000\', \'channels=2\',\n                   \'channel_map=front-left,front-right\', \'format=s16le\',\n                   \'sink_properties=device.description=PC-Free\']\n\n\ndef audio_daemon_conf():\n    """Stable 48 kHz pipeline: no flat volumes, cheap resampling, fewer wakeups.\n\n    Without rtkit (Colab has none) realtime scheduling always fails; a negative\n    nice applied before runuser does work. Larger fragments absorb NVENC bursts\n    instead of producing xruns, which the client hears as crackle or freezes.\n    """\n    return \'\'\'# PC-Free: audio estable bajo carga de NVENC\ndefault-sample-format = s16le\ndefault-sample-rate = 48000\nalternate-sample-rate = 48000\nresample-method = speex-float-1\nenable-lfe-remixing = no\nflat-volumes = no\nexit-idle-time = -1\nhigh-priority = yes\nnice-level = -10\nrealtime-scheduling = no\ndefault-fragments = 4\ndefault-fragment-size-msec = 40\n\'\'\'\n\n\ndef private_json(path, value):\n    """Atomic, root-private state; no secrets in session.json."""\n    temporary = path.with_suffix(\'.tmp\')\n    fd = os.open(temporary, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)\n    with os.fdopen(fd, \'w\') as stream:\n        json.dump(value, stream, indent=2)\n    temporary.chmod(0o600)\n    temporary.replace(path)\n\n\ndef capture(argv, env=None, timeout=60):\n    return subprocess.check_output(argv, env=env, text=True, stderr=subprocess.STDOUT,\n                                   timeout=timeout)\n\n\ndef logged(argv, name=\'install\', env=None, timeout=600):\n    path = BASE / (name + \'.log\')\n    with path.open(\'a\') as stream:\n        result = subprocess.run(argv, env=env, stdout=stream, stderr=subprocess.STDOUT,\n                                timeout=timeout)\n    if result.returncode:\n        raise RuntimeError(f\'Falló {argv[0]}; consultar {path}.\\n\' + path.read_text()[-1800:])\n\n\ndef download(url, destination):\n    """HTTPS official publisher URLs; publish only a completed download."""\n    if urllib.parse.urlparse(url).scheme != \'https\':\n        raise ValueError(\'Se requiere HTTPS.\')\n    temporary = destination.with_suffix(destination.suffix + \'.part\')\n    try:\n        with urllib.request.urlopen(url, timeout=90) as response, temporary.open(\'wb\') as out:\n            if urllib.parse.urlparse(response.url).scheme != \'https\':\n                raise RuntimeError(\'Redirección insegura.\')\n            shutil.copyfileobj(response, out)\n        temporary.replace(destination)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef require_sha256(path, expected):\n    """Fail closed when the official digest does not match."""\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    actual = digest.hexdigest()\n    if actual.lower() != expected.lower():\n        Path(path).unlink(missing_ok=True)\n        raise RuntimeError(f\'SHA-256 inesperado para {Path(path).name}: {actual}; \'\n                           \'descarga eliminada.\')\n\n\n@contextlib.contextmanager\ndef phase(name):\n    start = time.monotonic()\n    print(name + \'…\', flush=True)\n    yield\n    print(f\'  {time.monotonic() - start:.1f} s\', flush=True)\n\n\ndef wait_for(test, description, processes=(), timeout=45):\n    deadline = time.monotonic() + timeout\n    while time.monotonic() < deadline:\n        if any(process.poll() is not None for process in processes):\n            raise RuntimeError(f\'{description}: proceso detenido; registros en {BASE}.\')\n        try:\n            if test():\n                return\n        except (OSError, subprocess.SubprocessError, urllib.error.URLError):\n            pass\n        time.sleep(0.25)  # Readiness polling, never a session keepalive.\n    raise RuntimeError(f\'Tiempo agotado: {description}; registros en {BASE}.\')\n\n\ndef pid_identity(pid):\n    try:\n        # comm can contain spaces: fields after the closing parenthesis start at #3.\n        return Path(f\'/proc/{pid}/stat\').read_text().rsplit(\')\', 1)[1].split()[19]\n    except (OSError, IndexError):\n        return None\n\n\ndef already_running(session):\n    entries = session.get(\'processes\', {})\n    return bool(entries) and all(pid_identity(item[\'pid\']) == item[\'start\']\n                                 and item[\'start\'] is not None for item in entries.values())\n\n\ndef preflight():\n    if os.geteuid() != 0:\n        raise RuntimeError(\'Requiere una VM Ubuntu administrada con permisos root.\')\n    release = dict(line.split(\'=\', 1) for line in Path(\'/etc/os-release\').read_text().splitlines()\n                   if \'=\' in line)\n    if release.get(\'ID\', \'\').strip(\'"\') != \'ubuntu\' or release.get(\'VERSION_ID\', \'\').strip(\'"\') != \'24.04\':\n        raise RuntimeError(\'Esta receta requiere Ubuntu 24.04; no forzar otros paquetes.\')\n    if platform.machine() != \'x86_64\':\n        raise RuntimeError(\'Se requiere amd64.\')\n    if os.path.ismount(\'/content/drive\'):\n        raise RuntimeError(\'Desmontar Drive antes de preparar el escritorio.\')\n    if shutil.disk_usage(\'/tmp\').free < 8 * 1024**3:\n        raise RuntimeError(\'Se requieren al menos 8 GiB libres; los juegos necesitan más.\')\n    gpu_rows = [row.strip() for row in capture(\n        [\'nvidia-smi\', \'--query-gpu=name,driver_version,pci.bus_id\', \'--format=csv,noheader\']\n    ).splitlines() if row.strip()]\n    # Un runtime puede exponer más de una T4. El escritorio necesita una GPU concreta;\n    # se selecciona la primera T4 de nvidia-smi para el BusID de Xorg/Selkies.\n    t4_rows = [row for row in gpu_rows if \'T4\' in row.split(\',\', 1)[0]]\n    if not t4_rows:\n        raise RuntimeError(\'Se requiere al menos una GPU NVIDIA Tesla T4 visible por nvidia-smi.\')\n    name, driver, bus = (value.strip() for value in t4_rows[0].split(\',\', 2))\n    if not re.fullmatch(r\'\\d+\\.\\d+(?:\\.\\d+)?\', driver):\n        raise RuntimeError(\'GPU/versión NVIDIA no compatible.\')\n    if len(t4_rows) > 1:\n        print(f\'Detectadas {len(t4_rows)} Tesla T4; Xorg/Selkies usará solo la primera: {name} ({bus}).\', flush=True)\n        print(\'Las GPU no se combinan: la segunda no duplica los FPS de un juego.\', flush=True)\n    if Path(\'/tmp/.X11-unix/X98\').exists() or Path(\'/tmp/.X98-lock\').exists():\n        raise RuntimeError(\'Ya existe Xorg :98. No se reinicia ni se reinstala la sesión activa.\')\n    # Colab asigna vCPU/RAM al runtime; la receta no limita la afinidad de CPU.\n    try:\n        cpu_count = os.cpu_count() or 0\n        mem_kib = next((int(line.split()[1]) for line in Path(\'/proc/meminfo\').read_text().splitlines()\n                        if line.startswith(\'MemTotal:\')), 0)\n        print(f\'Recursos visibles: {cpu_count} CPU lógicas | RAM {mem_kib / 1024**2:.1f} GiB\', flush=True)\n    except Exception as exc:\n        print(f\'Aviso: no se pudieron leer todos los recursos CPU/RAM: {exc}\', flush=True)\n    print(f\'GPU: {name} | controlador del kernel: {driver}\', flush=True)\n    return driver, bus\n\n\ndef safe_apt_install(packages, validated_local_deb=None):\n    """Do not permit apt dependency resolution to replace a GPU/kernel stack."""\n    local_path = None\n    local_filename = None\n    if validated_local_deb is not None:\n        # Only the exact official download explicitly passed by bootstrap. Never\n        # permit arbitrary file: origins or infer trust from a package argument.\n        local_path = Path(validated_local_deb)\n        if (local_path != BASE / \'selkies.deb\' or str(local_path) not in packages\n                or local_path.is_symlink() or not local_path.is_file()):\n            raise RuntimeError(\'DEB local no validado; operación cancelada.\')\n        metadata = dict(line.split(\': \', 1) for line in\n                        capture([\'dpkg-deb\', \'-f\', str(local_path)]).splitlines() if \': \' in line)\n        if (metadata.get(\'Package\') != \'selkies\' or metadata.get(\'Version\') != SELKIES_DEB_VERSION\n                or metadata.get(\'Architecture\') != \'amd64\'):\n            raise RuntimeError(\'Identidad del DEB Selkies inesperada; operación cancelada.\')\n        local_filename = f"selkies_{SELKIES_DEB_VERSION}_amd64.deb"\n    plan = capture([\'apt-get\', \'-s\', \'install\', \'-y\', \'--no-install-recommends\', *packages], timeout=120)\n    for line in plan.splitlines():\n        if line.startswith(\'Remv \'):\n            raise RuntimeError(f\'APT propone eliminar {line[5:].strip()}; operación cancelada.\')\n        if line.startswith(\'Inst \'):\n            name = line.split()[1].split(\':\')[0]\n            if \'nvidia\' in name or name.startswith((\'linux-image\', \'linux-modules\', \'linux-headers\')):\n                raise RuntimeError(f\'APT propone cambiar {name}; no se modifica el controlador del host.\')\n    origins = capture([\'apt-get\', \'--print-uris\', \'--download-only\', \'install\', \'-y\',\n                       \'--no-install-recommends\', *packages], timeout=120)\n    for uri, filename in re.findall(r"^\'([^\']*)\'\\s+(\\S+)", origins, re.MULTILINE):\n        parsed = urllib.parse.urlparse(uri)\n        if local_path is not None:\n            if (parsed.scheme == \'file\' and not parsed.netloc and not parsed.query\n                    and not parsed.fragment and urllib.parse.unquote(parsed.path) == str(local_path)):\n                continue\n            # Some APT versions omit the URI for an explicitly supplied local\n            # archive. Match its metadata-derived cache filename, not any \'\'.\n            if uri == \'\' and filename == local_filename:\n                continue\n        if not official_ubuntu_uri(uri):\n            raise RuntimeError(\'APT propone un origen no permitido; operación cancelada.\')\n    logged([\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', *packages],\n           env={**os.environ, \'DEBIAN_FRONTEND\': \'noninteractive\'})\n\n\ndef safe_optional_install(packages, label):\n    """Install optional extras without aborting the desktop on APT conflicts."""\n    try:\n        safe_apt_install(packages)\n        print(f\'{label}: instalados en grupo.\', flush=True)\n        return\n    except RuntimeError as exc:\n        print(f\'Aviso: {label} se omite en grupo ({exc}); se intentará por paquete.\', flush=True)\n    for package in packages:\n        try:\n            safe_apt_install([package])\n            print(f\'  [OK] {package}\', flush=True)\n        except RuntimeError as exc:\n            print(f\'  [SKIP] {package}: {exc}\', flush=True)\n\n\ndef install_compat_libraries():\n    """32-bit runtime libs for native games and Wine/Proton."""\n    logged([\'dpkg\', \'--add-architecture\', \'i386\'], \'i386-arch\', timeout=120)\n    logged([\'apt-get\', \'update\', \'-qq\'], \'i386-update\', timeout=240)\n    safe_optional_install(I386_LIBRARIES, \'Bibliotecas 32 bits\')\n    safe_optional_install(TEST_PACKAGES, \'Herramientas de prueba 32/64 bits\')\n\n\ndef install_optional_packages():\n    safe_optional_install(OPTIONAL_PACKAGES, \'Códecs, fuentes y utilidades del navegador\')\n\n\ndef matching_version(madison, driver):\n    for line in madison.splitlines():\n        columns = [part.strip() for part in line.split(\'|\')]\n        if len(columns) >= 3:\n            version = columns[1]\n            upstream = version.split(\':\')[-1].split(\'-\')[0]\n            if upstream == driver:\n                return version\n    raise RuntimeError(f\'Los repositorios oficiales no ofrecen bibliotecas NVIDIA {driver} exactas. \'\n                       \'Detenerse: no instalar otro controlador ni sustituir el del kernel.\')\n\n\ndef official_ubuntu_uri(uri):\n    """Ubuntu archives or NVIDIA\'s official CUDA APT repo for this exact OS/arch.\n\n    APT must already have authenticated the repository metadata; never add keys,\n    PPAs or trusted=yes, and never use a different driver version as fallback.\n    """\n    parsed = urllib.parse.urlparse(uri)\n    if parsed.username or parsed.password or parsed.port not in (None, 80, 443):\n        return False\n    ubuntu = (parsed.scheme in (\'http\', \'https\') and\n              (parsed.hostname in (\'archive.ubuntu.com\', \'security.ubuntu.com\', \'ports.ubuntu.com\')\n               or (parsed.hostname or \'\').endswith(\'.archive.ubuntu.com\')))\n    path = urllib.parse.unquote(parsed.path)\n    nvidia = (parsed.scheme == \'https\' and parsed.port in (None, 443) and\n              parsed.hostname == \'developer.download.nvidia.com\' and\n              path.startswith(\'/compute/cuda/repos/ubuntu2404/x86_64/\') and\n              \'..\' not in path.split(\'/\'))\n    return ubuntu or nvidia\n\n\ndef graphics(driver):\n    branch = driver.split(\'.\')[0]\n    packages = [f\'{name}-{branch}\' for name in (\'libnvidia-gl\', \'libnvidia-gpucomp\',\n                \'libnvidia-cfg1\', \'libnvidia-encode\', \'libnvidia-compute\', \'xserver-xorg-video-nvidia\')]\n    destination = BASE / \'nvidia\'; destination.mkdir(exist_ok=True)\n    # Resolve ALL versions and origins before extraction. Never install these debs.\n    selected = []\n    for package in packages:\n        version = matching_version(capture([\'apt-cache\', \'madison\', package]), driver)\n        spec = package + \'=\' + version\n        origins = capture([\'apt-get\', \'--print-uris\', \'download\', spec])\n        uris = re.findall(r"^\'([^\']+)\'", origins, re.MULTILINE)\n        if not uris or not all(official_ubuntu_uri(uri) for uri in uris):\n            raise RuntimeError(f\'{package}: origen no permitido (Ubuntu/NVIDIA oficial); operación cancelada.\')\n        selected.append(spec)\n    for spec in selected:\n        with (BASE / \'nvidia-download.log\').open(\'a\') as log:\n            subprocess.run([\'apt-get\', \'download\', spec], cwd=destination,\n                           stdout=log, stderr=subprocess.STDOUT, check=True, timeout=180)\n    root = destination / \'root\'\n    for deb in destination.glob(\'*.deb\'):\n        # dpkg-deb extraction executes no maintainer scripts and installs no driver.\n        logged([\'dpkg-deb\', \'-x\', str(deb), str(root)], \'nvidia-extract\')\n    libraries = sorted({str(path.parent) for path in root.rglob(\'*.so*\') if path.is_file()})\n    if not libraries or not list(root.rglob(\'nvidia_drv.so\')):\n        raise RuntimeError(\'Paquetes NVIDIA incompletos; revisar nvidia-download.log.\')\n    # `nvidia-smi --query-gpu=driver_version` imprime una línea por GPU.\n    # En un runtime multi-GPU, comparar todo el texto con una sola versión causa un falso\n    # positivo aunque el controlador siga igual: las dos líneas repiten la versión.\n    # Valida de nuevo cada T4 visible sin instalar ni cambiar el controlador del kernel.\n    actual_output = capture([\n        \'nvidia-smi\', \'--query-gpu=name,driver_version\', \'--format=csv,noheader\'\n    ]).strip()\n    actual_t4_versions = []\n    for line in actual_output.splitlines():\n        if not line.strip():\n            continue\n        parts = [part.strip() for part in line.split(\',\', 1)]\n        if len(parts) == 2 and \'T4\' in parts[0]:\n            actual_t4_versions.append(parts[1])\n    if not actual_t4_versions or any(version != driver for version in actual_t4_versions):\n        observed = \', \'.join(actual_t4_versions) if actual_t4_versions else \'ninguna T4 visible\'\n        raise RuntimeError(\n            f\'La versión del controlador NVIDIA no coincide: esperada {driver}; \'\n            f\'detectada en T4: {observed}. No se instaló ningún controlador del kernel.\'\n        )\n    print(\n        f\'Controlador verificado en {len(actual_t4_versions)} T4(s): {driver}; \'\n        \'las bibliotecas de la cuenta permanecen aisladas del kernel.\', flush=True\n    )\n    return root, \':\'.join(libraries)\n\n\ndef prepare_user(root, libraries):\n    try:\n        account = pwd.getpwnam(USER)\n    except KeyError:\n        logged([\'useradd\', \'--create-home\', \'--home-dir\', str(BASE / \'home\'), \'--shell\', \'/bin/bash\', USER])\n        account = pwd.getpwnam(USER)\n    if account.pw_uid == 0:\n        raise RuntimeError(\'La cuenta de escritorio no puede ser root.\')\n    import grp\n    groups = []\n    for name in (\'video\', \'render\'):\n        try:\n            grp.getgrnam(name)\n            groups.append(name)\n        except KeyError:\n            pass\n    if groups:\n        logged([\'usermod\', \'-aG\', \',\'.join(groups), USER])\n    home = Path(account.pw_dir)\n    runtime = BASE / \'runtime\'\n    for directory in (home, runtime, home / \'Desktop\', home / \'Downloads\'):\n        directory.mkdir(parents=True, exist_ok=True)\n        os.chown(directory, account.pw_uid, account.pw_gid)\n    runtime.chmod(0o700)\n    authority = BASE / \'.Xauthority\'\n    authority.touch(mode=0o600); authority.chmod(0o600)\n    logged([\'xauth\', \'-f\', str(authority), \'add\', DISPLAY, \'MIT-MAGIC-COOKIE-1\', secrets.token_hex(16)])\n    os.chown(authority, account.pw_uid, account.pw_gid)\n    pulse_config = home / \'.config/pulse\'\n    pulse_config.mkdir(parents=True, exist_ok=True)\n    cookie = pulse_config / \'cookie\'\n    if not cookie.exists():\n        with cookie.open(\'xb\') as stream:\n            stream.write(secrets.token_bytes(256))\n    cookie.chmod(0o600)\n    daemon_conf = pulse_config / \'daemon.conf\'\n    daemon_conf.write_text(audio_daemon_conf())\n    daemon_conf.chmod(0o644)\n    for item in (home / \'.config\', pulse_config, cookie, daemon_conf):\n        os.chown(item, account.pw_uid, account.pw_gid)\n    env = {**os.environ, \'DISPLAY\': DISPLAY, \'XAUTHORITY\': str(authority),\n           \'HOME\': str(home), \'USER\': USER, \'LOGNAME\': USER, \'XDG_RUNTIME_DIR\': str(runtime),\n           \'PULSE_SERVER\': f\'unix:{runtime}/pulse/native\', \'PULSE_COOKIE\': str(cookie), \'LD_LIBRARY_PATH\': libraries,\n           \'__GLX_VENDOR_LIBRARY_NAME\': \'nvidia\', \'SELKIES_INTERPOSER\': \'/usr/$LIB/selkies_input_interposer.so\'}\n    for key in (\'LD_PRELOAD\', \'DBUS_SESSION_BUS_ADDRESS\', \'SESSION_MANAGER\',\n                \'SELKIES_BASIC_AUTH_USER\', \'SELKIES_BASIC_AUTH_PASSWORD\', \'PASSWORD\', \'PASSWD\'):\n        env.pop(key, None)\n    env[\'DBUS_SESSION_BUS_ADDRESS\'] = f\'unix:path={runtime}/bus\'\n    # Persist only an explicit environment allowlist; never inherited Colab tokens.\n    keep = (\'DISPLAY\', \'XAUTHORITY\', \'HOME\', \'USER\', \'LOGNAME\', \'XDG_RUNTIME_DIR\',\n            \'PULSE_SERVER\', \'PULSE_COOKIE\', \'LD_LIBRARY_PATH\', \'__GLX_VENDOR_LIBRARY_NAME\',\n            \'SELKIES_INTERPOSER\', \'DBUS_SESSION_BUS_ADDRESS\')\n    session = dict(user=USER, home=str(home), runtime=str(runtime), display=DISPLAY,\n                   xauthority=str(authority), gpu_root=str(root),\n                   env={key: env[key] for key in keep}, run_launcher=\'/usr/local/bin/pcfree-run\')\n    return session, env\n\n\ndef vulkan_icd(root):\n    """Create a private NVIDIA ICD manifest without changing system-wide Vulkan config.\n\n    In this runtime, a direct probe found the NVIDIA EGL library exposed the T4,\n    while the GLX library path failed ICD entry-point loading. Prefer EGL when present;\n    keep the driver-provided api_version and all other manifest fields unchanged.\n    """\n    candidates = list(root.rglob(\'nvidia_icd.json\'))\n    if len(candidates) != 1:\n        raise RuntimeError(\'ICD Vulkan NVIDIA oficial no encontrado o ambiguo.\')\n    document = json.loads(candidates[0].read_text())\n    egl_library = root / \'usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0\'\n    glx_library = root / \'usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0\'\n    library = egl_library if egl_library.is_file() else glx_library\n    if not library.is_file() or not document.get(\'ICD\', {}).get(\'api_version\'):\n        raise RuntimeError(\'ICD/biblioteca NVIDIA incompletos: EGL y GLX no encontrados.\')\n    document[\'ICD\'][\'library_path\'] = str(library)\n    destination = BASE / \'nvidia-vulkan64.json\'\n    destination.write_text(json.dumps(document, indent=2) + \'\\n\')\n    destination.chmod(0o644)  # Not secret; the unprivileged Vulkan loader reads it.\n    return str(destination)\n\nSHM_PERMS_SOURCE = r\'\'\'#define _GNU_SOURCE\n#include <dlfcn.h>\n#include <sys/ipc.h>\n#include <sys/shm.h>\n/* Colab containers: Xorg cannot read SO_PEERCRED of the capture client, so MIT-SHM\n   only attaches world-accessible segments (else BadAccess, X11 error 10). */\nint shmget(key_t key, size_t size, int flags) {\n    static int (*real)(key_t, size_t, int);\n    if (!real) real = (int (*)(key_t, size_t, int))dlsym(RTLD_NEXT, "shmget");\n    if (flags & IPC_CREAT) flags |= 0666;\n    return real(key, size, flags);\n}\n\'\'\'\n\n\ndef shm_perms_library():\n    """Build shm-perms.so on every fresh VM; never trust a leftover file."""\n    library = BASE / \'shm-perms.so\'\n    source = BASE / \'shm-perms.c\'\n    source.write_text(SHM_PERMS_SOURCE)\n    if not shutil.which(\'gcc\'):\n        safe_apt_install([\'gcc\', \'libc6-dev\'])\n    library.unlink(missing_ok=True)\n    logged([\'gcc\', \'-shared\', \'-fPIC\', \'-O2\', \'-o\', str(library), str(source), \'-ldl\'])\n    library.chmod(0o755)\n    if \'shmget\' not in capture([\'nm\', \'-D\', \'--defined-only\', str(library)]):\n        raise RuntimeError(\'shm-perms.so no exporta shmget; XShm no quedaría corregido.\')\n    return str(library)\n\n\ndef as_user(argv):\n    return [\'runuser\', \'-u\', USER, \'--\', *argv]\n\n\ndef spawn(argv, name, env, processes, user=True, nice=None):\n    command = as_user(argv) if user else argv\n    if nice is not None:\n        # Lower the nice value as root before runuser drops privileges: setuid\n        # keeps it, while the desktop user could not raise its own priority.\n        command = [\'nice\', \'-n\', str(nice), *command]\n    with (BASE / (name + \'.log\')).open(\'a\') as log:\n        process = subprocess.Popen(command, env=env,\n                                   stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n    processes[name] = process\n    return process\n\n\ndef launcher_source(session):\n    # $LIB is expanded by the ELF loader per-process architecture, not by a shell.\n    # This handles 32-bit children of 64-bit Steam/Wine too, unlike probing argv[0].\n    return \'\'\'#!/usr/bin/python3.12\nimport json, os, pwd, sys\nif len(sys.argv) < 2:\n    raise SystemExit(\'Uso: pcfree-run programa [argumentos]\')\nsettings = SETTINGS\naccount = pwd.getpwnam(settings[\'user\'])\nif os.geteuid() not in (0, account.pw_uid):\n    raise SystemExit(\'Cuenta de escritorio requerido.\')\nenv = dict(os.environ)\nenv.update(settings[\'env\'])\nenv.pop(\'LD_PRELOAD\', None)\n# Ubuntu amd64 glibc $LIB: lib/x86_64-linux-gnu; i386: lib/i386-linux-gnu.\nif os.path.exists(\'/usr/lib/x86_64-linux-gnu/selkies_input_interposer.so\'):\n    env[\'LD_PRELOAD\'] = \'/usr/$LIB/selkies_input_interposer.so\'\n    fake = \'/tmp/pcfree-selkies/fake-udev/\'\n    if all(os.path.exists(fake+\'lib/\'+arch+\'/libudev.so.1\')\n           for arch in (\'x86_64-linux-gnu\', \'i386-linux-gnu\')):\n        env[\'LD_PRELOAD\'] = fake+\'$LIB/libudev.so.1:\'+env[\'LD_PRELOAD\']\n        env.pop(\'SDL_JOYSTICK_DISABLE_UDEV\', None)\n        env.pop(\'SDL_JOYSTICK_DEVICE\', None)\n    else:\n        env[\'SDL_JOYSTICK_DISABLE_UDEV\'] = \'1\'\n        env[\'SDL_JOYSTICK_DEVICE\'] = \':\'.join(\'/dev/input/event\'+str(i) for i in range(1000,1004))\nelse:\n    print(\'Aviso: interposer no encontrado; mandos no preparados.\', file=sys.stderr)\nif os.geteuid() == 0:\n    os.initgroups(account.pw_name, account.pw_gid)\n    os.setgid(account.pw_gid)\n    os.setuid(account.pw_uid)\nos.execvpe(sys.argv[1], sys.argv[1:], env)\n\'\'\'.replace(\'SETTINGS\', repr(session))\n\n\ndef shortcut(session, filename, name, command, icon, terminal=False):\n    path = Path(session[\'home\']) / \'Desktop\' / (filename + \'.desktop\')\n    path.write_text(\'[Desktop Entry]\\nVersion=1.0\\nType=Application\\n\'\n                    f\'Name={name}\\nExec={command}\\nIcon={icon}\\nTerminal={str(terminal).lower()}\\n\')\n    path.chmod(0o755)\n    account = pwd.getpwnam(session[\'user\']); os.chown(path, account.pw_uid, account.pw_gid)\n\n\ndef desktop_files(session):\n    path = Path(session[\'home\']) / \'.config/xfce4/xfconf/xfce-perchannel-xml\'\n    path.mkdir(parents=True, exist_ok=True)\n    (path / \'xfwm4.xml\').write_text(\'\'\'<?xml version="1.0" encoding="UTF-8"?>\n<channel name="xfwm4" version="1.0"><property name="general" type="empty">\n<property name="use_compositing" type="bool" value="false"/>\n</property></channel>\n\'\'\')\n    account = pwd.getpwnam(USER)\n    config_home = Path(session[\'home\']) / \'.config\'\n    for item in [config_home, *config_home.rglob(\'*\')]:\n        os.chown(item, account.pw_uid, account.pw_gid)\n    runner = Path(session[\'run_launcher\']); runner.write_text(launcher_source(session)); runner.chmod(0o755)\n    inputs = Path(\'/dev/input\')\n    if not inputs.exists():\n        inputs.mkdir(mode=0o1777)\n        inputs.chmod(0o1777)\n    shortcut(session, \'Terminal\', \'Terminal\', \'pcfree-run xfce4-terminal\', \'utilities-terminal\')\n    shortcut(session, \'Archivos\', \'Archivos\', \'pcfree-run thunar\', \'system-file-manager\')\n    shortcut(session, \'Descargas\', \'Descargas\', \'pcfree-run thunar \' + session[\'home\'] + \'/Downloads\', \'folder-download\')\n    package_installer(session)\n\n\ndef package_installer(session):\n    """Interactive, small allowlist: no unrestricted passwordless sudo/apt."""\n    allowed = [\'mousepad\', \'gimp\', \'vlc\', \'p7zip-full\', \'unzip\', \'file-roller\', \'htop\']\n    helper = Path(\'/usr/local/sbin/pcfree-install-package\')\n    # Root-owned script; official origins and safe dependency plan.\n    helper.write_text(\'\'\'#!/usr/bin/python3.12\nimport os, re, subprocess, sys, urllib.parse\ndef official_ubuntu_uri(uri):\n    parsed = urllib.parse.urlparse(uri)\n    if parsed.username or parsed.password or parsed.port not in (None, 80, 443):\n        return False\n    host = parsed.hostname or \'\'\n    ubuntu = parsed.scheme in (\'http\', \'https\') and (\n        host in (\'archive.ubuntu.com\',\'security.ubuntu.com\',\'ports.ubuntu.com\')\n        or host.endswith(\'.archive.ubuntu.com\'))\n    path = urllib.parse.unquote(parsed.path)\n    nvidia = (parsed.scheme == \'https\' and parsed.port in (None, 443)\n        and host == \'developer.download.nvidia.com\'\n        and path.startswith(\'/compute/cuda/repos/ubuntu2404/x86_64/\')\n        and \'..\' not in path.split(\'/\'))\n    return ubuntu or nvidia\nALLOWED = ALLOWLIST\nif os.geteuid() != 0 or len(sys.argv) != 2 or sys.argv[1] not in ALLOWED:\n    raise SystemExit(\'Seleccionar un paquete permitido desde el icono Instalador.\')\nplan = subprocess.check_output([\'apt-get\',\'-s\',\'install\',\'-y\',\'--no-install-recommends\',sys.argv[1]], text=True)\nfor line in plan.splitlines():\n    if line.startswith(\'Remv \') or (line.startswith(\'Inst \') and\n       (\'nvidia\' in line.split()[1] or line.split()[1].startswith((\'linux-image\',\'linux-modules\',\'linux-headers\')))):\n        raise SystemExit(\'Operación insegura: cambios del controlador/kernel o eliminaciones.\')\nuris = subprocess.check_output([\'apt-get\',\'--print-uris\',\'--download-only\',\'install\',\'-y\',\'--no-install-recommends\',sys.argv[1]], text=True)\nfor uri in re.findall(r"^\'([^\']+)\'", uris, re.MULTILINE):\n    if not official_ubuntu_uri(uri):\n        raise SystemExit(\'Origen no permitido: solamente Ubuntu o NVIDIA CUDA oficial Ubuntu24.04 amd64.\')\nsubprocess.run([\'apt-get\',\'install\',\'-y\',\'--no-install-recommends\',sys.argv[1]], check=True,\n               env={**os.environ,\'DEBIAN_FRONTEND\':\'noninteractive\'})\n\'\'\'.replace(\'ALLOWLIST\', repr(allowed)))\n    helper.chmod(0o755)\n    rule = Path(\'/etc/sudoers.d/pcfree-packages\')\n    rule.write_text(f\'{USER} ALL=(root) NOPASSWD: /usr/local/sbin/pcfree-install-package *\\n\')\n    rule.chmod(0o440)\n    logged([\'visudo\', \'-cf\', str(rule)])\n    gui = Path(\'/usr/local/bin/pcfree-install\')\n    gui.write_text(\'#!/bin/sh\\npackage=$(zenity --list --title="Paquetes Ubuntu" --column="Paquete" \'\n                   + \' \'.join(shlex.quote(item) for item in allowed) + \') || exit\\n\'\n                   + \'sudo -n /usr/local/sbin/pcfree-install-package "$package"\\nread -r _\\n\')\n    gui.chmod(0o755)\n    shortcut(session, \'Instalador\', \'Instalador de paquetes Ubuntu\', \'pcfree-run pcfree-install\', \'system-software-install\', True)\n\n\ndef install_chrome():\n    """Install the pinned official Chrome .deb after validating the APT plan."""\n    deb = BASE / \'chrome.deb\'\n    download(CHROME_URL, deb)\n    require_sha256(deb, CHROME_SHA256)\n    plan = capture([\'apt-get\', \'-s\', \'install\', \'-y\', \'--no-install-recommends\', str(deb)],\n                   timeout=180)\n    for line in plan.splitlines():\n        if line.startswith(\'Remv \'):\n            raise RuntimeError(\'APT propone eliminar paquetes para Chrome; cancelado.\')\n        if line.startswith(\'Inst \'):\n            name = line.split()[1].split(\':\', 1)[0]\n            if \'nvidia\' in name or name.startswith((\'linux-image\', \'linux-modules\',\n                                                    \'linux-headers\', \'linux-generic\')):\n                raise RuntimeError(f\'Chrome intentaría cambiar {name}; cancelado.\')\n    origins = capture([\'apt-get\', \'--print-uris\', \'--download-only\', \'install\', \'-y\',\n                       \'--no-install-recommends\', str(deb)], timeout=180)\n    for uri, filename in re.findall(r"^\'([^\']*)\'\\s+(\\S+)", origins, re.MULTILINE):\n        if uri.startswith(\'file:\'):\n            local = Path(urllib.parse.unquote(urllib.parse.urlparse(uri).path)).resolve()\n            if local == deb.resolve():\n                continue\n        if uri == \'\' and filename.startswith(\'google-chrome-stable_\'):\n            continue\n        if not official_ubuntu_uri(uri):\n            raise RuntimeError(f\'Origen no oficial para Chrome: {uri}\')\n    logged([\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', str(deb)],\n           \'chrome-install\', env={**os.environ, \'DEBIAN_FRONTEND\': \'noninteractive\'},\n           timeout=900)\n    deb.unlink(missing_ok=True)\n    version = capture([\'google-chrome\', \'--version\'], timeout=30)\n    if \'Google Chrome\' not in version:\n        raise RuntimeError(\'Google Chrome no se instaló correctamente.\')\n    print(f\'Chrome: {version.strip()}\', flush=True)\n\n\ndef chrome_profile(session):\n    """Private profile, low-overhead flags and default browser registration."""\n    BASE.mkdir(mode=0o755, exist_ok=True)\n    profile = BASE / \'chrome-profile\'\n    profile.mkdir(mode=0o700, exist_ok=True)\n    account = pwd.getpwnam(session[\'user\'])\n    os.chown(profile, account.pw_uid, account.pw_gid)\n    flags = [\n        f\'--user-data-dir={profile}\',\n        \'--no-first-run\', \'--no-default-browser-check\',\n        \'--disable-background-networking\', \'--disable-component-update\',\n        \'--disable-sync\', \'--disable-extensions\', \'--disable-default-apps\',\n        \'--disable-breakpad\', \'--disable-hang-monitor\', \'--disable-notifications\',\n        \'--disable-features=Translate,MediaRouter,OptimizationHints\',\n        \'--disable-background-timer-throttling\', \'--disable-renderer-backgrounding\',\n        \'--disable-backgrounding-occluded-windows\', \'--disable-dev-shm-usage\',\n        \'--renderer-process-limit=2\', \'--process-per-site\',\n        \'--disk-cache-size=134217728\', \'--media-cache-size=33554432\',\n        \'--use-gl=egl\', \'--ignore-gpu-blocklist\', \'--ozone-platform=x11\',\n    ]\n    wrapper = Path(session.get(\'bin_dir\', \'/usr/local/bin\')) / \'pcfree-chrome\'\n    wrapper.parent.mkdir(parents=True, exist_ok=True)\n    wrapper.write_text(\'#!/bin/sh\\nexec /usr/local/bin/pcfree-run /usr/bin/google-chrome \'\n                       + \' \'.join(shlex.quote(flag) for flag in flags) + \' "$@"\\n\')\n    wrapper.chmod(0o755)\n    shortcut(session, \'Chrome\', \'Google Chrome\', str(wrapper) + \' %u\', \'google-chrome\')\n    home = Path(session[\'home\'])\n    helpers = home / \'.config\' / \'xfce4\' / \'helpers.rc\'\n    helpers.parent.mkdir(parents=True, exist_ok=True)\n    lines = helpers.read_text().splitlines() if helpers.exists() else []\n    lines = [line for line in lines if not line.startswith(\'WebBrowser=\')]\n    lines.append(\'WebBrowser=google-chrome\')\n    helpers.write_text(\'\\n\'.join(lines) + \'\\n\')\n    os.chown(helpers, account.pw_uid, account.pw_gid)\n    env = dict(session[\'env\'], XDG_CURRENT_DESKTOP=\'XFCE\',\n               PATH=\'/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin\')\n    subprocess.run(as_user([\'xdg-settings\', \'set\', \'default-web-browser\',\n                            \'google-chrome.desktop\']), env=env,\n                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=20)\n    (BASE / \'chrome-status.txt\').write_text(\'listo\\n\')\n    print(\'Chrome listo: perfil aislado y navegador predeterminado.\', flush=True)\n    return profile\n\n\ndef show_access():\n    access = json.loads((BASE / \'access.json\').read_text())\n    print(\'\\nAcceso privado — no guardar estas salidas en GitHub:\', flush=True)\n    print(\'URL:\', access[\'url\'], \'\\nCuenta:\', access[\'username\'], \'\\nContraseña:\', access[\'password\'], flush=True)\n    state = BASE / \'session.json\'\n    settings = json.loads(state.read_text()).get(\'settings\', {}) if state.exists() else {}\n    fps, kbps = settings.get(\'framerate\', 30), settings.get(\'bitrate\', 2000)\n    resolution = settings.get(\'resolution\', \'960x540\')\n    encoder = settings.get(\'encoder\', \'h264enc\')\n    cpu_encode = str(settings.get(\'use_cpu\', False)).lower()\n    queue_size = settings.get(\'backpressure_queue_size\', 120)\n    congestion = \'activa\' if settings.get(\'congestion_control\') else \'desactivada\'\n    print(f\'{resolution} · {encoder} · use-cpu={cpu_encode} · cola={queue_size} · {fps} FPS · {kbps} kbps CBR · congestión {congestion}; NVENC se confirma en TX · Diagnóstico.\')\n    chrome = \'instalado, perfil aislado\' if settings.get(\'chrome\') else \'desactivado\'\n    print(f\'Registros: {BASE} | Chrome: {chrome}, sin abrir ventanas.\')\n\n\ndef auth_ready(username, password):\n    url = f\'http://127.0.0.1:{PORT}/\'\n    try:\n        urllib.request.urlopen(url, timeout=2).close()\n        raise RuntimeError(\'Selkies sirve contenido sin autenticación; no publicar túnel.\')\n    except urllib.error.HTTPError as error:\n        if error.code != 401:\n            return False\n    header = base64.b64encode(f\'{username}:{password}\'.encode()).decode()\n    request = urllib.request.Request(url, headers={\'Authorization\': \'Basic \' + header})\n    with urllib.request.urlopen(request, timeout=2) as response:\n        return response.status == 200\n\n\nINPUT_GUARD_JS = r\'\'\'(() => {\n  // Only cancel browser defaults over the stream, never stop input forwarding.\n  // OS-level Android BACK and reserved system shortcuts cannot be intercepted.\n  const streamTarget = target => target instanceof Element &&\n    !!target.closest(\'#overlayInput, #videoCanvas, #videoWorkerCanvas, #videoStream, #stream\');\n  document.addEventListener(\'mousedown\', event => {\n    if (event.button === 2 && streamTarget(event.target)) event.preventDefault();\n  }, {capture: true, passive: false});\n  document.addEventListener(\'contextmenu\', event => {\n    if (streamTarget(event.target)) event.preventDefault();\n  }, {capture: true, passive: false});\n  window.pcfreeInputGuard = {version: 1, osBackCapture: false};\n  const addGamingButton = () => {\n    const nativeButton = document.querySelector(\'button.gaming-mode-button\');\n    if (!nativeButton || document.getElementById(\'pcfree-play\')) return !!nativeButton;\n    const button = document.createElement(\'button\');\n    button.id = \'pcfree-play\'; button.type = \'button\'; button.textContent = \'Jugar\';\n    button.title = \'Pantalla completa y captura de controles; requiere permisos del navegador\';\n    button.style.cssText = \'position:fixed;bottom:12px;left:12px;z-index:10000;padding:8px 16px;background:#166534;color:white;border:1px solid #86efac;border-radius:8px;cursor:pointer\';\n    button.addEventListener(\'click\', () => {\n      // Use the upstream handler within this real user-activated click.\n      if (typeof window.webrtcInput?.enterGamingMode === \'function\') window.webrtcInput.enterGamingMode();\n      else document.querySelector(\'button.gaming-mode-button\')?.click();\n    });\n    const visibility = () => { button.hidden = !!document.fullscreenElement; };\n    document.addEventListener(\'fullscreenchange\', visibility);\n    document.body.appendChild(button); visibility();\n    return true;\n  };\n  const start = () => {\n    if (addGamingButton()) return;\n    // Only the dashboard needs the CTA. Input-only iframe stays lightweight.\n    if (window.self !== window.top) return;\n    const observer = new MutationObserver(() => { if (addGamingButton()) observer.disconnect(); });\n    observer.observe(document.body, {childList: true, subtree: true});\n  };\n  if (document.readyState === \'loading\') document.addEventListener(\'DOMContentLoaded\', start, {once: true});\n  else start();\n})();\n\'\'\'\n\n\ndef prepare_web_client(source=None):\n    """Private static copy; keep upstream bundles and native Gaming Mode intact."""\n    source = Path(source or \'/opt/selkies/lib/python3.12/site-packages/selkies/selkies_web\')\n    if not (source / \'index.html\').is_file() or not (source / \'assets\').is_dir():\n        raise RuntimeError(\'Cambió la estructura del cliente Selkies; revisar paquete antes de aplicar controles.\')\n    target = BASE / \'web-client\'\n    shutil.copytree(source, target, dirs_exist_ok=True)\n    marker = \'<script src="./pcfree-input-guard.js"></script>\'\n    for index in target.glob(\'*.html\'):\n        html = index.read_text()\n        if \'</head>\' not in html:\n            raise RuntimeError(\'Cliente web sin cabecera reconocible; no modificar bundles.\')\n        if marker not in html:\n            index.write_text(html.replace(\'</head>\', marker + \'\\n</head>\', 1))\n    (target / \'pcfree-input-guard.js\').write_text(INPUT_GUARD_JS)\n    return str(target)\n\n\ndef verify_pixelflux(python):\n    """Selkies 2.1 bundles pixelflux 2.2.1; never replace it with an older wheel."""\n    result = capture([\n        str(python), \'-c\',\n        \'import importlib.metadata as m, pixelflux; \'\n        \'print(m.version("pixelflux")); print(pixelflux.__file__)\'\n    ], timeout=60)\n    lines = [line.strip() for line in result.splitlines() if line.strip()]\n    version = lines[0] if lines else \'\'\n    module = lines[1] if len(lines) > 1 else \'\'\n    try:\n        parts = tuple(int(part) for part in version.split(\'.\')[:3])\n    except ValueError:\n        raise RuntimeError(f\'Versión de pixelflux ilegible: {version!r}\')\n    if parts < (2, 2, 1):\n        raise RuntimeError(f\'Pixelflux {version} es anterior a 2.2.1; no se inicia.\')\n    if \'/opt/selkies/lib/python3.12/site-packages/pixelflux\' not in module:\n        raise RuntimeError(f\'Pixelflux no se importa desde /opt/selkies: {module}\')\n    (BASE / \'pixelflux-version.log\').write_text(f\'{version}\\n{module}\\n\')\n    print(f\'Pixelflux {version} verificado: {module}\', flush=True)\n\n\ndef bootstrap(chrome=True, framerate=60, bitrate=3000, width=960, height=540,\n              backpressure_queue_size=120, congestion_control=False):\n    if (\n        not isinstance(framerate, int) or not 1 <= framerate <= 60\n        or not isinstance(bitrate, int) or not 500 <= bitrate <= 20000\n        or not isinstance(width, int) or not 320 <= width <= 3840\n        or not isinstance(height, int) or not 240 <= height <= 2160\n        or not isinstance(backpressure_queue_size, int) or not 1 <= backpressure_queue_size <= 100000\n        or not isinstance(congestion_control, bool)\n    ):\n        raise ValueError(\'FPS: 1–60; bitrate: 500–20000 kbps; resolución: 320–3840 × 240–2160; \'\n                         \'cola: 1–100000; congestion_control: booleano.\')\n    BASE.mkdir(mode=0o755, exist_ok=True)\n    with (BASE / \'bootstrap.lock\').open(\'w\') as lock:\n        try:\n            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError:\n            raise RuntimeError(\'Otra preparación está en curso; no ejecutar dos veces.\')\n        state = BASE / \'session.json\'\n        if state.exists():\n            session = json.loads(state.read_text())\n            if already_running(session):\n                print(\'Sesión ya activa; no se reinicia ni se reinstala.\')\n                show_access()\n                return session\n            raise RuntimeError(\'Estado previo incompleto/detenido. Revisar registros; no reiniciar automáticamente. Usar VM nueva.\')\n        started = time.monotonic()\n        with phase(\'Comprobaciones\'):\n            driver, bus = preflight()\n        with phase(\'Paquetes mínimos y Selkies oficial\'):\n            logged([\'apt-get\', \'update\', \'-qq\'], timeout=240)\n            safe_apt_install(PACKAGES)\n            install_optional_packages()\n            install_compat_libraries()\n            chrome_ok = False\n            if chrome:\n                try:\n                    install_chrome()\n                    chrome_ok = True\n                except Exception as exc:\n                    print(f\'AVISO: Chrome no se instaló: {exc}\', flush=True)\n            deb = BASE / \'selkies.deb\'; download(SELKIES_URL, deb)\n            require_sha256(deb, SELKIES_DEB_SHA256)\n            safe_apt_install([str(deb)], validated_local_deb=deb)\n            deb.unlink(missing_ok=True)\n            # Only fix the private package venv, NEVER /usr/bin/python3.\n            python = Path(\'/opt/selkies/bin/python3\')\n            if not python.is_symlink():\n                raise RuntimeError(\'Python privado Selkies inesperado; no modificar Python del sistema.\')\n            python.unlink(); python.symlink_to(\'/usr/bin/python3.12\')\n            verify_pixelflux(python)\n            capture([\'/opt/selkies/bin/selkies\', \'--help\'])\n            web_root = prepare_web_client()\n            shm_perms = shm_perms_library()\n            tunnel = BASE / \'cloudflared\'; download(CLOUDFLARED_URL, tunnel)\n            require_sha256(tunnel, CLOUDFLARED_SHA256)\n            tunnel.chmod(0o755)\n            logged([\'apt-get\', \'clean\'], \'apt-clean\', timeout=120)\n        with phase(\'Bibliotecas 3D exactas, aisladas del kernel\'):\n            root, libraries = graphics(driver)\n            session, env = prepare_user(root, libraries)\n            for deb in (BASE / \'nvidia\').glob(\'*.deb\'):\n                deb.unlink(missing_ok=True)\n            icd = vulkan_icd(root)\n            for key in (\'VK_DRIVER_FILES\', \'VK_ICD_FILENAMES\'):\n                env[key] = icd\n                session[\'env\'][key] = icd\n            egl = BASE / \'nvidia-egl.json\'\n            egl_library = next(root.rglob(\'libEGL_nvidia.so.0\'), None)\n            if egl_library is None:\n                raise RuntimeError(\'Falta EGL NVIDIA para el navegador; no usar renderizado software silencioso.\')\n            egl.write_text(json.dumps({\'file_format_version\': \'1.0.0\',\n                                      \'ICD\': {\'library_path\': str(egl_library)}}))\n            for mapping in (env, session[\'env\']):\n                mapping[\'__EGL_VENDOR_LIBRARY_FILENAMES\'] = str(egl)\n            session[\'driver_version\'] = driver\n            session[\'settings\'] = {\n                \'framerate\': framerate, \'bitrate\': bitrate, \'chrome\': chrome_ok,\n                \'width\': width, \'height\': height, \'resolution\': f\'{width}x{height}\',\n                \'encoder\': \'h264enc\', \'use_cpu\': False,\n                \'backpressure_queue_size\': backpressure_queue_size,\n                \'rate_control_mode\': \'cbr\', \'enable_resize\': True,\n                \'congestion_control\': congestion_control,\n            }\n            desktop_files(session)\n            if chrome_ok:\n                try:\n                    chrome_profile(session)\n                except Exception as exc:\n                    session[\'settings\'][\'chrome\'] = False\n                    print(f\'AVISO: perfil de Chrome no creado: {exc}\', flush=True)\n        # Mark setup BEFORE launching, so failures never cause a destructive rerun.\n        session[\'processes\'] = {}\n        private_json(state, session)\n        processes = {}\n        with phase(\'Xorg NVIDIA, XFCE y audio\'):\n            parts = bus.split(\':\'); device, function = parts[-1].split(\'.\')\n            busid = f\'PCI:{int(parts[-2],16)}:{int(device,16)}:{int(function,16)}\'\n            config = BASE / \'xorg.conf\'\n            config.write_text(\'\'\'Section "ServerFlags"\n Option "AutoAddDevices" "false"\n Option "AutoAddGPU" "false"\nEndSection\nSection "Device"\n Identifier "T4"\n Driver "nvidia"\n BusID "BUSID"\n Option "AllowEmptyInitialConfiguration" "true"\nEndSection\nSection "Screen"\n Identifier "GPU-Screen"\n Device "T4"\n DefaultDepth 24\n SubSection "Display"\n  Depth 24\n  Virtual MAX_WIDTH MAX_HEIGHT\n EndSubSection\nEndSection\n\'\'\'.replace(\'BUSID\', busid).replace(\'MAX_WIDTH\', str(max(width, 1920))).replace(\'MAX_HEIGHT\', str(max(height, 1080))))\n            module = next(root.rglob(\'nvidia_drv.so\')).parent.parent\n            xorg = spawn([\'/usr/lib/xorg/Xorg\', DISPLAY, \'-config\', str(config), \'-modulepath\',\n                          f\'{module},/usr/lib/xorg/modules\', \'-logfile\', str(BASE / \'Xorg.log\'),\n                          \'-nolisten\', \'tcp\', \'-noreset\', \'-novtswitch\', \'-sharevts\', \'-auth\',\n                          session[\'xauthority\']], \'xorg\', env, processes, user=False)\n            def display_ready():\n                return subprocess.run(as_user([\'xdpyinfo\']), env=env, stdout=subprocess.DEVNULL,\n                                      stderr=subprocess.DEVNULL, timeout=3).returncode == 0\n            wait_for(display_ready, \'Xorg\', [xorg])\n            renderer = capture(as_user([\'glxinfo\', \'-B\']), env=env)\n            (BASE / \'opengl.log\').write_text(renderer)\n            if \'NVIDIA\' not in renderer or \'T4\' not in renderer:\n                raise RuntimeError(\'OpenGL T4 no confirmado; revisar Xorg.log.\')\n            # Vulkan is optional for desktop availability: a Vulkan loader/WSI failure must not\n            # prevent the known-good NVIDIA OpenGL + Steam + Selkies path from starting.\n            try:\n                vulkan = capture(as_user([\'vulkaninfo\', \'--summary\']), env=env, timeout=30)\n                vulkan_rc = 0\n            except Exception as exc:\n                vulkan = f\'vulkaninfo no pudo completar: {type(exc).__name__}: {exc}\\n\'\n                vulkan_rc = 1\n            (BASE / \'vulkan.log\').write_text(vulkan)\n            vulkan_ok = vulkan_rc == 0 and bool(re.search(r\'deviceName\\s*=\\s*[^\\n]*T4\', vulkan))\n            if not vulkan_ok:\n                print(\'AVISO: Vulkan no confirmado; se conserva OpenGL y continúa el escritorio. Revisar /tmp/pcfree-selkies/vulkan.log.\', flush=True)\n            session[\'graphics\'] = {\'opengl_nvidia_t4\': True, \'vulkan_nvidia_t4\': vulkan_ok,\n                \'vulkan_log\': str(BASE / \'vulkan.log\'),\n                \'gles_profile\': next((line.strip() for line in renderer.splitlines()\n                                      if \'OpenGL ES profile version string\' in line), \'No informado por glxinfo\')}\n            audio_env = dict(env); audio_env.pop(\'PULSE_SERVER\', None)\n            pulse = spawn([\'pulseaudio\', \'--daemonize=no\', \'--exit-idle-time=-1\', \'--disallow-exit\',\n                           \'--log-target=stderr\'], \'pulse\', audio_env, processes, nice=-10)\n            def audio_ready():\n                return subprocess.run(as_user([\'pactl\', \'info\']), env=env, stdout=subprocess.DEVNULL,\n                                      stderr=subprocess.DEVNULL, timeout=3).returncode == 0\n            wait_for(audio_ready, \'PulseAudio\', [pulse])\n            sinks = capture(as_user([\'pactl\', \'list\', \'short\', \'sinks\']), env=env)\n            if not any(line.split()[1] == \'output\' for line in sinks.splitlines() if len(line.split()) >= 2):\n                try:\n                    capture(as_user([\'pactl\', \'load-module\', \'module-null-sink\', *AUDIO_SINK_ARGS]), env=env)\n                except Exception as exc:\n                    print(f\'Aviso: el sink de 48 kHz no se pudo crear ({exc}); se usa el sink simple.\', flush=True)\n                    capture(as_user([\'pactl\', \'load-module\', \'module-null-sink\', \'sink_name=output\',\n                                     \'sink_properties=device.description=PC-Free\']), env=env)\n            capture(as_user([\'pactl\', \'set-default-sink\', \'output\']), env=env)\n            capture(as_user([\'pactl\', \'set-default-source\', \'output.monitor\']), env=env)\n            # flat-volumes=no keeps gains independent; 100% avoids clipping when a\n            # game or the client raises its own volume at medium/high settings.\n            capture(as_user([\'pactl\', \'set-sink-volume\', \'output\', \'100%\']), env=env)\n            capture(as_user([\'pactl\', \'set-sink-mute\', \'output\', \'0\']), env=env)\n            dbus = spawn([\'dbus-daemon\', \'--session\', \'--nofork\',\n                          \'--address=\' + env[\'DBUS_SESSION_BUS_ADDRESS\']], \'dbus\', env, processes)\n            def bus_ready():\n                return subprocess.run(as_user([\'dbus-send\', \'--session\', \'--print-reply\',\n                                      \'--dest=org.freedesktop.DBus\', \'/org/freedesktop/DBus\',\n                                      \'org.freedesktop.DBus.ListNames\']), env=env,\n                                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\n                                      timeout=3).returncode == 0\n            wait_for(bus_ready, \'D-Bus de escritorio\', [dbus])\n            desktop = spawn([\'xfce4-session\'], \'xfce\', env, processes)\n            def desktop_ready():\n                result = subprocess.run(as_user([\'xprop\', \'-root\', \'_NET_SUPPORTING_WM_CHECK\']),\n                                        env=env, capture_output=True, text=True, timeout=3)\n                return result.returncode == 0 and \'window id #\' in result.stdout\n            wait_for(desktop_ready, \'XFCE\', [desktop])\n        with phase(\'Selkies autenticado y túnel HTTPS HTTP/2\'):\n            access = dict(username=\'pcfree\', password=secrets.token_urlsafe(24), url=None)\n            private_json(BASE / \'access.json\', access)\n            server_env = dict(env, SELKIES_BASIC_AUTH_USER=access[\'username\'],\n                              SELKIES_BASIC_AUTH_PASSWORD=access[\'password\'],\n                              SELKIES_KEYBOARD_SHORTCUTS=\'true\',\n                              SELKIES_UI_SIDEBAR_SHOW_GAMING_MODE=\'true\',\n                              SELKIES_UI_SIDEBAR_SHOW_FULLSCREEN=\'true\',\n                              SELKIES_UI_SIDEBAR_SHOW_KEYBOARD_BUTTON=\'true\',\n                              SELKIES_VIDEO_STREAMING_MODE=\'true|locked\',\n                              SELKIES_USE_CPU=\'false|locked\', SELKIES_AUTO_GPU=\'nvidia\',\n                              SELKIES_AUDIO_ENABLED=\'true|locked\',\n                              SELKIES_AUDIO_BITRATE=\'96000\',\n                              SELKIES_AUDIO_CHANNELS=\'2\',\n                              SELKIES_AUDIO_DEVICE_NAME=\'output.monitor\',\n                              SELKIES_AUDIO_FRAME_DURATION_MS=\'20\',\n                              SELKIES_AUDIO_REDUNDANCY=\'true|locked\',\n                              SELKIES_AUDIO_REDUNDANCY_DISTANCE=\'2\',\n                              SELKIES_RATE_CONTROL_MODE=\'cbr\')\n            # NVENC FIX (Google Colab Tesla T4): SOLO afecta al proceso de Selkies.\n            # Conserva LD_LIBRARY_PATH y añade las bibliotecas NVIDIA del runtime.\n            # No modifica Steam, Xorg ni Cloudflare.\n            server_env[\'LD_LIBRARY_PATH\'] = \':\'.join(\n                p for p in (\n                    server_env.get(\'LD_LIBRARY_PATH\', \'\'),\n                    \'/usr/lib64-nvidia\'\n                ) if p\n            )\n\n            # PATCH 2/2: keep the stable XShm permissions shim; revert experimental system libxcb preload.\n            server_env.pop(\'LD_PRELOAD\', None)\n            server_env[\'LD_PRELOAD\'] = shm_perms\n            server = spawn([\'/opt/selkies/bin/selkies\', \'--addr=127.0.0.1\', f\'--port={PORT}\',\n                            \'--web-root=\' + web_root,\n                            \'--mode=websockets\', \'--enable-basic-auth=true\', \'--enable-https=false\',\n                            \'--encoder=h264enc\', \'--use-cpu=false\', \'--auto-gpu=nvidia\',\n                            f\'--framerate={framerate}\', f\'--video-bitrate={bitrate}\',\n                            \'--rate-control-mode=cbr\', \'--keyframe-interval=0\',\n                            \'--video-streaming-mode=true\', \'--video-fullcolor=false\',\n                            \'--enable-resize=true\', f\'--backpressure-queue-size={backpressure_queue_size}\',\n                            \'--congestion-control=\' + (\'true\' if congestion_control else \'false\'),\n                            \'--frame-ancestors=self\',\n                            \'--use-paint-over-quality=false\', \'--second-screen=false\',\n                            \'--audio-enabled=true\', \'--audio-bitrate=96000\',\n                            \'--audio-channels=2\', \'--audio-device-name=output.monitor\',\n                            \'--audio-frame-duration-ms=20\', \'--audio-redundancy=true\'],\n                           \'selkies\', server_env, processes)\n            wait_for(lambda: auth_ready(access[\'username\'], access[\'password\']), \'Selkies + autenticación\', [server], 60)\n            selkies_log = (BASE / \'selkies.log\').read_text(errors=\'replace\')\n            if re.search(r\'X11 Access error|BadAccess|XShm.*(fail|error)\', selkies_log, re.I):\n                raise RuntimeError(\'Selkies no puede capturar la pantalla (XShm/X11); revisar selkies.log.\')\n            cloud = spawn([str(tunnel), \'tunnel\', \'--url\', f\'http://127.0.0.1:{PORT}\',\n                           \'--protocol\', \'http2\', \'--no-autoupdate\'], \'cloudflared\', env, processes)\n            def tunnel_ready():\n                log = (BASE / \'cloudflared.log\').read_text()\n                links = re.findall(r\'https://[a-z0-9-]+\\.trycloudflare\\.com\', log)\n                if links and \'Registered tunnel connection\' in log and \'protocol=http2\' in log:\n                    access[\'url\'] = links[-1]\n                    return True\n                return False\n            # Do NOT require VM DNS/outbound reachability of the quick-tunnel URL.\n            wait_for(tunnel_ready, \'Cloudflare HTTP/2\', [cloud, server], 90)\n            private_json(BASE / \'access.json\', access)\n        session[\'processes\'] = {name: {\'pid\': process.pid, \'start\': pid_identity(process.pid)}\n                                for name, process in processes.items()}\n        private_json(state, session)\n        print(f\'Escritorio preparado en {time.monotonic() - started:.1f} s.\', flush=True)\n        show_access()\n        return session\n\n\nif __name__ == \'__main__\':\n    bootstrap()\n', 'compat-steam.py': '#!/usr/bin/env python3\n"""Steam oficial opcional; instala, pero no abre ventanas.\n\nAPI: install_steam(session), install_gamepads(session) (sin Steam).\nEstado: /tmp/pcfree-selkies/session.json.\nOpciones en session[\'steam\']: gamepads=False, repair_client=False,\n    bwrap=None (ruta a un proot-bwrap ya instalado y revisado).\nNo cambia el controlador del kernel, libudev del sistema ni el Python global.\n\nFuentes: repo.steampowered.com/steam/archive/stable/steam_latest.deb;\ngithub.com/selkies-project/selkies/tree/2.1.0/addons/fake-udev;\ngithub.com/selkies-project/proot-bwrap (alternativa, no frontera de seguridad).\nfake-udev sirve para mandos; no es una reparación demostrada del login.\n"""\n\nimport argparse\nimport datetime\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport posixpath\nimport pwd\nimport re\nimport shlex\nimport shutil\nimport struct\nimport subprocess\nimport tarfile\nimport tempfile\nimport urllib.parse\nimport urllib.request\n\n\nSTEAM_DEB = \'https://repo.steampowered.com/steam/archive/stable/steam_latest.deb\'\nSELKIES_TAG = \'2.1.0\'\nGE_PROTON_VERSION = \'GE-Proton11-11\'\nGE_PROTON_URL = (\'https://github.com/GloriousEggroll/proton-ge-custom/releases/download/\'\n                 f\'{GE_PROTON_VERSION}/{GE_PROTON_VERSION}-x86_64.tar.gz\')\nGE_PROTON_SHA512 = (\'45721a7b625b2404d8896bb5e2a8b87fc9623a394873a6dd6b900c839681d5c0\'\n                    \'1ec694ca00d003fc9c733c5f6c5648af8171985656f9fa47e20999fbebdbe235\')\nTRIPLETS = {\'amd64\': \'x86_64-linux-gnu\', \'i386\': \'i386-linux-gnu\'}\n\n\ndef _run(argv, *, cwd=None, log=None, timeout=600, env=None, host_gpu=False):\n    """APT/build sin heredar preloads del escritorio ni imprimir credenciales."""\n    child_env = dict(os.environ)\n    child_env.update(env or {})\n    child_env.update(DEBIAN_FRONTEND=\'noninteractive\', LC_ALL=\'C\')\n    for key in (\'LD_PRELOAD\', \'PYTHONHOME\', \'PYTHONPATH\'):\n        child_env.pop(key, None)\n    if host_gpu:\n        # Sólo nvidia-smi necesita NVML del host Colab; APT/build quedan limpios.\n        if Path(argv[0]).name != \'nvidia-smi\':\n            raise ValueError(\'host_gpu sólo está permitido para nvidia-smi.\')\n        paths = [part for part in child_env.get(\'LD_LIBRARY_PATH\', \'\').split(\':\') if part]\n        paths += [part for part in os.environ.get(\'LD_LIBRARY_PATH\', \'\').split(\':\') if part]\n        child_env[\'LD_LIBRARY_PATH\'] = \':\'.join(dict.fromkeys(paths + [\'/usr/lib64-nvidia\']))\n    else:\n        child_env.pop(\'LD_LIBRARY_PATH\', None)\n    if argv[0] == \'apt-get\':\n        argv = [argv[0], \'-o\', \'APT::Get::AllowUnauthenticated=false\',\n                \'-o\', \'Acquire::AllowInsecureRepositories=false\',\n                \'-o\', \'Acquire::AllowDowngradeToInsecureRepositories=false\', *argv[1:]]\n    result = subprocess.run(argv, cwd=cwd, env=child_env, text=True,\n                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                            timeout=timeout)\n    if log:\n        with Path(log).open(\'a\') as stream:\n            stream.write(\'$ \' + shlex.join(map(str, argv)) + \'\\n\' + result.stdout + \'\\n\')\n    if result.returncode:\n        raise RuntimeError(f"Falló {argv[0]} ({result.returncode}). "\n                           f"Revisar {log or \'la salida\'}: {result.stdout[-1800:]}")\n    return result.stdout.strip()\n\n\ndef _official_apt_uri(uri):\n    """Mismos orígenes que el núcleo; /./ de APT es válido, /../ no."""\n    parsed = urllib.parse.urlparse(uri)\n    if (parsed.username or parsed.password or parsed.port not in (None, 80, 443)\n            or parsed.query or parsed.fragment):\n        return False\n    path = urllib.parse.unquote(parsed.path)\n    if \'..\' in path.split(\'/\'):\n        return False\n    ubuntu = (parsed.scheme in (\'http\', \'https\') and\n              (parsed.hostname in (\'archive.ubuntu.com\', \'security.ubuntu.com\', \'ports.ubuntu.com\')\n               or (parsed.hostname or \'\').endswith(\'.archive.ubuntu.com\')))\n    nvidia = (parsed.scheme == \'https\' and parsed.port in (None, 443) and\n              parsed.hostname == \'developer.download.nvidia.com\' and\n              path.startswith(\'/compute/cuda/repos/ubuntu2404/x86_64/\'))\n    return ubuntu or nvidia\n\n\ndef _safe_install(packages, log):\n    plan = _run([\'apt-get\', \'-s\', \'install\', \'-y\', \'--no-install-recommends\', *packages], log=log)\n    for line in plan.splitlines():\n        if line.startswith(\'Remv \'):\n            raise RuntimeError(\'APT propone eliminar paquetes; operación cancelada.\')\n        if line.startswith(\'Inst \'):\n            name = line.split()[1].split(\':\')[0]\n            if (\'nvidia\' in name or name.startswith((\'linux-image\', \'linux-modules\',\n                                                   \'linux-headers\', \'linux-generic\', \'linux-virtual\'))):\n                raise RuntimeError(f\'APT propone cambiar {name}; no se modifica el kernel/GPU del host.\')\n    origins = _run([\'apt-get\', \'--print-uris\', \'--download-only\', \'install\', \'-y\',\n                    \'--no-install-recommends\', *packages], log=log)\n    for uri in re.findall(r"^\'([^\']*)\'", origins, re.MULTILINE):\n        if not _official_apt_uri(uri):\n            raise RuntimeError(\'APT propone un origen no oficial; operación cancelada.\')\n    _run([\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', *packages], log=log)\n\n\ndef _download(url, target):\n    # HTTPS, hosts oficiales; nunca ejecutar un instalador remoto mediante pipe.\n    with urllib.request.urlopen(url, timeout=90) as response:\n        if urllib.parse.urlparse(response.url).hostname not in {\n                \'repo.steampowered.com\', \'github.com\', \'codeload.github.com\',\n                \'objects.githubusercontent.com\', \'release-assets.githubusercontent.com\'}:\n            raise RuntimeError(\'Redirección de descarga fuera de los hosts oficiales.\')\n        with Path(target).open(\'wb\') as stream:\n            shutil.copyfileobj(response, stream)\n\n\ndef _elf(path, arch):\n    with Path(path).open(\'rb\') as stream:\n        head = stream.read(20)\n    expected = (2, 62) if arch == \'amd64\' else (1, 3)\n    if (len(head) != 20 or head[:4] != b\'\\x7fELF\' or head[5] != 1 or\n            (head[4], struct.unpack(\'<H\', head[18:20])[0]) != expected):\n        raise RuntimeError(f\'Biblioteca ELF incompatible ({arch}): {path}\')\n\n\ndef _owned_write(path, text, account, executable=False):\n    path = Path(path)\n    _user_directory(path.parent, account)\n    path.write_text(text)\n    path.chmod(0o755 if executable else 0o644)\n    os.chown(path, account.pw_uid, account.pw_gid)\n\n\ndef _user_directory(path, account):\n    missing = []\n    cursor = Path(path)\n    while not cursor.exists():\n        missing.append(cursor)\n        cursor = cursor.parent\n    for directory in reversed(missing):\n        directory.mkdir(mode=0o755)\n        os.chown(directory, account.pw_uid, account.pw_gid)\n\n\ndef _exact_package(package, arch, driver, log):\n    lines = _run([\'apt-cache\', \'madison\', f\'{package}:{arch}\'], log=log)\n    versions = [line.split(\'|\')[1].strip() for line in lines.splitlines()\n                if len(line.split(\'|\')) >= 3]\n    candidates = [v for v in versions if v.split(\':\')[-1].split(\'-\')[0] == driver]\n    if not candidates:\n        raise RuntimeError(f\'Falta {package}:{arch} para NVIDIA {driver} en APT. \'\n                           \'Habilitar un repositorio oficial Ubuntu/NVIDIA con esa versión; \'\n                           \'no instalar otro controlador ni sustituir por una versión distinta.\')\n    # APT ordena sus resultados; no asumir que la revisión Ubuntu es la de otra arquitectura.\n    return candidates[0]\n\n\ndef _graphics(base, driver, log):\n    """Extrae sólo GL/gpucomp; no instala metapaquetes NVIDIA, DKMS o módulos."""\n    branch = driver.split(\'.\')[0]\n    root = base / \'nvidia\' / driver\n    root.mkdir(parents=True, exist_ok=True)\n    specs = []\n    for arch in TRIPLETS:\n        package = f\'libnvidia-gl-{branch}\'\n        version = _exact_package(package, arch, driver, log)\n        specs.append((package, arch, version))\n        # Algunas ramas separan gpucomp; leer la dependencia de la versión exacta.\n        metadata = _run([\'apt-cache\', \'show\', f\'{package}:{arch}={version}\'], log=log)\n        dependencies = re.findall(r\'libnvidia-gpucomp-[0-9]+\', metadata)\n        for dependency in sorted(set(dependencies)):\n            specs.append((dependency, arch,\n                          _exact_package(dependency, arch, driver, log)))\n    # Resolver todos los orígenes ANTES de descargar; APT verifica hashes contra\n    # metadatos autenticados. No añadir claves, trusted=yes ni PPAs.\n    for package, arch, version in specs:\n        origins = _run([\'apt-get\', \'--print-uris\', \'download\', f\'{package}:{arch}={version}\'], log=log)\n        uris = re.findall(r"^\'([^\']+)\'", origins, re.MULTILINE)\n        if not uris or not all(_official_apt_uri(uri) for uri in uris):\n            raise RuntimeError(f\'{package}:{arch}: origen NVIDIA/Ubuntu no permitido.\')\n    with tempfile.TemporaryDirectory(prefix=\'download-\', dir=base) as temporary:\n        for package, arch, version in specs:\n            _run([\'apt-get\', \'download\', f\'{package}:{arch}={version}\'],\n                 cwd=temporary, log=log)\n        for deb in sorted(Path(temporary).glob(\'*.deb\')):\n            _run([\'dpkg-deb\', \'-x\', str(deb), str(root)], log=log)\n    directories = []\n    for arch, triplet in TRIPLETS.items():\n        directory = root / \'usr/lib\' / triplet\n        for library in (\'libGLX_nvidia.so.0\', \'libEGL_nvidia.so.0\'):\n            _elf(directory / library, arch)\n        directories.append(str(directory))\n    # El mismo nombre se resuelve con el ELF correcto en cada proceso. No fijar\n    # una ruta absoluta amd64 en el ICD que también utilizará Steam de 32 bits.\n    icd = root / \'share/vulkan/icd.d/nvidia.json\'\n    icd.parent.mkdir(parents=True, exist_ok=True)\n    icd.write_text(json.dumps({\'file_format_version\': \'1.0.0\', \'ICD\': {\n        \'library_path\': \'libGLX_nvidia.so.0\', \'api_version\': \'1.3.0\'}}, indent=2))\n    egl = root / \'share/glvnd/egl_vendor.d/nvidia.json\'\n    egl.parent.mkdir(parents=True, exist_ok=True)\n    egl.write_text(json.dumps({\'file_format_version\': \'1.0.0\', \'ICD\': {\n        \'library_path\': \'libEGL_nvidia.so.0\'}}, indent=2))\n    return {\'LD_LIBRARY_PATH\': \':\'.join(directories),\n            \'__GLX_VENDOR_LIBRARY_NAME\': \'nvidia\',\n            \'__EGL_VENDOR_LIBRARY_FILENAMES\': str(egl),\n            \'VK_DRIVER_FILES\': str(icd), \'VK_ICD_FILENAMES\': str(icd)}, specs\n\n\ndef _gamepads(base, log, shared_root):\n    """fake-udev privado, 2 ELF verificados. Nunca sustituye libudev del sistema."""\n    _safe_install([\'build-essential\', \'gcc-multilib\', \'libc6-dev-i386\',\n                   \'libudev1:amd64\', \'libudev1:i386\'], log)\n    with tempfile.TemporaryDirectory(prefix=\'fake-udev-\', dir=base) as temporary:\n        archive = Path(temporary) / \'selkies.tar.gz\'\n        _download(f\'https://github.com/selkies-project/selkies/archive/refs/tags/{SELKIES_TAG}.tar.gz\',\n                  archive)\n        with tarfile.open(archive) as source:\n            # Extraer sólo este addon y rechazar enlaces/path traversal del archivo.\n            prefix = f\'selkies-{SELKIES_TAG}/addons/fake-udev/\'\n            members = [member for member in source.getmembers()\n                       if member.name.startswith(prefix)]\n            if not members:\n                raise RuntimeError(\'El archivo oficial no contiene fake-udev.\')\n            for member in members:\n                if (\'..\' in Path(member.name).parts or\n                        not (member.isfile() or member.isdir())):\n                    raise RuntimeError(\'Entrada de archivo no segura en fake-udev.\')\n            source.extractall(temporary, members=members, filter=\'data\')\n        build = Path(temporary) / prefix\n        _run([\'make\', \'-C\', str(build), \'all\', \'all32\'], log=log)\n        for arch, triplet in TRIPLETS.items():\n            filename = \'libudev.so.1.0.0-fake\' if arch == \'amd64\' else \'libudev_x86.so.1.0.0-fake\'\n            source = build / filename\n            _elf(source, arch)\n            destination = shared_root / \'lib\' / triplet\n            destination.mkdir(parents=True, exist_ok=True)\n            # Reinstalar no debe truncar una biblioteca ya mapeada por un juego.\n            staged = destination / \'libudev.so.1.new\'\n            shutil.copy2(source, staged)\n            staged.replace(destination / \'libudev.so.1\')\n            # La .deb oficial Selkies proporciona ambas variantes del interposer.\n            interposer = Path(\'/usr/lib\') / triplet / \'selkies_input_interposer.so\'\n            _elf(interposer, arch)\n    input_directory = Path(\'/dev/input\')\n    if not input_directory.exists():\n        input_directory.mkdir(mode=0o1777)\n        input_directory.chmod(0o1777)\n    # $LIB es literal: glibc lo expande según la arquitectura DEL PROCESO.\n    library = str(shared_root / \'$LIB\')\n    return {\'SELKIES_INTERPOSER\': \'/usr/$LIB/selkies_input_interposer.so\',\n            \'fake_udev\': library + \'/libudev.so.1\'}\n\n\ndef _quarantine_client(home, base):\n    """Reparación solicitada: sólo carpetas redistribuibles, jamás cuentas/juegos."""\n    client = home / \'.local/share/Steam\'\n    if client.is_symlink():\n        raise RuntimeError(\'Steam usa un enlace externo; revisar manualmente antes de reparar.\')\n    quarantine = base / (\'quarantine-\' + datetime.datetime.now(datetime.timezone.utc)\n                         .strftime(\'%Y%m%dT%H%M%S%fZ\'))\n    for name in (\'ubuntu12_32\', \'ubuntu12_64\', \'steam.sh\'):\n        target = client / name\n        if target.exists() or target.is_symlink():\n            quarantine.mkdir(exist_ok=True)\n            target.rename(quarantine / name)\n    return str(quarantine) if quarantine.exists() else None\n\n\ndef install_gamepads(session):\n    """Mandos opcionales independientes de Steam; no lanza/reinicia procesos."""\n    if os.geteuid() != 0:\n        raise RuntimeError(\'La instalación de mandos requiere root; las aplicaciones no.\')\n    account = pwd.getpwnam(session[\'user\'])\n    if account.pw_uid == 0:\n        raise ValueError(\'Los juegos deben ejecutarse como cuenta normal.\')\n    home, runtime = Path(session[\'home\']), Path(session[\'runtime\'])\n    if not home.is_absolute() or not runtime.is_absolute() or not home.is_dir() or not runtime.is_dir():\n        raise ValueError(\'Preparar primero home y runtime absolutos del escritorio.\')\n    shared_root = Path(session.get(\'fake_udev_root\', runtime.parent / \'fake-udev\'))\n    if not shared_root.is_absolute():\n        raise ValueError(\'fake_udev_root debe ser una ruta absoluta.\')\n    base = home / \'.local/share/pcfree/gamepads\'\n    _user_directory(base, account)\n    log = base / \'install.log\'\n    _run([\'dpkg\', \'--add-architecture\', \'i386\'], log=log)\n    _run([\'apt-get\', \'update\', \'-qq\'], log=log)\n    preload = _gamepads(base, log, shared_root)\n    result = {\'installed\': True, \'root\': str(shared_root), \'log\': str(log),\n              \'source_tag\': SELKIES_TAG, \'architectures\': list(TRIPLETS),\n              \'env\': {\'SELKIES_INTERPOSER\': preload[\'SELKIES_INTERPOSER\'],\n                      \'LD_PRELOAD\': preload[\'SELKIES_INTERPOSER\'] + \':\' + preload[\'fake_udev\']},\n              \'unset_env\': [\'SDL_JOYSTICK_DISABLE_UDEV\', \'SDL_JOYSTICK_DEVICE\']}\n    _owned_write(base / \'installation.json\', json.dumps(result, indent=2), account)\n    print(\'Mandos: fake-udev aislado e interposer oficial verificados para 32 y 64 bits.\')\n    print(\'Aplicar sólo a aplicaciones/juegos; no precargar en el servidor Selkies.\')\n    return result\n\n\ndef _launch_script(client, graphics, gamepads, bwrap):\n    assignments = \'\'.join(\'export \' + key + \'=\' + shlex.quote(value) + \'\\n\'\n                          for key, value in graphics.items() if key != \'LD_LIBRARY_PATH\')\n    assignments += (\'export LD_LIBRARY_PATH=\' + shlex.quote(graphics[\'LD_LIBRARY_PATH\']) +\n                    \'${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}\\n\')\n    if gamepads:\n        assignments += \'unset SDL_JOYSTICK_DISABLE_UDEV SDL_JOYSTICK_DEVICE\\n\'\n        # Reusar el input interposer oficial heredado de pcfree-run; copiarlo\n        # a otra ruta podría cargar dos instancias de los mismos hooks.\n        # No usar expansión shell de $LIB en este valor por defecto.\n        assignments += (\'if [ -z "${SELKIES_INTERPOSER:-}" ]; then export SELKIES_INTERPOSER=\' +\n                        shlex.quote(gamepads[\'SELKIES_INTERPOSER\']) + \'; fi\\n\')\n        assignments += \'\'\'prepend_preload() {\n  case ":${LD_PRELOAD:-}:" in *":$1:"*) return;; esac\n  LD_PRELOAD="$1${LD_PRELOAD:+:$LD_PRELOAD}"\n  export LD_PRELOAD\n}\nprepend_preload "$SELKIES_INTERPOSER"\n\'\'\'\n        assignments += \'prepend_preload \' + shlex.quote(gamepads[\'fake_udev\']) + \'\\n\'\n    if bwrap:\n        assignments += \'export BWRAP=\' + shlex.quote(bwrap) + \'\\n\'\n    return \'\'\'#!/bin/sh\nset -eu\nif [ -r "$HOME/.config/pcfree/game.env" ]; then\n  . "$HOME/.config/pcfree/game.env"\nfi\nif [ "$(id -u)" = 0 ]; then echo \'Steam debe ejecutarse como cuenta normal.\' >&2; exit 1; fi\n: "${HOME:?}" "${XDG_RUNTIME_DIR:?}" "${DISPLAY:?}" "${XAUTHORITY:?}"\nbus_ok() {\n  dbus-send --bus="$1" --type=method_call --print-reply --dest=org.freedesktop.DBus \\\n    /org/freedesktop/DBus org.freedesktop.DBus.ListNames >/dev/null 2>&1\n}\n# Preferir el bus persistente del escritorio. Nunca dbus-run-session alrededor\n# del bootstrap: éste puede terminar antes que su steamwebhelper desacoplado.\nif [ -z "${DBUS_SESSION_BUS_ADDRESS:-}" ] || ! bus_ok "$DBUS_SESSION_BUS_ADDRESS"; then\n  address="unix:path=$XDG_RUNTIME_DIR/steam-session-bus"\n  (\n    flock -x 9\n    if ! bus_ok "$address"; then\n      dbus-daemon --session --fork --address="$address"\n    fi\n  ) 9>"$XDG_RUNTIME_DIR/steam-session-bus.lock"\n  export DBUS_SESSION_BUS_ADDRESS="$address"\nfi\n\'\'\' + assignments + \'exec \' + shlex.quote(str(client)) + \' "$@"\\n\'\n\n\ndef _sha512(path):\n    digest = hashlib.sha512()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef install_ge_proton(session):\n    """Download the pinned GE-Proton release on demand and verify its SHA-512."""\n    required = (\'user\', \'home\')\n    if any(key not in session for key in required):\n        raise ValueError(\'Estado incompleto para GE-Proton.\')\n    account = pwd.getpwnam(session[\'user\'])\n    home = Path(session[\'home\'])\n    base = home / \'.cache\' / \'pcfree\' / \'ge-proton\'\n    base.mkdir(parents=True, exist_ok=True)\n    os.chown(base.parent, account.pw_uid, account.pw_gid)\n    os.chown(base, account.pw_uid, account.pw_gid)\n\n    tools = home / \'.steam\' / \'steam\' / \'compatibilitytools.d\'\n    tools.mkdir(parents=True, exist_ok=True)\n    os.chown(tools, account.pw_uid, account.pw_gid)\n    target = tools / GE_PROTON_VERSION\n    proton = target / \'proton\'\n    if proton.is_file() and os.access(proton, os.X_OK):\n        print(f\'GE-Proton ya instalado: {target}\')\n        return {\'installed\': True, \'path\': str(target), \'version\': GE_PROTON_VERSION}\n\n    archive = base / f\'{GE_PROTON_VERSION}.tar.gz\'\n    if not archive.is_file() or _sha512(archive) != GE_PROTON_SHA512:\n        if archive.exists():\n            archive.unlink()\n        print(f\'GE-Proton: descargando {GE_PROTON_URL}\')\n        _download(GE_PROTON_URL, archive)\n    if _sha512(archive) != GE_PROTON_SHA512:\n        archive.unlink(missing_ok=True)\n        raise RuntimeError(\'El SHA-512 de GE-Proton no coincide con la suma oficial; \'\n                           \'descarga eliminada.\')\n    extract_dir = base / \'extract\'\n    if extract_dir.exists():\n        shutil.rmtree(extract_dir)\n    extract_dir.mkdir(parents=True)\n    with tarfile.open(archive) as source:\n        members = source.getmembers()\n        for member in members:\n            path = Path(member.name)\n            if path.is_absolute() or \'..\' in path.parts:\n                raise RuntimeError(\'Entrada no segura en el archivo de GE-Proton.\')\n            if member.issym() or member.islnk():\n                link = member.linkname\n                if link.startswith(\'/\'):\n                    raise RuntimeError(\'Enlace no seguro en el archivo de GE-Proton.\')\n                if member.issym():\n                    resolved = posixpath.normpath(\n                        posixpath.join(posixpath.dirname(member.name), link))\n                else:\n                    resolved = posixpath.normpath(link)\n                if resolved == \'..\' or resolved.startswith(\'../\') or resolved.startswith(\'/\'):\n                    raise RuntimeError(\'Enlace no seguro en el archivo de GE-Proton.\')\n                continue\n            if not (member.isfile() or member.isdir()):\n                raise RuntimeError(\'Tipo de entrada no admitido en GE-Proton.\')\n        for member in members:\n            source.extract(member, extract_dir)\n    roots = [entry for entry in extract_dir.iterdir() if entry.is_dir()]\n    if len(roots) != 1:\n        raise RuntimeError(\'El archivo de GE-Proton no tiene una única carpeta raíz.\')\n    if target.exists():\n        shutil.rmtree(target)\n    roots[0].rename(target)\n    shutil.rmtree(extract_dir, ignore_errors=True)\n    archive.unlink(missing_ok=True)\n    if not proton.is_file():\n        raise RuntimeError(\'GE-Proton no contiene el ejecutable proton.\')\n    proton.chmod(0o755)\n    for directory, directories, files in os.walk(target):\n        os.chown(directory, account.pw_uid, account.pw_gid)\n        for name in directories + files:\n            os.chown(Path(directory) / name, account.pw_uid, account.pw_gid,\n                     follow_symlinks=False)\n    result = {\'installed\': True, \'path\': str(target), \'version\': GE_PROTON_VERSION,\n              \'sha512\': GE_PROTON_SHA512}\n    (base / \'installation.json\').write_text(json.dumps(result, indent=2))\n    os.chown(base / \'installation.json\', account.pw_uid, account.pw_gid)\n    print(f\'GE-Proton instalado: {target}\')\n    print(\'En Steam: Configuración > Compatibilidad > Steam Play para todos los títulos > \'\n          + GE_PROTON_VERSION)\n    return result\n\n\ndef install_steam(session):\n    """Instalación explícita opcional, sin lanzar Steam ni modificar el estado global."""\n    if os.geteuid() != 0:\n        raise RuntimeError(\'Ejecutar la celda instaladora como root; Steam abrirá como cuenta normal.\')\n    required = (\'user\', \'home\', \'runtime\', \'display\', \'xauthority\', \'gpu_root\',\n                \'driver_version\', \'env\', \'run_launcher\')\n    missing = [key for key in required if key not in session]\n    if missing:\n        raise ValueError(\'Estado incompleto: \' + \', \'.join(missing))\n    account = pwd.getpwnam(session[\'user\'])\n    if account.pw_uid == 0:\n        raise ValueError(\'La cuenta de Steam no puede ser root.\')\n    home = Path(session[\'home\'])\n    runtime = Path(session[\'runtime\'])\n    runner = Path(session[\'run_launcher\'])\n    if not all(path.is_absolute() for path in (home, runtime, runner, Path(session[\'gpu_root\']))):\n        raise ValueError(\'El estado debe proporcionar rutas absolutas.\')\n    if not home.is_dir() or not runtime.is_dir() or not os.access(runner, os.X_OK):\n        raise RuntimeError(\'Preparar primero el escritorio y pcfree-run.\')\n    for key, value in {\'HOME\': str(home), \'USER\': session[\'user\'],\n                       \'DISPLAY\': session[\'display\'], \'XAUTHORITY\': session[\'xauthority\'],\n                       \'XDG_RUNTIME_DIR\': str(runtime)}.items():\n        if session[\'env\'].get(key) != value:\n            raise ValueError(f\'Estado y env no coinciden en {key}.\')\n    driver = session[\'driver_version\']\n    if not re.fullmatch(r\'\\d+\\.\\d+(?:\\.\\d+)?\', driver):\n        raise ValueError(\'Versión NVIDIA no válida.\')\n    active = _run([\'nvidia-smi\', \'--query-gpu=driver_version\', \'--format=csv,noheader\'],\n                  env={\'LD_LIBRARY_PATH\': session[\'env\'].get(\'LD_LIBRARY_PATH\', \'\')}, host_gpu=True)\n    if not active.splitlines() or any(v.strip() != driver for v in active.splitlines()):\n        raise RuntimeError(\'El controlador activo no coincide con session.json; no se cambiará.\')\n    options = session.get(\'steam\', {})\n    if not isinstance(options, dict):\n        raise ValueError(\'session[steam] debe ser un objeto.\')\n    bwrap = options.get(\'bwrap\')\n    if bwrap and (not Path(bwrap).is_absolute() or not os.access(bwrap, os.X_OK)):\n        raise ValueError(\'bwrap debe ser una ruta absoluta a un instalador revisado ya ejecutable.\')\n    base = home / \'.local/share/pcfree/steam\'\n    _user_directory(base, account)\n    log = base / \'install.log\'\n    _run([\'dpkg\', \'--add-architecture\', \'i386\'], log=log)\n    _run([\'apt-get\', \'update\', \'-qq\'], log=log)\n    packages = [\'ca-certificates\', \'dbus\', \'dbus-x11\', \'util-linux\',\n                \'lib32gcc-s1\', \'lib32stdc++6\']\n    # Ubuntu 24.04+ usa libasound2t64; no solicitar el paquete virtual libasound2.\n    asound = (\'libasound2t64\' if _run([\'apt-cache\', \'search\', \'^libasound2t64$\'])\n              else \'libasound2\')\n    for arch in TRIPLETS:\n        packages += [f\'{name}:{arch}\' for name in (\n            \'libc6\', \'libstdc++6\', \'libgcc-s1\', \'libgl1\', \'libegl1\', \'libgles2\',\n            \'libvulkan1\', \'libdrm2\', \'libx11-6\', \'libxext6\', \'libxrandr2\',\n            \'libxinerama1\', \'libxcursor1\', \'libxi6\', \'libxss1\', \'libxtst6\',\n            \'libxkbcommon-x11-0\', \'libudev1\', \'libdbus-1-3\', \'libpulse0\',\n            asound, \'libasound2-plugins\', \'libfontconfig1\', \'libfreetype6\')]\n    _safe_install(packages, log)\n    graphics, specs = _graphics(base, driver, log)\n    fake_root = Path(session.get(\'fake_udev_root\', runtime.parent / \'fake-udev\'))\n    if not fake_root.is_absolute():\n        raise ValueError(\'fake_udev_root debe ser una ruta absoluta.\')\n    gamepads = _gamepads(base, log, fake_root) if options.get(\'gamepads\', False) else {}\n    deb = base / \'steam_latest.deb\'\n    _download(STEAM_DEB, deb)\n    version = _run([\'dpkg-deb\', \'-f\', str(deb), \'Version\'], log=log)\n    if _run([\'dpkg-deb\', \'-f\', str(deb), \'Package\'], log=log) != \'steam-launcher\':\n        raise RuntimeError(\'La descarga oficial no contiene steam-launcher.\')\n    with tempfile.TemporaryDirectory(prefix=\'launcher-\', dir=base) as temporary:\n        extracted = Path(temporary) / \'root\'\n        _run([\'dpkg-deb\', \'-x\', str(deb), str(extracted)], log=log)\n        if (not (extracted / \'usr/lib/steam/bin_steam.sh\').is_file() or\n                not (extracted / \'usr/lib/steam/bootstraplinux_ubuntu12_32.tar.xz\').is_file()):\n            raise RuntimeError(\'Cambió el diseño del paquete oficial; revisar el bootstrap.\')\n        destination = base / \'launcher\'\n        if destination.exists():\n            destination.rename(base / (\'launcher-old-\' + datetime.datetime.now(datetime.timezone.utc)\n                                       .strftime(\'%Y%m%dT%H%M%S%fZ\')))\n        extracted.rename(destination)\n    quarantine = None\n    if options.get(\'repair_client\', False):\n        # No terminar procesos de una sesión activa, ni mover archivos en uso.\n        running = subprocess.run([\'pgrep\', \'-u\', str(account.pw_uid), \'-x\', \'steam\'],\n                                 stdout=subprocess.DEVNULL, check=False)\n        if running.returncode == 0:\n            raise RuntimeError(\'Cerrar Steam manualmente antes de repair_client; no se ha detenido.\')\n        quarantine = _quarantine_client(home, base)\n    # Su nombre debe ser steam: algunos bootstrap consultan basename($0).\n    client = destination / \'usr/lib/steam/steam\'\n    if not client.exists():\n        client.symlink_to(\'bin_steam.sh\')\n    launch = base / \'start-steam.sh\'\n    _owned_write(launch, _launch_script(client, graphics, gamepads, bwrap), account, True)\n    command = shlex.join([str(runner), str(launch)])\n    wrapper = home / \'.local/bin/pcfree-steam\'\n    _owned_write(wrapper, \'#!/bin/sh\\nexec \' + command + \' "$@"\\n\', account, True)\n    desktop = \'[Desktop Entry]\\nType=Application\\nName=Steam\\nComment=Cliente oficial (inicio manual)\\n\'\n    # Desktop Entry Exec usa comillas dobles, no el quoting de un shell.\n    quote = lambda value: \'"\' + str(value).replace(\'\\\\\', \'\\\\\\\\\').replace(\'"\', \'\\\\"\').replace(\'`\', \'\\\\`\').replace(\'$\', \'\\\\$\') + \'"\'\n    desktop += (\'Exec=\' + quote(runner) + \' \' + quote(launch) + \'\\n\'\n                \'Icon=\' + str(destination / \'usr/share/icons/hicolor/256x256/apps/steam.png\') + \'\\n\'\n                \'Terminal=false\\nCategories=Game;\\n\')\n    _owned_write(home / \'.local/share/applications/pcfree-steam.desktop\', desktop, account)\n    if (home / \'Desktop\').is_dir():\n        _owned_write(home / \'Desktop/Steam.desktop\', desktop, account, True)\n    # La cuenta puede abrir desde el menú; no confiar en metadatos GVFS root.\n    result = {\'installed\': True, \'login_validated\': False, \'version\': version,\n              \'command\': command, \'log\': str(log), \'driver_version\': driver,\n              \'nvidia_packages\': specs, \'gamepads\': bool(gamepads),\n              \'quarantine\': quarantine, \'bwrap\': bwrap}\n    _owned_write(base / \'installation.json\', json.dumps(result, indent=2), account)\n    # Todo el árbol privado debe ser modificable por la cuenta del cliente.\n    for directory, directories, files in os.walk(base):\n        os.chown(directory, account.pw_uid, account.pw_gid)\n        for name in directories + files:\n            os.chown(Path(directory) / name, account.pw_uid, account.pw_gid,\n                     follow_symlinks=False)\n    print(\'Steam oficial instalado; no se ha abierto. Abrir Steam desde el menú.\')\n    print(\'El primer inicio descarga el cliente. Login y juegos requieren prueba manual.\')\n    print(\'Si pressure-vessel rechaza namespaces, revisar proot-bwrap de Selkies; no desactivar seccomp globalmente.\')\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--session\', default=\'/tmp/pcfree-selkies/session.json\')\n    parser.add_argument(\'--gamepads\', action=\'store_true\', help=\'Compilar fake-udev oficial aislado (ambos ELF).\')\n    parser.add_argument(\'--gamepads-only\', action=\'store_true\', help=\'Instalar sólo mandos, sin descargar Steam.\')\n    parser.add_argument(\'--repair-client\', action=\'store_true\', help=\'Poner sólo binarios redistribuibles en cuarentena.\')\n    parser.add_argument(\'--ge-proton\', action=\'store_true\', help=\'Descargar GE-Proton fijado y verificado.\')\n    args = parser.parse_args()\n    session = json.loads(Path(args.session).read_text())\n    session[\'steam\'] = dict(session.get(\'steam\', {}))\n    if args.ge_proton:\n        install_ge_proton(session)\n        return\n    if args.gamepads:\n        session[\'steam\'][\'gamepads\'] = True\n    if args.repair_client:\n        session[\'steam\'][\'repair_client\'] = True\n    if args.gamepads_only:\n        install_gamepads(session)\n    else:\n        install_steam(session)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'compat-apps.py': '"""Optional applications. Install only; never start graphical applications."""\nimport hashlib\nimport fcntl\nimport json\nimport os\nimport platform\nimport pwd\nimport re\nimport shlex\nimport subprocess\nimport tarfile\nimport threading\nimport urllib.request\nimport urllib.parse\nimport zipfile\nfrom pathlib import Path\n\nBASE = Path(\'/tmp/pcfree-selkies\')\nOPTIONS = {\n    \'wine\': [\'wine64\', \'wine32:i386\', \'winetricks\'],\n    \'lutris\': [\'lutris\'],\n    \'java\': [\'openjdk-21-jre\'],\n    \'gaming-libs\': [\'libsdl2-2.0-0\', \'libsdl2-2.0-0:i386\',\n                    \'libopenal1\', \'libopenal1:i386\', \'libvulkan1:i386\',\n                    \'libasound2t64:i386\', \'libpulse0:i386\', \'libstdc++6:i386\'],\n    \'deb-installer\': [\'gdebi\'],\n    \'blender\': [\'blender\'],\n}\nAPPIMAGES = {\n    \'heroic\': (\'Heroic-Games-Launcher/HeroicGamesLauncher\', \'Heroic\'),\n    \'prism\': (\'PrismLauncher/PrismLauncher\', \'Prism Launcher\'),\n}\nEXPERIMENTAL = {\'godot\', \'vinegar\', \'roblox\'}\n\n\ndef command(argv, log, timeout=600):\n    env = {**os.environ, \'DEBIAN_FRONTEND\': \'noninteractive\', \'LC_ALL\': \'C\'}\n    for key in (\'LD_PRELOAD\', \'LD_LIBRARY_PATH\', \'PYTHONPATH\', \'PYTHONHOME\'):\n        env.pop(key, None)\n    if argv[:2] == [\'apt-get\', \'install\']:\n        plan = subprocess.check_output([\'apt-get\', \'-s\', *argv[1:]], env=env,\n                                       text=True, stderr=subprocess.STDOUT, timeout=120)\n        for line in plan.splitlines():\n            if line.startswith(\'Remv \'):\n                raise RuntimeError(\'APT propone eliminar paquetes; se canceló la instalación.\')\n            if line.startswith(\'Inst \'):\n                package = line.split()[1].split(\':\')[0]\n                if \'nvidia\' in package or package.startswith((\'linux-image\', \'linux-modules\', \'linux-headers\')):\n                    raise RuntimeError(\'APT propone modificar GPU/kernel; se canceló la instalación.\')\n        origins = subprocess.check_output([\'apt-get\', \'--print-uris\', \'--download-only\', *argv[1:]],\n                                          env=env, text=True, stderr=subprocess.STDOUT, timeout=120)\n        for uri in re.findall(r"^\'([^\']+)\'", origins, re.MULTILINE):\n            parsed = urllib.parse.urlparse(uri)\n            host = parsed.hostname or \'\'\n            ubuntu = host in {\'archive.ubuntu.com\', \'security.ubuntu.com\', \'ports.ubuntu.com\'} or host.endswith(\'.archive.ubuntu.com\')\n            nvidia = host == \'developer.download.nvidia.com\' and parsed.scheme == \'https\' and parsed.path.startswith(\'/compute/cuda/repos/ubuntu2404/x86_64/\')\n            if parsed.scheme not in {\'http\', \'https\'} or not (ubuntu or nvidia) or parsed.username or parsed.password:\n                raise RuntimeError(\'La instalación requiere paquetes de repositorios oficiales Ubuntu/NVIDIA.\')\n    with log.open(\'a\') as stream:\n        result = subprocess.run(argv, stdout=stream, stderr=subprocess.STDOUT,\n                                env=env,\n                                timeout=timeout)\n    if result.returncode:\n        raise RuntimeError(f\'Falló {shlex.join(argv)}; revisar {log}\\n\'\n                           + log.read_text(errors=\'replace\')[-3000:])\n\n\ndef shortcut(session, name, executable, category=\'Game\', icon=\'applications-games\'):\n    home = Path(session[\'home\'])\n    desktop = home / \'Desktop\'\n    desktop.mkdir(parents=True, exist_ok=True)\n    path = desktop / (name.replace(\' \', \'-\') + \'.desktop\')\n    # Quote Desktop Entry arguments, independently of shell quoting.\n    def quote(arg):\n        return \'"\' + str(arg).replace(\'\\\\\', \'\\\\\\\\\').replace(\'"\', \'\\\\"\').replace(\'`\', \'\\\\`\').replace(\'$\', \'\\\\$\') + \'"\'\n    launcher = session.get(\'run_launcher\', \'/usr/local/bin/pcfree-run\')\n    path.write_text(\'[Desktop Entry]\\nType=Application\\nVersion=1.0\\n\'\n                    f\'Name={name}\\nExec={quote(launcher)} {quote(executable)}\\n\'\n                    f\'Icon={icon}\\nTerminal=false\\nCategories={category};\\n\')\n    path.chmod(0o755)\n    account = pwd.getpwnam(session[\'user\'])\n    os.chown(path, account.pw_uid, account.pw_gid)\n    env = session[\'env\']\n    subprocess.run([\'runuser\', \'-u\', session[\'user\'], \'--\', \'env\',\n                    *[f\'{k}={v}\' for k, v in env.items()],\n                    \'gio\', \'set\', str(path), \'metadata::trusted\', \'true\'],\n                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n\n\ndef official_appimage(option, session):\n    repository, label = APPIMAGES[option]\n    directory = BASE / \'apps\' / option\n    directory.mkdir(parents=True, exist_ok=True)\n    request = urllib.request.Request(\n        f\'https://api.github.com/repos/{repository}/releases/latest\',\n        headers={\'User-Agent\': \'PC-Free-T4\'})\n    with urllib.request.urlopen(request, timeout=45) as response:\n        release = json.load(response)\n    matches = [a for a in release.get(\'assets\', [])\n               if a[\'name\'].lower().endswith(\'.appimage\')\n               and (\'x86_64\' in a[\'name\'].lower() or \'amd64\' in a[\'name\'].lower())\n               and \'portable\' not in a[\'name\'].lower()]\n    if len(matches) != 1:\n        raise RuntimeError(f\'No se pudo elegir el AppImage oficial de {repository}: \'\n                           + \', \'.join(a[\'name\'] for a in matches))\n    asset = matches[0]\n    url = asset[\'browser_download_url\']\n    if not url.startswith(f\'https://github.com/{repository}/releases/download/\'):\n        raise RuntimeError(\'El enlace del paquete no pertenece al repositorio oficial.\')\n    image = directory / Path(asset[\'name\']).name\n    if not image.exists():\n        temporary = image.with_suffix(image.suffix + \'.part\')\n        urllib.request.urlretrieve(url, temporary)\n        digest = asset.get(\'digest\') or \'\'\n        if digest.startswith(\'sha256:\'):\n            sha = hashlib.sha256()\n            with temporary.open(\'rb\') as source:\n                for block in iter(lambda: source.read(1024 * 1024), b\'\'):\n                    sha.update(block)\n            if sha.hexdigest() != digest.partition(\':\')[2]:\n                temporary.unlink()\n                raise RuntimeError(\'La descarga no coincide con el SHA-256 oficial.\')\n        temporary.replace(image)\n    image.chmod(0o755)\n    extracted = directory / \'squashfs-root\'\n    log = directory / \'install.log\'\n    if not (extracted / \'AppRun\').exists():\n        # Native extraction avoids FUSE. Runs only the official installer runtime.\n        with log.open(\'w\') as stream:\n            result = subprocess.run([str(image), \'--appimage-extract\'], cwd=directory,\n                                    stdout=stream, stderr=subprocess.STDOUT, timeout=180)\n        if result.returncode or not (extracted / \'AppRun\').exists():\n            raise RuntimeError(f\'No se pudo extraer el AppImage; revisar {log}.\')\n    shortcut(session, label, str(extracted / \'AppRun\'))\n    print(f\'{label} {release["tag_name"]}: instalado, sin abrir. \'\n          \'Su sandbox y los juegos requieren pruebas en este entorno.\')\n\n\ndef install_godot(session):\n    directory = BASE / \'apps/godot\'\n    directory.mkdir(parents=True, exist_ok=True)\n    repository = \'godotengine/godot\'\n    request = urllib.request.Request(f\'https://api.github.com/repos/{repository}/releases/latest\',\n                                     headers={\'User-Agent\': \'PC-Free-T4\'})\n    with urllib.request.urlopen(request, timeout=45) as response:\n        release = json.load(response)\n    candidates = [a for a in release[\'assets\']\n                  if a[\'name\'].endswith(\'_linux.x86_64.zip\') and \'mono\' not in a[\'name\'].lower()]\n    if len(candidates) != 1:\n        raise RuntimeError(\'No se encontró un binario Linux x86_64 de Godot en su publicación oficial.\')\n    asset = candidates[0]\n    if not asset[\'browser_download_url\'].startswith(f\'https://github.com/{repository}/releases/download/\'):\n        raise RuntimeError(\'Origen de Godot inesperado.\')\n    archive = directory / Path(asset[\'name\']).name\n    if not archive.exists():\n        urllib.request.urlretrieve(asset[\'browser_download_url\'], archive)\n    with zipfile.ZipFile(archive) as bundle:\n        entries = [item for item in bundle.infolist() if not item.is_dir()]\n        if len(entries) != 1 or \'/\' in entries[0].filename or \'\\\\\' in entries[0].filename:\n            raise RuntimeError(\'Estructura del paquete Godot inesperada.\')\n        executable = directory / entries[0].filename\n        with bundle.open(entries[0]) as source, executable.open(\'wb\') as destination:\n            import shutil\n            shutil.copyfileobj(source, destination)\n    executable.chmod(0o755)\n    shortcut(session, \'Godot\', str(executable), \'Development\', \'applications-development\')\n    print(f\'Godot {release["tag_name"]}: instalado, sin abrir. \'\n          \'Vulkan NVIDIA se prepara en el arranque; Compatibility usa OpenGL si el proyecto lo requiere.\')\n\n\ndef install_vinegar(session):\n    """Official native source build; Studio/Wine downloads start only when opened."""\n    directory = BASE / \'apps/vinegar\'\n    directory.mkdir(parents=True, exist_ok=True)\n    log = directory / \'install.log\'\n    packages = [\'build-essential\', \'git\', \'make\', \'pkg-config\', \'gettext\',\n                \'libgtk-4-dev\', \'libadwaita-1-dev\', \'libvulkan-dev\', \'libglib2.0-dev-bin\']\n    command([\'apt-get\', \'update\', \'-qq\'], log)\n    command([\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', *packages], log)\n    request = urllib.request.Request(\'https://go.dev/dl/?mode=json\', headers={\'User-Agent\': \'PC-Free-T4\'})\n    with urllib.request.urlopen(request, timeout=45) as response:\n        releases = json.load(response)\n    choices = [release for release in releases if release.get(\'stable\')\n               and tuple(int(n) for n in release[\'version\'][2:].split(\'.\')[:2]) >= (1, 26)]\n    if not choices:\n        raise RuntimeError(\'Vinegar 1.9.4 requiere Go >=1.26; no se encontró un compilador estable oficial.\')\n    toolchain = next(f for f in choices[0][\'files\'] if f[\'os\'] == \'linux\'\n                     and f[\'arch\'] == \'amd64\' and f[\'kind\'] == \'archive\')\n    tools = directory / \'toolchain\'\n    tools.mkdir(exist_ok=True)\n    archive = tools / toolchain[\'filename\']\n    if not (tools / \'go/bin/go\').exists():\n        urllib.request.urlretrieve(\'https://go.dev/dl/\' + toolchain[\'filename\'], archive)\n        sha = hashlib.sha256()\n        with archive.open(\'rb\') as source:\n            for block in iter(lambda: source.read(1024 * 1024), b\'\'):\n                sha.update(block)\n        if sha.hexdigest() != toolchain[\'sha256\']:\n            archive.unlink()\n            raise RuntimeError(\'SHA-256 del compilador Go no válido.\')\n        with tarfile.open(archive) as bundle:\n            bundle.extractall(tools, filter=\'data\')\n        archive.unlink()\n    source = directory / \'source\'\n    if not source.exists():\n        command([\'git\', \'clone\', \'--branch\', \'v1.9.4\', \'--depth\', \'1\',\n                 \'https://github.com/vinegarhq/vinegar.git\', str(source)], log)\n    prefix = directory / \'installed\'\n    env = {**os.environ, \'PATH\': str(tools / \'go/bin\') + \':\' + os.environ.get(\'PATH\', \'\')}\n    env.pop(\'LD_PRELOAD\', None)\n    for argv in [[\'make\', \'-j2\', \'PREFIX=\' + str(prefix)],\n                 [\'make\', \'install\', \'PREFIX=\' + str(prefix)]]:\n        with log.open(\'a\') as stream:\n            result = subprocess.run(argv, cwd=source, env=env, stdout=stream,\n                                    stderr=subprocess.STDOUT, timeout=1200)\n        if result.returncode:\n            raise RuntimeError(f\'Falló la compilación de Vinegar; revisar {log}.\')\n    shortcut(session, \'Roblox Studio - Vinegar\', str(prefix / \'bin/vinegar\'),\n             \'Development\', str(prefix / \'share/icons/hicolor/scalable/apps/org.vinegarhq.Vinegar.svg\'))\n    print(\'Vinegar 1.9.4: instalado desde el código oficial, sin abrir. \'\n          \'Wine y Roblox Studio se descargan al abrirlo. Experimental: Studio no validado en Colab; \'\n          \'Vinegar no ejecuta Roblox Player.\')\n\n\ndef sober_requirements(cpuinfo=None, machine=None, kernel=None):\n    """Validate hardware, not emulate CPU features or bypass kernel restrictions."""\n    machine = machine or platform.machine()\n    kernel = kernel or platform.release()\n    if machine not in {\'x86_64\', \'amd64\'}:\n        raise RuntimeError(\'Sober requiere CPU x86-64; ARM no está soportado.\')\n    match = re.match(r\'(\\d+)\\.(\\d+)\', kernel)\n    if not match or tuple(map(int, match.groups())) < (5, 11):\n        raise RuntimeError(\'Sober requiere kernel Linux >=5.11.\')\n    cpuinfo = Path(\'/proc/cpuinfo\').read_text() if cpuinfo is None else cpuinfo\n    flags = [set(line.partition(\':\')[2].split()) for line in cpuinfo.splitlines()\n             if line.partition(\':\')[0].strip() == \'flags\']\n    if not flags or not all(\'sse4_1\' in item for item in flags):\n        raise RuntimeError(\'La CPU asignada no ofrece SSE4.1. No se puede añadir con un paquete; usar otra VM.\')\n    return {\'sse4_1\': True, \'sse4_2\': all(\'sse4_2\' in item for item in flags), \'kernel\': kernel}\n\n\ndef ensure_flatpak_system_bus(log):\n    """Flatpak consults parental-control policy even for --user installs."""\n    def ready():\n        result = subprocess.run([\'dbus-send\', \'--system\', \'--print-reply\',\n                                 \'--dest=org.freedesktop.DBus\', \'/org/freedesktop/DBus\',\n                                 \'org.freedesktop.DBus.ListNames\'], capture_output=True,\n                                text=True, timeout=5)\n        return result.returncode == 0\n    with (BASE / \'flatpak-system-bus.lock\').open(\'w\') as lock:\n        fcntl.flock(lock, fcntl.LOCK_EX)\n        if ready():\n            return\n        if Path(\'/run/dbus/system_bus_socket\').exists():\n            raise RuntimeError(\'El bus de sistema existente no responde; no se reemplaza automáticamente.\')\n        Path(\'/run/dbus\').mkdir(mode=0o755, parents=True, exist_ok=True)\n        command([\'dbus-daemon\', \'--system\', \'--fork\', \'--nopidfile\'], log, timeout=15)\n        if not ready():\n            raise RuntimeError(\'D-Bus de sistema no disponible para Flatpak; revisar install-roblox.log.\')\n\n\ndef install_roblox(session):\n    """Sober\'s official Flatpak only where the required kernel sandbox is available."""\n    requirements = sober_requirements()\n    result = subprocess.run([\'runuser\', \'-u\', session[\'user\'], \'--\', \'unshare\',\n                             \'--user\', \'--map-root-user\', \'--mount\', \'--pid\', \'--fork\', \'true\'],\n                            capture_output=True, text=True, timeout=10)\n    if result.returncode:\n        raise RuntimeError(\'Roblox Player necesita Sober/Flatpak, pero esta VM no permite \'\n                           \'el sandbox de namespaces. No se instala un cliente inoperante ni se \'\n                           \'desactiva el sandbox. Fuente: https://sober.vinegarhq.org/\')\n    log = BASE / \'install-roblox.log\'\n    command([\'apt-get\', \'update\', \'-qq\'], log)\n    command([\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', \'flatpak\'], log)\n    try:\n        # unshare alone can succeed while this required proc mount is denied.\n        # Probe a real Bubblewrap sandbox before downloading large runtimes.\n        command([\'runuser\', \'-u\', session[\'user\'], \'--\', \'bwrap\',\n                 \'--unshare-user\', \'--unshare-pid\', \'--ro-bind\', \'/\', \'/\',\n                 \'--proc\', \'/proc\', \'--dev\', \'/dev\', \'--\', \'/usr/bin/true\'], log, timeout=15)\n    except RuntimeError as error:\n        raise RuntimeError(\'Flatpak está instalado, pero esta VM no permite el sandbox \'\n                           \'Bubblewrap con /proc. No se descargan los runtimes ni se fuerza \'\n                           \'Roblox sin sandbox. Revisar install-roblox.log; usar un entorno \'\n                           \'Linux que permita Flatpak.\') from error\n    ensure_flatpak_system_bus(log)\n    version = session.get(\'driver_version\', \'\')\n    if not re.fullmatch(r\'\\d+\\.\\d+(?:\\.\\d+)?\', version):\n        raise RuntimeError(\'No se conoce la versión NVIDIA exacta para Flatpak; no usar renderizado por software.\')\n    driver = \'nvidia-\' + version.replace(\'.\', \'-\')\n    # Flatpak cannot use the host\'s isolated LD_LIBRARY_PATH/ICD. It needs its\n    # own GL extension matching the loaded NVIDIA kernel driver exactly.\n    env = {k: v for k, v in session[\'env\'].items()\n           if k not in {\'LD_PRELOAD\', \'LD_LIBRARY_PATH\', \'VK_DRIVER_FILES\', \'VK_ICD_FILENAMES\', \'__EGL_VENDOR_LIBRARY_FILENAMES\', \'SELKIES_INTERPOSER\'}}\n    env[\'FLATPAK_GL_DRIVERS\'] = driver + \':default\'\n    prefix = [\'runuser\', \'-u\', session[\'user\'], \'--\', \'env\',\n              \'-u\', \'LD_PRELOAD\', \'-u\', \'LD_LIBRARY_PATH\', \'-u\', \'VK_DRIVER_FILES\', \'-u\', \'VK_ICD_FILENAMES\',\n              \'-u\', \'__EGL_VENDOR_LIBRARY_FILENAMES\',\n              *[f\'{k}={v}\' for k, v in env.items()], \'flatpak\', \'--user\']\n    command([*prefix, \'remote-add\', \'--if-not-exists\', \'flathub\',\n             \'https://flathub.org/repo/flathub.flatpakrepo\'], log)\n    command([*prefix, \'install\', \'-y\', \'--noninteractive\', \'flathub\',\n             f\'org.freedesktop.Platform.GL.{driver}//1.4\'], log, timeout=1200)\n    command([*prefix, \'install\', \'-y\', \'--noninteractive\', \'flathub\',\n             \'org.vinegarhq.Sober\'], log, timeout=1200)\n    # Runs only /usr/bin/true inside the real app sandbox, never opens Roblox.\n    command([*prefix, \'run\', \'--command=true\', \'org.vinegarhq.Sober\'], log, timeout=30)\n    wrapper = BASE / \'apps/sober-launch\'\n    wrapper.parent.mkdir(parents=True, exist_ok=True)\n    wrapper.write_text(\'#!/bin/sh\\nexec env -u LD_PRELOAD -u LD_LIBRARY_PATH \'\n                       \'-u VK_DRIVER_FILES -u VK_ICD_FILENAMES -u __EGL_VENDOR_LIBRARY_FILENAMES \'\n                       f\'FLATPAK_GL_DRIVERS={shlex.quote(driver + ":default")} \'\n                       \'flatpak run org.vinegarhq.Sober "$@"\\n\')\n    wrapper.chmod(0o755)\n    shortcut(session, \'Roblox - Sober\', str(wrapper))\n    (BASE / \'sober-requirements.json\').write_text(json.dumps({**requirements, \'flatpak_gl_driver\': driver,\n        \'sandbox_smoke_test\': True, \'game_validated\': False}, indent=2))\n    print(\'Sober instalado, sin abrir. Proyecto comunitario no respaldado por Roblox; \'\n          \'inicio de sesión, juego y controladores todavía requieren pruebas.\')\n\n\n_roblox_thread = None\n\n\ndef roblox_background(session):\n    """Keep desktop access available while downloading the Flatpak runtimes."""\n    global _roblox_thread\n    if _roblox_thread is not None and _roblox_thread.is_alive():\n        print(\'Sober: instalación ya en curso.\')\n        return _roblox_thread\n    def worker():\n        status = BASE / \'sober-status.txt\'\n        try:\n            with (BASE / \'sober-install.lock\').open(\'w\') as lock:\n                try:\n                    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n                except BlockingIOError:\n                    print(\'Sober: otra instalación está en curso.\', flush=True)\n                    return\n                install_roblox(session)\n                status.write_text(\'Instalado; abrir Roblox - Sober. Roblox se instala en el primer inicio.\\n\')\n        except Exception as error:\n            status.write_text(\'Instalación incompleta: \' + str(error) + \'\\n\')\n            print(\'Sober: \' + str(error), flush=True)\n    BASE.mkdir(exist_ok=True)\n    (BASE / \'sober-status.txt\').write_text(\'Instalando Flatpak, runtime NVIDIA y Sober en segundo plano.\\n\')\n    _roblox_thread = threading.Thread(target=worker, name=\'pcfree-sober\', daemon=True)\n    _roblox_thread.start()\n    print(\'Sober/Flatpak: descarga en segundo plano; el escritorio ya está disponible.\', flush=True)\n    return _roblox_thread\n\n\ndef install_app(option, session):\n    if os.geteuid() != 0:\n        raise RuntimeError(\'Requiere ejecutar desde Colab.\')\n    if option not in OPTIONS and option not in APPIMAGES and option not in EXPERIMENTAL:\n        raise ValueError(\'Opción desconocida: \' + option)\n    BASE.mkdir(exist_ok=True)\n    log = BASE / f\'install-{option}.log\'\n    if option in EXPERIMENTAL:\n        {\'godot\': install_godot, \'vinegar\': install_vinegar, \'roblox\': install_roblox}[option](session)\n        return\n    if option in APPIMAGES:\n        if option == \'prism\':\n            install_app(\'java\', session)\n        official_appimage(option, session)\n        return\n    packages = OPTIONS[option]\n    if any(name.endswith(\':i386\') for name in packages):\n        command([\'dpkg\', \'--add-architecture\', \'i386\'], log)\n    command([\'apt-get\', \'update\', \'-qq\'], log)\n    command([\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', *packages], log)\n    if option == \'wine\':\n        shortcut(session, \'Wine Configuration\', \'/usr/bin/winecfg\', \'Settings\', \'wine\')\n        shortcut(session, \'Winetricks\', \'/usr/bin/winetricks\', \'Settings\', \'wine\')\n    elif option == \'lutris\':\n        shortcut(session, \'Lutris\', \'/usr/bin/lutris\', icon=\'lutris\')\n    elif option == \'deb-installer\':\n        shortcut(session, \'Instalador DEB\', \'/usr/bin/gdebi-gtk\', \'System\', \'package-x-generic\')\n    elif option == \'blender\':\n        shortcut(session, \'Blender\', \'/usr/bin/blender\', \'Graphics\', \'blender\')\n    print(f\'{option}: instalado, sin abrir aplicaciones. Registro: {log}\')\n    if option == \'wine\':\n        print(\'Wine no equivale a Windows. DXVK/VKD3D requieren Vulkan NVIDIA \'\n              \'verificado y bibliotecas NVIDIA de 32/64 bits; usa la celda de Steam \'\n              \'para preparar el soporte multiarch. No se instalan runners sin verificar.\')\n\n\nif __name__ == \'__main__\':\n    import argparse\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'option\', choices=sorted([*OPTIONS, *APPIMAGES, *EXPERIMENTAL]))\n    parser.add_argument(\'--session\', default=str(BASE / \'session.json\'))\n    args = parser.parse_args()\n    install_app(args.option, json.loads(Path(args.session).read_text()))\n', '10-export-worlds.py': '# Explicit, manual export. Never copies launcher accounts, browser profiles,\n# Steam, RustDesk, tokens, or the complete application home directory.\nimport zipfile, hashlib, json, datetime\nassert \'browser_env\' in globals(), \'Primero preparar la cuenta de aplicaciones.\'\napp_home=Path(browser_env[\'HOME\']).resolve()\nworld_candidates=[]\nfor level in app_home.rglob(\'level.dat\'):\n    if level.is_symlink():\n        continue\n    world=level.parent.resolve()\n    if world.is_relative_to(app_home) and world.is_dir():\n        world_candidates.append(world)\nworld_candidates=sorted(set(world_candidates))\nprint(\'Mundos encontrados; seleccionar explícitamente los deseados:\')\nfor index,world in enumerate(world_candidates):\n    print(index,world)\n\n# Edit these values only after "Guardar y salir" in Minecraft.\nCONFIRM_GAME_SAVED_AND_CLOSED=False\nSELECTED_WORLD_INDICES=[]\nif not CONFIRM_GAME_SAVED_AND_CLOSED or not SELECTED_WORLD_INDICES:\n    print(\'Sin respaldo todavía. Guardar y salir del mundo, editar confirmación e índices, y repetir solo la operación.\')\nelse:\n    assert all(isinstance(i,int) and 0<=i<len(world_candidates) for i in SELECTED_WORLD_INDICES)\n    stamp=datetime.datetime.now(datetime.timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')\n    destination=Path(\'/tmp\')/(\'PC-Free-worlds-\'+stamp+\'.zip\')\n    total=0\n    manifest={\'format\':\'pc-free-worlds-v1\',\'created_utc\':stamp,\'files\':[]}\n    with zipfile.ZipFile(destination,\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n        for index in sorted(set(SELECTED_WORLD_INDICES)):\n            world=world_candidates[index]\n            for source in sorted(world.rglob(\'*\')):\n                if source.is_symlink() or not source.is_file() or source.name==\'session.lock\':\n                    continue\n                assert source.resolve().is_relative_to(world)\n                size=source.stat().st_size\n                total+=size\n                assert total<=2*1024**3, \'Respaldo mayor de 2 GiB; preparar exportación específica.\'\n                name=\'worlds/\'+str(index)+\'-\'+world.name+\'/\'+source.relative_to(world).as_posix()\n                digest=hashlib.file_digest(source.open(\'rb\'),\'sha256\').hexdigest()\n                archive.write(source,name)\n                manifest[\'files\'].append({\'path\':name,\'size\':size,\'sha256\':digest})\n        archive.writestr(\'manifest.json\',json.dumps(manifest,ensure_ascii=False,indent=2))\n    print(\'Respaldo creado:\',destination.name,\'| Bytes:\',destination.stat().st_size)\n    print(\'Solo mundos. Para mundos con mods, conservar también su lista/versiones de mods por separado.\')\n    from google.colab import files\n    files.download(str(destination))\n', '11-restore-worlds.py': "# Upload a previously exported ZIP without mounting Drive in the desktop.\nimport zipfile, json, hashlib, stat, datetime\nfrom google.colab import files\nfrom pathlib import PurePosixPath\nRESTORE_SAVES_DIRECTORY='' # Set the exact saves directory for this Minecraft instance.\nassert RESTORE_SAVES_DIRECTORY, 'Indicar primero la carpeta saves de la instancia de Minecraft.'\ntarget=Path(RESTORE_SAVES_DIRECTORY).expanduser().resolve()\napp_home=Path(browser_env['HOME']).resolve()\nassert target.is_relative_to(app_home), 'Restaurar solo dentro del home de aplicaciones.'\nassert target.name=='saves', 'La carpeta de destino debe llamarse saves.'\nuploaded=files.upload()\nassert len(uploaded)==1, 'Subir solamente un respaldo ZIP por intento.'\nfilename=next(iter(uploaded))\narchive_path=Path(filename)\nassert archive_path.suffix.lower()=='.zip'\nwith zipfile.ZipFile(archive_path) as archive:\n    infos=archive.infolist()\n    assert len({i.filename for i in infos})==len(infos), 'ZIP con rutas repetidas.'\n    assert sum(i.file_size for i in infos)<=2*1024**3, 'Respaldo demasiado grande.'\n    manifest=json.loads(archive.read('manifest.json'))\n    assert manifest['format']=='pc-free-worlds-v1'\n    expected={item['path']:item for item in manifest['files']}\n    assert set(expected)=={i.filename for i in infos if i.filename!='manifest.json' and not i.is_dir()}\n    for info in infos:\n        path=PurePosixPath(info.filename)\n        assert not path.is_absolute() and '..' not in path.parts and '\\\\' not in info.filename\n        assert not stat.S_ISLNK(info.external_attr>>16), 'No admitir enlaces simbólicos.'\n        if info.filename=='manifest.json' or info.is_dir():\n            continue\n        assert len(path.parts)>=3 and path.parts[0]=='worlds'\n        item=expected[info.filename]\n        assert info.file_size==item['size']\n        with archive.open(info) as source:\n            assert hashlib.file_digest(source,'sha256').hexdigest()==item['sha256'], 'Integridad incorrecta.'\n    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')\n    target.mkdir(parents=True,exist_ok=True)\n    restored={}\n    for item in manifest['files']:\n        path=PurePosixPath(item['path'])\n        name=path.parts[1]\n        if name not in restored:\n            directory=target/(name+'-restored-'+stamp)\n            directory.mkdir(exist_ok=False)\n            restored[name]=directory\n        output=restored[name].joinpath(*path.parts[2:])\n        assert output.resolve().is_relative_to(restored[name].resolve())\n        output.parent.mkdir(parents=True,exist_ok=True)\n        with archive.open(item['path']) as source, output.open('xb') as dest:\n            shutil.copyfileobj(source,dest)\n        account=pwd.getpwnam('pcfree-browser')\n        os.chown(output,account.pw_uid,account.pw_gid)\n    account=pwd.getpwnam('pcfree-browser')\n    for directory in restored.values():\n        for d in [directory,*[p for p in directory.rglob('*') if p.is_dir()]]:\n            os.chown(d,account.pw_uid,account.pw_gid)\n    print('Mundos restaurados en carpetas nuevas; no se sobrescribieron mundos existentes.')\n    print(*restored.values(),sep='\\n')\n"}
import os, pwd, shutil
from pathlib import Path
required_sources = ('selkies-desktop.py', 'compat-steam.py', 'compat-apps.py')
if not isinstance(SOURCES, dict):
    raise RuntimeError('La carga de fuentes no produjo un diccionario SOURCES.')
missing_sources = [name for name in required_sources if not isinstance(SOURCES.get(name), str)]
if missing_sources:
    raise RuntimeError('Faltan fuentes internas: ' + ', '.join(missing_sources))
print(f"PC-Free-T4: {len(SOURCES)} fuentes internas cargadas; compilando núcleo…", flush=True)
_core = {'__name__': 'pcfree_core'}
exec(compile(SOURCES['selkies-desktop.py'], 'selkies-desktop.py', 'exec'), _core)
if not callable(_core.get('bootstrap')):
    raise RuntimeError('El núcleo se compiló, pero no expone bootstrap(); detener el inicio.')
def optional_module(name):
    if name not in SOURCES or not isinstance(SOURCES[name], str):
        raise RuntimeError(f'No existe la fuente interna solicitada: {name}')
    namespace = {'__name__': 'pcfree_optional'}
    exec(compile(SOURCES[name], name, 'exec'), namespace)
    return namespace
print("PC-Free-T4: NÚCLEO LISTO [OK] — bootstrap() disponible. Aún no se ha iniciado Selkies.", flush=True)

In [ ]:


# @title Preparación del escritorio (Vulkan + i386 + caché + accesos)
# @markdown Aplica los parches del núcleo antes de iniciar.

if not isinstance(globals().get("SOURCES"), dict) or "selkies-desktop.py" not in SOURCES:
    raise RuntimeError("Requiere 'Cargar núcleo interno PC-Free-T4'.")

src = SOURCES["selkies-desktop.py"]

def once(old, new, label):
    global src
    if new in src:
        print(f"[OK] {label}: ya aplicado.")
        return
    count = src.count(old)
    if count != 1:
        raise RuntimeError(f"{label}: {count} coincidencias; no se aplicó.")
    src = src.replace(old, new, 1)
    print(f"[OK] {label} aplicado.")

# ─── 1. Vulkan: preferir libGLX_nvidia ───
old_vulkan = "    library = egl_library if egl_library.is_file() else glx_library\n"
new_vulkan = "    library = glx_library if glx_library.is_file() else egl_library\n"
if new_vulkan not in src:
    if src.count(old_vulkan) != 1:
        raise RuntimeError(f"Parche Vulkan: {src.count(old_vulkan)} coincidencias.")
    src = src.replace(old_vulkan, new_vulkan, 1)
    print("[OK] Vulkan prefiere libGLX_nvidia.")
else:
    print("[OK] Vulkan ya parcheado.")

# ─── 2. Caché de shaders y entorno de juegos ───
once(
"""    keep = ('DISPLAY', 'XAUTHORITY', 'HOME', 'USER', 'LOGNAME', 'XDG_RUNTIME_DIR',
            'PULSE_SERVER', 'PULSE_COOKIE', 'LD_LIBRARY_PATH', '__GLX_VENDOR_LIBRARY_NAME',
            'SELKIES_INTERPOSER', 'DBUS_SESSION_BUS_ADDRESS')""",
"""    shader_cache = home / '.cache' / 'nvidia-shader'
    shader_cache.mkdir(parents=True, exist_ok=True)
    os.chown(shader_cache.parent, account.pw_uid, account.pw_gid)
    os.chown(shader_cache, account.pw_uid, account.pw_gid)
    env['__GL_MaxFramesAllowed'] = '1'
    env['__GL_SYNC_TO_VBLANK'] = '0'
    env['__GL_SHADER_DISK_CACHE'] = '1'
    env['__GL_SHADER_DISK_CACHE_SKIP_CLEANUP'] = '1'
    env['__GL_SHADER_DISK_CACHE_PATH'] = str(shader_cache)
    env['DXVK_STATE_CACHE'] = '1'
    env['DXVK_LOG_LEVEL'] = 'none'
    env['WINEDEBUG'] = '-all'
    keep = ('DISPLAY', 'XAUTHORITY', 'HOME', 'USER', 'LOGNAME', 'XDG_RUNTIME_DIR',
            'PULSE_SERVER', 'PULSE_COOKIE', 'LD_LIBRARY_PATH', '__GLX_VENDOR_LIBRARY_NAME',
            'SELKIES_INTERPOSER', 'DBUS_SESSION_BUS_ADDRESS',
            '__GL_MaxFramesAllowed', '__GL_SYNC_TO_VBLANK',
            '__GL_SHADER_DISK_CACHE', '__GL_SHADER_DISK_CACHE_SKIP_CLEANUP',
            '__GL_SHADER_DISK_CACHE_PATH', 'DXVK_STATE_CACHE', 'DXVK_LOG_LEVEL', 'WINEDEBUG')""",
"entorno de juegos",
)

# ─── 3. Xorg: HardDPMS false ───
once(
""" Option "AllowEmptyInitialConfiguration" "true"
EndSection""",
""" Option "AllowEmptyInitialConfiguration" "true"
 Option "HardDPMS" "false"
EndSection""",
"xorg",
)

# ─── 4. xset: ratón sin aceleración ───
once(
"""            wait_for(desktop_ready, 'XFCE', [desktop])""",
"""            wait_for(desktop_ready, 'XFCE', [desktop])
            with contextlib.suppress(Exception):
                capture(as_user(['xset', 'm', '0', '0']), env=env, timeout=5)
                capture(as_user(['xset', 'r', 'rate', '200', '30']), env=env, timeout=5)""",
"xset",
)

# ─── 5. Selkies: configuración de stream de baja latencia ───
once(
"""                              SELKIES_RATE_CONTROL_MODE='cbr')""",
"""                              SELKIES_RATE_CONTROL_MODE='cbr',
                              SELKIES_USE_PAINT_OVER_QUALITY='false|locked',
                              SELKIES_KEYFRAME_INTERVAL='0',
                              SELKIES_RAW_POINTER_MOTION='true|locked',
                              SELKIES_VIDEO_FULLCOLOR='false|locked',
                              SELKIES_SECOND_SCREEN='false|locked')""",
"selkies env",
)

# ─── 6. NVIDIA i386: soporte 32 bits para juegos ───
i386 = r'''
def add_i386_nvidia(driver, root, libraries):
    """Misma versión i386. No instala módulo del kernel."""
    try:
        logged(['dpkg', '--add-architecture', 'i386'], 'i386-arch')
        logged(['apt-get', 'update', '-qq'], 'i386-update', timeout=240)
        branch = driver.split('.')[0]
        selected = []
        for name in ('libnvidia-gl', 'libnvidia-compute', 'libnvidia-gpucomp', 'libnvidia-encode'):
            package = f'{name}-{branch}:i386'
            try:
                version = matching_version(capture(['apt-cache', 'madison', package], timeout=60), driver)
            except Exception as exc:
                print(f'Aviso: {package} no disponible ({exc}).', flush=True)
                continue
            spec = package + '=' + version
            origins = capture(['apt-get', '--print-uris', 'download', spec], timeout=60)
            uris = re.findall(r"^'([^']+)'", origins, re.MULTILINE)
            if not uris or not all(official_ubuntu_uri(uri) for uri in uris):
                print(f'Aviso: origen no permitido para {package}; se omite.', flush=True)
                continue
            selected.append(spec)
        if not any(item.startswith('libnvidia-gl-') for item in selected):
            print('Aviso: sin OpenGL NVIDIA i386.', flush=True)
            return libraries
        destination = BASE / 'nvidia'
        for spec in selected:
            with (BASE / 'nvidia-download.log').open('a') as log:
                subprocess.run(['apt-get', 'download', spec], cwd=destination,
                               stdout=log, stderr=subprocess.STDOUT, check=True, timeout=180)
        for deb in destination.glob('*_i386.deb'):
            logged(['dpkg-deb', '-x', str(deb), str(root)], 'nvidia-extract-i386')
        current = libraries.split(':')
        found = sorted({str(path.parent) for path in root.rglob('*.so*')
                        if path.is_file() and 'i386-linux-gnu' in str(path)})
        merged = current + [item for item in found if item not in current]
        try:
            safe_apt_install(['libvulkan1:i386'])
        except Exception as exc:
            print(f'Aviso: libvulkan1:i386 no instalado ({exc}).', flush=True)
        print('NVIDIA i386 listo.', flush=True)
        return ':'.join(merged)
    except Exception as exc:
        print(f'Aviso: NVIDIA i386 omitido ({exc}).', flush=True)
        return libraries


'''

needle = "def bootstrap("
if src.count(needle) != 1:
    raise RuntimeError("bootstrap no es único")
if 'def add_i386_nvidia(' not in src:
    src = src.replace(needle, i386 + needle, 1)
    print("[OK] Soporte NVIDIA i386 añadido.")
else:
    print("[OK] Soporte NVIDIA i386 ya presente.")

once(
"""            root, libraries = graphics(driver)
            session, env = prepare_user(root, libraries)""",
"""            root, libraries = graphics(driver)
            libraries = add_i386_nvidia(driver, root, libraries)
            session, env = prepare_user(root, libraries)""",
"llamada i386",
)

# ─── 7. Input Guard mejorado ───
start = src.find("INPUT_GUARD_JS = r'''")
end = src.find("def prepare_web_client", start)
if start < 0 or end < 0:
    raise RuntimeError("No se encontró INPUT_GUARD_JS")

guard = r"""INPUT_GUARD_JS = r'''(() => {
  const streamTarget = target => target instanceof Element &&
    !!target.closest('#overlayInput, #videoCanvas, #videoWorkerCanvas, #videoStream, #stream');
  document.addEventListener('mousedown', event => {
    if (event.button === 2 && streamTarget(event.target)) event.preventDefault();
  }, {capture: true, passive: false});
  document.addEventListener('contextmenu', event => {
    if (streamTarget(event.target)) event.preventDefault();
  }, {capture: true, passive: false});
  const fitOverlay = () => {
    const video = document.getElementById('videoCanvas')
      || document.getElementById('videoWorkerCanvas')
      || document.querySelector('video');
    const overlay = document.getElementById('overlayInput');
    if (!video || !overlay) return;
    const rect = video.getBoundingClientRect();
    if (rect.width < 32 || rect.height < 32) return;
    overlay.style.position = 'fixed';
    overlay.style.left = Math.round(rect.left) + 'px';
    overlay.style.top = Math.round(rect.top) + 'px';
    overlay.style.width = Math.round(rect.width) + 'px';
    overlay.style.height = Math.round(rect.height) + 'px';
    overlay.style.right = 'auto';
    overlay.style.bottom = 'auto';
    overlay.style.transform = 'none';
    overlay.style.margin = '0';
  };
  let scheduled = false;
  const schedule = () => {
    if (scheduled) return;
    scheduled = true;
    requestAnimationFrame(() => { scheduled = false; fitOverlay(); });
  };
  window.addEventListener('resize', schedule);
  window.addEventListener('orientationchange', () => { schedule(); setTimeout(schedule, 400); });
  document.addEventListener('fullscreenchange', () => setTimeout(schedule, 80));
  if (window.visualViewport) window.visualViewport.addEventListener('resize', schedule);
  const swallowed = event => {
    if (!document.pointerLockElement && !document.fullscreenElement) return;
    const key = event.key;
    const chord = (event.ctrlKey || event.metaKey) && 'wtrnlt'.includes((key || '').toLowerCase());
    if (chord || key === 'Tab' || key === 'Alt' || (key && /^F\d{1,2}$/.test(key))) event.preventDefault();
  };
  window.addEventListener('keydown', swallowed, true);
  window.addEventListener('keyup', swallowed, true);
  window.pcfreeInputGuard = {version: 2, osBackCapture: false};
  const addGamingButton = () => {
    const nativeButton = document.querySelector('button.gaming-mode-button');
    if (!nativeButton || document.getElementById('pcfree-play')) return !!nativeButton;
    const button = document.createElement('button');
    button.id = 'pcfree-play';
    button.type = 'button';
    button.textContent = 'Jugar';
    button.title = 'Pantalla completa y captura de ratón y teclado';
    button.style.cssText = 'position:fixed;bottom:12px;left:12px;z-index:10000;padding:8px 16px;background:#166534;color:white;border:1px solid #86efac;border-radius:8px;cursor:pointer';
    button.addEventListener('click', () => {
      if (typeof window.webrtcInput?.enterGamingMode === 'function') window.webrtcInput.enterGamingMode();
      else nativeButton.click();
      schedule();
    });
    const visibility = () => { button.hidden = !!document.fullscreenElement; };
    document.addEventListener('fullscreenchange', visibility);
    document.body.appendChild(button);
    visibility();
    return true;
  };
  const start = () => {
    schedule();
    setInterval(schedule, 1000);
    if (document.body) {
      const observer = new MutationObserver(schedule);
      observer.observe(document.body, {childList: true, subtree: true});
    }
    if (addGamingButton()) return;
    if (window.self !== window.top) return;
    const observer = new MutationObserver(() => { if (addGamingButton()) observer.disconnect(); });
    observer.observe(document.body, {childList: true, subtree: true});
  };
  if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', start, {once: true});
  else start();
})();
'''
"""

src = src[:start] + guard + "\n\n" + src[end:]

# ─── 8. ICD por soname ───
once(
"    document['ICD']['library_path'] = str(library)\n",
"    document['ICD']['library_path'] = library.name if 'libGLX_nvidia' in library.name else str(library)\n",
"icd soname",
)

# ─── 9. Accesos: ProtonUp-Qt e Instalador DEB ───
if 'def desktop_launchers(' not in src:
    call = '    package_installer(session)\n'
    if src.count(call) != 1:
        raise RuntimeError('Ancla de accesos no única (%s).' % src.count(call))
    src = src.replace(call, call + '    try:\n        desktop_launchers(session)\n    except Exception as exc:\n        print(f"Aviso: accesos opcionales no creados ({exc}).", flush=True)\n', 1)
    src = src.replace('def bootstrap(', r'''def desktop_launchers(session):
    """Iconos del escritorio: ProtonUp-Qt y el instalador gráfico de .deb."""
    helper = Path('/usr/local/sbin/pcfree-install-deb')
    helper.write_text("""#!/usr/bin/python3.12
import os, re, subprocess, sys, urllib.parse
from pathlib import Path
if os.geteuid() != 0 or len(sys.argv) != 2:
    raise SystemExit('Uso interno: un solo .deb')
raw = Path(sys.argv[1])
if not raw.is_file() or raw.is_symlink() or raw.suffix.lower() != '.deb':
    raise SystemExit('Hay que elegir un archivo .deb.')
home = Path('/tmp/pcfree-selkies/home').resolve()
chosen = raw.resolve()
if not chosen.is_relative_to(home):
    raise SystemExit('El .deb tiene que estar en Descargas u otra carpeta del escritorio.')
meta = subprocess.check_output(['dpkg-deb', '-f', str(chosen), 'Package', 'Version', 'Architecture'], text=True)
fields = dict(line.split(': ', 1) for line in meta.splitlines() if ': ' in line)
name = fields.get('Package', '')
if (not name) or ('nvidia' in name) or name.startswith(('linux-image', 'linux-modules', 'linux-headers')):
    raise SystemExit('Ese paquete cambiaría el driver o el kernel. No se instala.')
env = {**os.environ, 'DEBIAN_FRONTEND': 'noninteractive'}
plan = subprocess.run(['apt-get', 'install', '-y', '--no-install-recommends', '-s', str(chosen)],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
text = plan.stdout or ''
if plan.returncode:
    raise SystemExit(text[-2000:] or 'APT no pudo simular la instalación.')
for line in text.splitlines():
    if line.startswith('Remv '):
        raise SystemExit('La instalación borraría paquetes. Cancelada.')
    if line.startswith('Inst '):
        pkg = line.split()[1].split(':', 1)[0]
        if ('nvidia' in pkg) or pkg.startswith(('linux-image', 'linux-modules', 'linux-headers')):
            raise SystemExit('La instalación tocaría el driver o el kernel. Cancelada.')
uris = subprocess.check_output(['apt-get', '--print-uris', '--download-only', 'install', '-y',
                                '--no-install-recommends', str(chosen)], text=True, env=env, stderr=subprocess.STDOUT)
for uri in re.findall(r"^'([^']+)'", uris, re.MULTILINE):
    parsed = urllib.parse.urlparse(uri)
    if parsed.scheme == 'file' and Path(urllib.parse.unquote(parsed.path)).resolve() == chosen:
        continue
    host = parsed.hostname or ''
    ubuntu = parsed.scheme in ('http', 'https') and not parsed.username and (
        host in ('archive.ubuntu.com', 'security.ubuntu.com', 'ports.ubuntu.com')
        or host.endswith('.archive.ubuntu.com'))
    if not ubuntu:
        raise SystemExit('Dependencia fuera de Ubuntu oficial. Cancelada.')
subprocess.run(['apt-get', 'install', '-y', '--no-install-recommends', str(chosen)], check=True, env=env)
print('Instalado:', name, fields.get('Version', ''))
""")
    helper.chmod(0o755)
    rule = Path('/etc/sudoers.d/pcfree-deb')
    rule.write_text(USER + ' ALL=(root) NOPASSWD: /usr/local/sbin/pcfree-install-deb' + chr(10))
    rule.chmod(0o440)
    logged(['visudo', '-cf', str(rule)])
    gui = Path('/usr/local/bin/pcfree-deb-gui')
    gui.write_text("""#!/usr/bin/python3.12
import subprocess, sys
try:
    chosen = subprocess.check_output([
        'zenity', '--file-selection', '--title=Instalar paquete DEB',
        '--file-filter=Paquetes DEB | *.deb', '--file-filter=Todos | *',
    ], text=True).strip()
except subprocess.CalledProcessError:
    raise SystemExit(0)
if not chosen:
    raise SystemExit(0)
run = subprocess.run(['sudo', '-n', '/usr/local/sbin/pcfree-install-deb', chosen],
                      text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
detail = (run.stdout or '').strip()[-1500:] or 'Sin salida.'
box = 'info' if run.returncode == 0 else 'error'
subprocess.run(['zenity', '--' + box, '--title=Instalar DEB', '--width=420', '--text=' + detail])
raise SystemExit(run.returncode)
""")
    gui.chmod(0o755)
    launcher = Path('/usr/local/bin/pcfree-protonup')
    launcher.write_text("""#!/usr/bin/python3.12
import os, subprocess, sys
app = '/tmp/pcfree-selkies/apps/ProtonUp-Qt.AppImage'
status = '/tmp/pcfree-selkies/protonup-status.txt'
if not (os.path.isfile(app) and os.access(app, os.X_OK)):
    msg = 'ProtonUp-Qt se está descargando. Espera y vuelve a abrir el icono.'
    if os.path.isfile(status):
        msg = open(status, encoding='utf-8', errors='replace').read()[:800]
    subprocess.run(['zenity', '--info', '--title=ProtonUp-Qt', '--width=420', '--text=' + msg])
    raise SystemExit(1)
os.environ['APPIMAGE_EXTRACT_AND_RUN'] = '1'
os.execv(app, [app, *sys.argv[1:]])
""")
    launcher.chmod(0o755)
    for package in ('libxcb-cursor0', 'libfuse2t64'):
        try:
            safe_apt_install([package])
        except Exception as exc:
            print(f'Aviso: {package} no se instaló ({exc}).', flush=True)
    run = session.get('run_launcher', '/usr/local/bin/pcfree-run')
    shortcut(session, 'Instalar-DEB', 'Instalar DEB', run + ' /usr/local/bin/pcfree-deb-gui', 'system-software-install')
    shortcut(session, 'ProtonUp-Qt', 'ProtonUp-Qt', run + ' /usr/local/bin/pcfree-protonup', 'applications-games')
    def worker():
        status = BASE / 'protonup-status.txt'
        try:
            status.write_text('Descargando ProtonUp-Qt…\\n')
            api = 'https://api.github.com/repos/DavidoTek/ProtonUp-Qt/releases/latest'
            request = urllib.request.Request(api, headers={
                'Accept': 'application/vnd.github+json',
                'User-Agent': 'free-pc-t4',
            })
            with urllib.request.urlopen(request, timeout=40) as response:
                release = json.load(response)
            chosen = next((item for item in release.get('assets', [])
                           if str(item.get('name', '')).endswith('.AppImage')
                           and 'x86_64' in str(item.get('name', ''))
                           and str(item.get('name', '')).startswith('ProtonUp-Qt')), None)
            if not chosen:
                raise RuntimeError('El release no trae AppImage x86_64.')
            url = chosen['browser_download_url']
            parsed = urllib.parse.urlparse(url)
            host = parsed.hostname or ''
            if parsed.scheme != 'https' or not (host == 'github.com' or host.endswith('githubusercontent.com')):
                raise RuntimeError('URL de ProtonUp-Qt no permitida.')
            destination = BASE / 'apps'
            destination.mkdir(parents=True, exist_ok=True)
            app = destination / 'ProtonUp-Qt.AppImage'
            download(url, app)
            app.chmod(0o755)
            account = pwd.getpwnam(USER)
            os.chown(app, account.pw_uid, account.pw_gid)
            status.write_text('Listo: ' + chosen['name'] + '\\n')
            print('ProtonUp-Qt descargado. Icono en el escritorio.', flush=True)
        except Exception as exc:
            status.write_text('No se pudo descargar ProtonUp-Qt: ' + str(exc) + '\\n')
            print('ProtonUp-Qt: ' + str(exc), flush=True)
    threading.Thread(target=worker, name='pcfree-protonup', daemon=True).start()
''' + '\ndef bootstrap(', 1)
    print("[OK] Accesos de ProtonUp-Qt e Instalar DEB añadidos.")
else:
    print("[OK] Accesos ya aplicados.")

# ─── 10. Publicar el núcleo parcheado ───
# Sin esta asignación, 'Iniciar PC-Free-T4' recompila SOURCES sin los parches
# anteriores y la celda habría impreso [OK] sin efecto real.
SOURCES["selkies-desktop.py"] = src
_core = {"__name__": "pcfree_core"}
try:
    exec(compile(src, "selkies-desktop.py", "exec"), _core)
except SyntaxError as exc:
    raise RuntimeError(f"El núcleo parcheado no compila: {exc}") from exc
if not callable(_core.get("bootstrap")):
    raise RuntimeError("El núcleo parcheado no expone bootstrap(); detener el inicio.")
print("[OK] Núcleo parcheado publicado; 'Iniciar PC-Free-T4' usará esta versión.", flush=True)


In [ ]:

# @title Iniciar PC-Free-T4 (Selkies + Cloudflare HTTP/2)
INSTALL_CHROME = True # @param {type:"boolean"}
FRAMERATE = 60 # @param {type:"integer"}
VIDEO_BITRATE_KBPS = 4000 # @param {type:"integer"}
CONFIRM_COLAB_POLICY_ALLOWED = True # @param {type:"boolean"}
SCREEN_WIDTH = 960
SCREEN_HEIGHT = 540
BACKPRESSURE_QUEUE_SIZE = 120

# @markdown **Control de congestión (CBR).** Con `False`, el bitrate se mantiene fijo en
# @markdown `VIDEO_BITRATE_KBPS` aunque la red no lo soporte: la latencia sube o se pierden
# @markdown fotogramas. Con `True`, Selkies reduce el bitrate por debajo del valor pedido según
# @markdown la pérdida y el RTT medidos, y lo recupera al mejorar el enlace. Útil en redes
# @markdown móviles o inestables; añade variación de calidad y no cambia la latencia base.
ENABLE_CONGESTION_CONTROL = False # @param {type:"boolean"}

if not CONFIRM_COLAB_POLICY_ALLOWED:
    raise RuntimeError(
        "Confirmación requerida: esta libreta crea un escritorio remoto accesible desde el navegador. "
        "No ejecutar en el runtime administrado gratuito de Colab sin saldo positivo de unidades "
        "de cómputo. Revisar la FAQ de Colab y cambiar CONFIRM_COLAB_POLICY_ALLOWED=True solo si el "
        "plan/entorno permite expresamente este uso."
    )

# Recuperación defensiva: si SOURCES sigue cargado pero _core se perdió, reconstruirlo.
if not isinstance(globals().get('SOURCES'), dict) or 'selkies-desktop.py' not in SOURCES:
    raise RuntimeError(
        "No está cargado el núcleo PC-Free-T4. Ejecuta primero la celda "
        "'Cargar núcleo interno PC-Free-T4' inmediatamente anterior; si se reinició "
        "Google Colab, usar Ejecutar todo desde el principio."
    )
if not isinstance(globals().get('_core'), dict) or not callable(_core.get('bootstrap')):
    _core = {'__name__': 'pcfree_core'}
    exec(compile(SOURCES['selkies-desktop.py'], 'selkies-desktop.py', 'exec'), _core)
if not callable(globals().get('optional_module')):
    def optional_module(name):
        namespace = {'__name__': 'pcfree_optional'}
        exec(compile(SOURCES[name], name, 'exec'), namespace)
        return namespace

SESSION = _core['bootstrap'](
    chrome=INSTALL_CHROME,
    framerate=FRAMERATE,
    bitrate=VIDEO_BITRATE_KBPS,
    width=SCREEN_WIDTH,
    height=SCREEN_HEIGHT,
    backpressure_queue_size=BACKPRESSURE_QUEUE_SIZE,
    congestion_control=ENABLE_CONGESTION_CONTROL,
)
print(f"Perfil solicitado: {SESSION['settings']['resolution']} · h264enc · use-cpu=false · {FRAMERATE} FPS · CBR {VIDEO_BITRATE_KBPS} kbps · cola {BACKPRESSURE_QUEUE_SIZE} · resize automático · congestión {'activa' if ENABLE_CONGESTION_CONTROL else 'desactivada'} · sin max-qp")
browser_env = SESSION['env']

In [ ]:

# @title Liberar espacio: cachés y archivos temporales
# @markdown Limpia cachés de APT, pip, documentación y archivos temporales.
# @markdown Conserva los archivos en uso por la sesión activa.

import subprocess, os, shutil
from pathlib import Path

KEEP_APT_LISTS   = True   # @param {type:"boolean"}
REMOVE_DOCS      = False  # @param {type:"boolean"}
ALLOW_AUTOREMOVE = False  # @param {type:"boolean"}

if os.geteuid() != 0:
    raise RuntimeError('Requiere root (Colab).')

BASE = Path('/tmp/pcfree-selkies')
print("=== Liberando espacio en disco (limpieza rápida) ===\n")

freed = 0

def get_size(path):
    """Calcula el tamaño de un directorio o archivo."""
    try:
        if path.is_file():
            return path.stat().st_size
        total = 0
        for item in path.rglob('*'):
            if item.is_file():
                total += item.stat().st_size
        return total
    except (PermissionError, OSError):
        return 0

def clean(target, description):
    """Elimina un archivo o directorio y reporta el espacio liberado."""
    global freed
    target = Path(target)
    if not target.exists():
        return
    size = get_size(target)
    try:
        if target.is_dir():
            shutil.rmtree(target, ignore_errors=True)
        else:
            target.unlink(missing_ok=True)
        freed += size
        print(f"  [OK] {description}: {size / 1024**2:.1f} MB liberados")
    except Exception as e:
        print(f"  [FAIL] {description}: error ({e})")

# 1. Limpiar caché de APT (los .deb descargados)
print("1. Limpiando caché de APT...")
size_before = get_size(Path('/var/cache/apt/archives'))
subprocess.run(['apt-get', 'clean'], capture_output=True)
freed += size_before - get_size(Path('/var/cache/apt/archives'))
print(f"  [OK] Caché de APT: {(size_before - get_size(Path('/var/cache/apt/archives'))) / 1024**2:.1f} MB")

# 2. Limpiar listas de APT (se regeneran con apt-get update)
print("2. Listas de APT...")
if KEEP_APT_LISTS:
    print("  [OK] Se conservan: borrarlas obliga a 'apt-get update' antes de cada instalación.")
else:
    clean('/var/lib/apt/lists', 'Listas de APT')

# 3. Limpiar caché de pip
print("3. Limpiando caché de pip...")
subprocess.run(['/opt/selkies/bin/python3', '-m', 'pip', 'cache', 'purge'],
               capture_output=True, timeout=60)
clean('/root/.cache/pip', 'Caché de pip (root)')
clean(str(BASE / 'pixelflux-*.whl'), 'Wheel de Pixelflux descargado')

# 4. Eliminar documentación de paquetes
print("4. Documentación de paquetes...")
if REMOVE_DOCS:
    clean('/usr/share/doc', 'Documentación /usr/share/doc')
    clean('/usr/share/man', 'Páginas de manual /usr/share/man')
    clean('/usr/share/info', 'Info /usr/share/info')
    clean('/usr/share/lintian', 'Lintian /usr/share/lintian')
    clean('/usr/share/linda', 'Linda /usr/share/linda')
else:
    print("  [OK] Se conserva (desactivado por defecto).")

# 5. Limpiar archivos temporales de la instalación
print("5. Limpiando archivos temporales...")
clean(str(BASE / 'selkies.deb'), 'selkies.deb (ya instalado)')
clean(str(BASE / 'chrome.deb'), 'chrome.deb (ya instalado)')
clean(str(BASE / 'firefox'), 'Firefox antiguo (retirado)')
# Verificar si cloudflared está en uso antes de eliminar
cloudflared_proc = subprocess.run(['pgrep', '-f', 'cloudflared'], capture_output=True)
if cloudflared_proc.returncode != 0:
    clean(str(BASE / 'cloudflared'), 'cloudflared (túnel no activo)')
else:
    print("  [AVISO] cloudflared está en uso; no se elimina")

# 6. Limpiar .deb descargados de NVIDIA
print("6. Limpiando .deb de NVIDIA (ya extraídos)...")
nvidia_dir = BASE / 'nvidia'
if nvidia_dir.exists():
    for deb in nvidia_dir.glob('*.deb'):
        clean(str(deb), f'NVIDIA deb: {deb.name}')
    # Limpiar el log de descarga
    clean(str(nvidia_dir / 'nvidia-download.log'), 'Log de descarga NVIDIA')

# 7. Limpiar journal logs
print("7. Limpiando journal logs...")
result = subprocess.run(['journalctl', '--vacuum-size=10M'], capture_output=True, text=True)
if result.returncode == 0:
    print("  [OK] Journal logs reducidos a 10 MB")

# 8. Autoremove de paquetes innecesarios
print("8. Paquetes innecesarios (autoremove)...")
if ALLOW_AUTOREMOVE:
    result = subprocess.run(['apt-get', 'autoremove', '-y', '--purge'],
                            capture_output=True, text=True, timeout=300)
    if result.returncode == 0:
        print("  [OK] Autoremove completado")
    else:
        print("  [FAIL] Autoremove no disponible")
else:
    print("  [OK] Omitido: puede retirar paquetes de la imagen base de Colab.")

# 9. Limpiar cachés de la cuenta
print("9. Limpiando cachés de la cuenta...")
user_home = BASE / 'home'
if user_home.exists():
    cache_dir = user_home / '.cache'
    if cache_dir.exists():
        # No eliminar caché de NVIDIA shaders (es útil para juegos)
        for item in cache_dir.iterdir():
            if item.name != 'nvidia-shader':
                clean(str(item), f'Caché de la cuenta: {item.name}')

# 10. Limpiar archivos .part y temporales
print("10. Limpiando archivos .part y temporales...")
for pattern in ['*.part', '*.tmp', '*.log.old']:
    for item in BASE.glob(pattern):
        clean(str(item), f'Temporal: {item.name}')

# Reportar espacio total liberado
print(f"\n=== Espacio total liberado: {freed / 1024**3:.2f} GB ===\n")

# Mostrar uso de disco actual
result = subprocess.run(['df', '-h', '/tmp'], capture_output=True, text=True)
print("Uso de disco actual:")
print(result.stdout)

# Mostrar los directorios más grandes
print("\nDirectorios más grandes en /tmp:")
result = subprocess.run(['du', '-sh', '/tmp/pcfree-selkies/*'],
                        capture_output=True, text=True)
for line in sorted(result.stdout.strip().split('\n'), reverse=True)[:10]:
    print(f"  {line}")

print("\nLimpieza rápida completada. [OK]")

In [ ]:

# @title Optimización de red del kernel (elimina micro-cortes por congestión)
# @markdown Aumenta los buffers de red del kernel para que el stream de video
# @markdown no se sature cuando hay picos de tráfico. Requiere root.

import subprocess, os

if os.geteuid() != 0:
    raise RuntimeError('Requiere root (Colab).')

print("=== Optimización de red del kernel ===\n")

settings = [
    # Buffers UDP grandes para el stream de video
    ('net.core.rmem_max',          '26214400',  'Buffer máximo de recepción'),
    ('net.core.wmem_max',          '26214400',  'Buffer máximo de envío'),
    ('net.core.rmem_default',      '1048576',   'Buffer por defecto de recepción'),
    ('net.core.wmem_default',      '1048576',   'Buffer por defecto de envío'),
    # Buffers TCP
    ('net.ipv4.tcp_rmem',          '4096 1048576 26214400', 'Buffer TCP recepción'),
    ('net.ipv4.tcp_wmem',          '4096 1048576 26214400', 'Buffer TCP envío'),
    # Cola de red más grande
    ('net.core.netdev_max_backlog', '5000',     'Cola máxima de paquetes'),
    # Congestión TCP
    ('net.ipv4.tcp_congestion_control', 'cubic', 'Algoritmo de congestión'),
]

applied = 0
for key, value, desc in settings:
    result = subprocess.run(['sysctl', '-w', f'{key}={value}'],
                          capture_output=True, text=True)
    if result.returncode == 0:
        applied += 1
        print(f"  [OK] {desc}: {value}")
    else:
        print(f"  [FAIL] {desc}: no se pudo aplicar ({result.stderr.strip()})")

print(f"\n{applied}/{len(settings)} parámetros aplicados.")
print("Los ajustes se aplican solo si el contenedor permite escribir esos parámetros.")

In [ ]:

# @title Prioridades de CPU/I/O para juego y transmisión
# @markdown Asigna prioridad moderada a Selkies, Xorg, cloudflared y PulseAudio.
# @markdown Reduce la prioridad de procesos de escritorio. Resolución, bitrate y cola permanecen sin cambios.

import os, subprocess

if os.geteuid() != 0:
    raise RuntimeError('Requiere root (Colab).')

def set_priority(pattern, nice, io_class, label):
    result = subprocess.run(['pgrep', '-f', pattern], capture_output=True, text=True)
    pids = [pid for pid in result.stdout.split() if pid.isdigit()]
    for pid in pids:
        subprocess.run(['renice', '-n', str(nice), '-p', pid], capture_output=True)
        subprocess.run(['ionice', f'-c{io_class}', '-p', pid], capture_output=True)
    print(f'  [OK] {label}: {len(pids)} proceso(s) nice={nice} ionice={io_class}')

print('=== Prioridades de juego/transmisión ===')
for pattern, label in (('/opt/selkies/bin/selkies', 'Selkies'),
                       ('cloudflared', 'cloudflared'),
                       ('Xorg', 'Xorg'),
                       ('pulseaudio', 'PulseAudio')):
    set_priority(pattern, -10, 2, label)

for pattern, label in (('thunar', 'Thunar'), ('xfdesktop', 'xfdesktop'),
                       ('xfwm4', 'xfwm4'), ('tumbler', 'Tumbler'),
                       ('tracker', 'Tracker')):
    set_priority(pattern, 10, 3, label)

print('Valores moderados: el juego conserva CPU; los procesos de fondo la ceden.')


In [ ]:

# @title Optimización de captura y stream (NvFBC + NVENC + WebCodecs)
# @markdown Comprueba los assets de WebCodecs, prepara libnvidia-fbc para la
# @markdown captura zero-copy y reinicia Selkies con la configuración del stream.

APPLY_NVFBC         = True        # @param {type:"boolean"}
APPLY_LIBXCB_UNIFY  = True        # @param {type:"boolean"}
APPLY_STREAM_TUNING = True        # @param {type:"boolean"}
APPLY_LEAN_SESSION  = True        # @param {type:"boolean"}
VERIFY_WEBCODECS    = True        # @param {type:"boolean"}
PIXELFLUX_DEBUG     = False       # @param {type:"boolean"}

# @markdown ---
# @markdown ### Stream
VIDEO_BITRATE_KBPS   = 4000       # @param {type:"integer"}
VIDEO_VBV_MULTIPLIER = 1.0        # @param {type:"number"}
VIDEO_MIN_QP         = 0          # @param {type:"integer"}
VIDEO_MAX_QP         = 0          # @param {type:"integer"}
ENCODER_LOCK         = "h264enc"  # @param ["h264enc","h265enc",""]
AUDIO_FRAME_MS       = 20         # @param {type:"integer"}
AUDIO_REDUNDANCY     = True       # @param {type:"boolean"}

# @markdown ---
# @markdown ### Sesión ligera
LEAN_PRINTING        = True       # @param {type:"boolean"}
LEAN_SHARING         = True       # @param {type:"boolean"}
FILE_TRANSFER_MBPS   = 6          # @param {type:"integer"}

import base64, json, os, pwd, re, signal, subprocess, tempfile, time
import urllib.parse, urllib.request
from pathlib import Path

BASE        = Path('/tmp/pcfree-selkies')
USER        = 'pcfree-browser'
PORT        = 8082
SELKIES_BIN = Path('/opt/selkies/bin/selkies')
SITE        = Path('/opt/selkies/lib/python3.12/site-packages')
SYS_LIBDIRS = [Path('/lib/x86_64-linux-gnu'), Path('/usr/lib/x86_64-linux-gnu')]
WEB_CLIENT  = BASE / 'web-client'

if os.geteuid() != 0:
    raise RuntimeError('Requiere root (Colab).')
if not (BASE / 'session.json').exists():
    raise RuntimeError('Requiere haber ejecutado "Iniciar PC-Free-T4".')
if not SELKIES_BIN.exists():
    raise RuntimeError('Selkies no está instalado.')

uid     = pwd.getpwnam(USER).pw_uid
session = json.loads((BASE / 'session.json').read_text())
driver  = session['driver_version']
branch  = driver.split('.')[0]
root    = BASE / 'nvidia' / 'root'


def run(argv, **kw):
    return subprocess.run(argv, text=True, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, **kw)


def say(msg):
    print(msg, flush=True)


def official(uri):
    p = urllib.parse.urlparse(uri)
    host, path = p.hostname or '', urllib.parse.unquote(p.path)
    if p.username or p.password or p.port not in (None, 80, 443) or '..' in path.split('/'):
        return False
    ubuntu = p.scheme in ('http', 'https') and (
        host in ('archive.ubuntu.com', 'security.ubuntu.com', 'ports.ubuntu.com')
        or host.endswith('.archive.ubuntu.com'))
    nvidia = (p.scheme == 'https' and host == 'developer.download.nvidia.com'
              and path.startswith('/compute/cuda/repos/ubuntu2404/x86_64/'))
    return ubuntu or nvidia


# 1) libnvidia-fbc.so.1 para la captura zero-copy de pixelflux.
fbc_dir = None
if APPLY_NVFBC:
    existing = sorted(root.rglob('libnvidia-fbc.so.1'))
    if existing:
        fbc_dir = existing[0].parent
        say(f'1) NvFBC: libnvidia-fbc.so.1 ya presente -> {fbc_dir}')
    else:
        package = f'libnvidia-fbc1-{branch}'
        version = None
        for attempt in (0, 1):
            if attempt:
                run(['apt-get', 'update', '-qq'], timeout=240)
            for line in run(['apt-cache', 'madison', package]).stdout.splitlines():
                cols = [c.strip() for c in line.split('|')]
                if len(cols) >= 3 and cols[1].split(':')[-1].split('-')[0] == driver:
                    version = cols[1]
                    break
            if version:
                break
        if not version:
            say(f'1) NvFBC: no hay {package} para {driver}; se usará DRI3 o XShm.')
        else:
            spec = f'{package}={version}'
            uris = re.findall(r"^'([^']+)'", run(['apt-get', '--print-uris', 'download', spec]).stdout, re.M)
            if not uris or not all(official(u) for u in uris):
                raise RuntimeError('Origen no oficial para libnvidia-fbc1; cancelado.')
            try:
                with tempfile.TemporaryDirectory(dir=BASE) as tmp:
                    r = run(['apt-get', 'download', spec], cwd=tmp, timeout=180)
                    debs = list(Path(tmp).glob('*.deb'))
                    if r.returncode or not debs:
                        raise RuntimeError(f'Falló la descarga de {spec}\n' + r.stdout[-800:])
                    for deb in debs:
                        r = run(['dpkg-deb', '-x', str(deb), str(root)])
                        if r.returncode:
                            raise RuntimeError('Falló la extracción:\n' + r.stdout[-800:])
                found = sorted(root.rglob('libnvidia-fbc.so.1'))
                if not found:
                    raise RuntimeError(f'libnvidia-fbc.so.1 no apareció tras extraer {spec}')
                fbc_dir = found[0].parent
                say(f'1) NvFBC: extraída {spec} -> {found[0]}')
            except RuntimeError as exc:
                say(f'1) NvFBC: {exc} Se usará DRI3 o XShm.')
else:
    say('1) NvFBC: desactivado por la cuenta.')


# 2) argv y entorno del Selkies vivo.
def selkies_pids():
    return [int(p) for p in run(['pgrep', '-f', str(SELKIES_BIN)]).stdout.split()]


def find_selkies():
    for pid in selkies_pids():
        try:
            if os.stat(f'/proc/{pid}').st_uid != uid:
                continue
            cmd = [a.decode() for a in
                   Path(f'/proc/{pid}/cmdline').read_bytes().split(b'\0') if a]
            idx = next((i for i, a in enumerate(cmd)
                        if a.endswith('/opt/selkies/bin/selkies')), None)
            if idx is None:
                continue
            raw = Path(f'/proc/{pid}/environ').read_bytes()
            env = dict(e.decode().split('=', 1) for e in raw.split(b'\0') if b'=' in e)
            return pid, ['/opt/selkies/bin/selkies', *cmd[idx + 1:]], env
        except FileNotFoundError:
            continue
    raise RuntimeError(f'No hay un Selkies en ejecución como {USER}.')


pid, args, env = find_selkies()
orig_args = list(args)
orig_env  = dict(env)
say(f'2) Selkies vivo (PID {pid}): {len(args) - 1} argumentos leídos.')


# 3) LD_LIBRARY_PATH y bibliotecas necesarias.
def rebuild_library_path(extra_first=()):
    parts, seen = [], set()
    def add(p):
        p = str(p or '').strip()
        if p and p not in seen:
            seen.add(p)
            parts.append(p)
    for p in extra_first:
        add(p)
    for p in env.get('LD_LIBRARY_PATH', '').split(':'):
        add(p)
    add('/usr/lib64-nvidia')
    env['LD_LIBRARY_PATH'] = ':'.join(parts)
    return parts


paths = rebuild_library_path(extra_first=(fbc_dir,) if fbc_dir else ())


def resolve_lib(name):
    for d in paths:
        cand = Path(d) / name
        if cand.exists():
            return cand.resolve()
    return None


say('3) Bibliotecas para captura y codificación:')
missing = []
for lib in ('libcuda.so.1', 'libnvidia-encode.so.1', 'libnvidia-fbc.so.1', 'libnvidia-glcore.so.1'):
    got = resolve_lib(lib)
    if got is None:
        missing.append(lib)
        say(f'   [FAIL] {lib}: no resuelta')
    else:
        say(f'   [OK] {lib} -> {got}')
if 'libnvidia-encode.so.1' in missing:
    raise RuntimeError('Sin libnvidia-encode no hay NVENC; no tiene sentido continuar.')


# 4) Unificar libxcb con el sistema; NvFBC declina si libX11 usa otra copia.
def normalize_soname(name):
    m = re.match(r'^(.*?)-[0-9a-f]{6,16}(\.so[\d.]*)$', name)
    return (m.group(1) + m.group(2)) if m else name


def system_counterpart(name):
    for candidate in (name, normalize_soname(name)):
        for d in SYS_LIBDIRS:
            p = d / candidate
            if p.is_file():
                return p.resolve()
    base = normalize_soname(name).split('.so')[0]
    for d in SYS_LIBDIRS:
        for c in sorted(d.glob(base + '.so*'), key=lambda x: (len(x.name), x.name)):
            if c.is_file():
                return c.resolve()
    return None


def unify_libxcb():
    targets = set()
    for pattern in ('libxcb*.so*', 'libX11-xcb.so*'):
        for path in SITE.rglob(pattern):
            if path.name.endswith('.orig') or path.is_symlink() or not path.is_file():
                continue
            targets.add(path)
    changed, skipped = [], []
    for bundled in sorted(targets):
        host = system_counterpart(bundled.name)
        if host is None:
            skipped.append(bundled.name)
            continue
        backup = bundled.with_name(bundled.name + '.orig')
        if backup.exists():
            bundled.unlink()
        else:
            bundled.rename(backup)
        bundled.symlink_to(host)
        changed.append(f'{bundled.relative_to(SITE)} -> {host.name}')
    return changed, skipped


def restore_libxcb():
    for backup in SITE.rglob('*.orig'):
        original = backup.with_name(backup.name[:-len('.orig')])
        if original.is_symlink() or original.exists():
            original.unlink()
        backup.rename(original)


if APPLY_LIBXCB_UNIFY:
    changed, skipped = unify_libxcb()
    say('4) libxcb unificada (mapeo 1:1 por nombre):')
    for line in changed or ['   (ya estaba unificada)']:
        say('   ' + line)
    for name in skipped:
        say(f'   ! sin contraparte del sistema para {name}; se dejó intacta')
else:
    say('4) Unificación de libxcb: desactivada.')


# 5) WebCodecs: el cliente ya lo usa; se verifica, no se fuerza.
webcodecs_assets = []
if VERIFY_WEBCODECS:
    say('5) WebCodecs: comprobando el cliente servido...')
    index_html = WEB_CLIENT / 'index.html'
    if index_html.exists():
        html = index_html.read_text(errors='replace')
        cleaned = re.sub(r'\n?<script>\s*// PC-Free: Forzar WebCodecs.*?</script>\n?',
                         '\n', html, flags=re.S)
        if cleaned != html:
            index_html.write_text(cleaned)
            say('   [OK] Retirada una inyección WebCodecs anterior sin efecto real.')
    for asset in WEB_CLIENT.rglob('*.js'):
        try:
            if 'VideoDecoder' in asset.read_text(errors='ignore'):
                webcodecs_assets.append(asset)
        except OSError:
            pass
    if webcodecs_assets:
        say(f'   [OK] Cliente WebCodecs presente: {len(webcodecs_assets)} archivo(s) con VideoDecoder.')
    else:
        say('   [AVISO] No se encontró VideoDecoder en los assets; el cliente usará la ruta de respaldo.')
else:
    say('5) WebCodecs: comprobación desactivada.')


# 6) Sintonía del stream con opciones reales de Selkies.
help_out = run([str(SELKIES_BIN), '--help'], timeout=30).stdout
KNOWN = set(re.findall(r'--[a-z0-9][a-z0-9-]*', help_out))


def set_flag(argv, flag, value):
    out, skip = [], False
    for a in argv:
        if a == flag:
            skip = True
            continue
        if skip:
            skip = False
            continue
        if a.startswith(flag + '='):
            continue
        out.append(a)
    out.append(f'{flag}={value}')
    return out


def apply_settings(argv, wanted, title):
    applied, absent = [], []
    for flag, value in wanted:
        if flag in KNOWN:
            argv = set_flag(argv, flag, value)
            applied.append(f'{flag}={value}')
            env['SELKIES_' + flag.lstrip('-').upper().replace('-', '_')] = str(value)
        else:
            absent.append(flag)
    say(f'6) {title}:')
    for line in applied or ['   (ninguna)']:
        say('   + ' + line)
    if absent:
        say('   ~ no existen en esta versión: ' + ', '.join(absent))
    return argv


if APPLY_STREAM_TUNING:
    video = [
        ('--video-bitrate',        str(int(VIDEO_BITRATE_KBPS))),
        ('--rate-control-mode',    'cbr'),
        ('--keyframe-interval',    '0'),
        ('--video-streaming-mode', 'true'),
        ('--video-fullcolor',      'false'),
        ('--use-cpu',              'false'),
        ('--auto-gpu',             'nvidia'),
    ]
    if VIDEO_VBV_MULTIPLIER and VIDEO_VBV_MULTIPLIER > 0:
        video.append(('--video-vbv-multiplier', f'{float(VIDEO_VBV_MULTIPLIER):g}'))
    if VIDEO_MIN_QP and VIDEO_MIN_QP > 0:
        video.append(('--video-min-qp', str(int(VIDEO_MIN_QP))))
    if VIDEO_MAX_QP and VIDEO_MAX_QP > 0:
        video.append(('--video-max-qp', str(int(VIDEO_MAX_QP))))
    if ENCODER_LOCK:
        video.append(('--encoder', ENCODER_LOCK))
    args = apply_settings(args, video, 'Video (NVENC)')

    audio = [('--audio-frame-duration-ms', str(int(AUDIO_FRAME_MS))),
             ('--audio-redundancy', 'true' if AUDIO_REDUNDANCY else 'false')]
    args = apply_settings(args, audio, 'Audio (Opus)')
else:
    say('6) Sintonía de stream: desactivada.')

if APPLY_LEAN_SESSION:
    lean = [('--microphone-enabled', 'false'), ('--webcam-enabled', 'false')]
    if LEAN_PRINTING:
        lean.append(('--printing-enabled', 'false'))
    if LEAN_SHARING:
        lean.append(('--enable-sharing', 'false'))
    if FILE_TRANSFER_MBPS and FILE_TRANSFER_MBPS > 0:
        lean.append(('--file-transfer-limit-mbps', str(int(FILE_TRANSFER_MBPS))))
    args = apply_settings(args, lean, 'Sesión ligera')
else:
    say('6) Sesión ligera: desactivada.')

if PIXELFLUX_DEBUG:
    env['PIXELFLUX_DEBUG'] = '1'
else:
    env.pop('PIXELFLUX_DEBUG', None)

enc = next((a.split('=', 1)[1] for a in args if a.startswith('--encoder=')), '')
if enc and enc not in ('h264enc', 'h265enc'):
    raise RuntimeError(f'--encoder={enc} no tiene motor NVENC en una Tesla T4.')


# 7) Reiniciar solo Selkies.
def stop_selkies():
    for p in selkies_pids():
        try:
            os.kill(p, signal.SIGTERM)
        except ProcessLookupError:
            pass
    end = time.monotonic() + 10
    while time.monotonic() < end and selkies_pids():
        time.sleep(0.3)
    for p in selkies_pids():
        try:
            os.kill(p, signal.SIGKILL)
        except ProcessLookupError:
            pass
    time.sleep(1)


def start_selkies(argv, environ, note):
    with (BASE / 'selkies.log').open('a') as log:
        log.write(f'\n=== {note} ===\n')
        log.flush()
        return subprocess.Popen(['runuser', '-u', USER, '--', *argv], env=environ,
                                stdout=log, stderr=subprocess.STDOUT, start_new_session=True)


def ready(proc, timeout=75):
    access = json.loads((BASE / 'access.json').read_text())
    token = base64.b64encode(
        f"{access['username']}:{access['password']}".encode()).decode()
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}/',
                                 headers={'Authorization': 'Basic ' + token})
    end = time.monotonic() + timeout
    while time.monotonic() < end:
        if proc.poll() is not None:
            return False
        try:
            with urllib.request.urlopen(req, timeout=2) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(0.5)
    return False


def save_pid(proc, libxcb_unified):
    state = BASE / 'session.json'
    data = json.loads(state.read_text())
    start = Path(f'/proc/{proc.pid}/stat').read_text().rsplit(')', 1)[1].split()[19]
    data.setdefault('processes', {})['selkies'] = {'pid': proc.pid, 'start': start}
    data['optimized'] = {'nvfbc': bool(fbc_dir),
                         'webcodecs_assets': len(webcodecs_assets),
                         'libxcb_unified': bool(libxcb_unified),
                         'bitrate_kbps': int(VIDEO_BITRATE_KBPS),
                         'vbv_multiplier': float(VIDEO_VBV_MULTIPLIER),
                         'ts': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
    tmp = state.with_suffix('.tmp')
    tmp.write_text(json.dumps(data, indent=2))
    tmp.chmod(0o600)
    tmp.replace(state)


say('7) Reiniciando solo Selkies...')
stop_selkies()
proc = start_selkies(args, env, 'Optimización de captura y stream')

if ready(proc):
    save_pid(proc, APPLY_LIBXCB_UNIFY)
    say('   [OK] Selkies arriba. Recarga la página del cliente (Ctrl+F5).')
else:
    say('   [FAIL] Selkies no arrancó. Revirtiendo...')
    stop_selkies()
    if APPLY_LIBXCB_UNIFY:
        restore_libxcb()
    proc = start_selkies(orig_args, orig_env, 'ROLLBACK: configuración original')
    if ready(proc):
        save_pid(proc, False)
        say('   Rollback correcto: Selkies funciona como antes.')
        say('   Revisa las últimas 60 líneas de /tmp/pcfree-selkies/selkies.log.')
    else:
        raise RuntimeError(
            'Selkies tampoco arrancó tras el rollback. Revisa '
            '/tmp/pcfree-selkies/selkies.log antes de reiniciar la VM.')


# 8) Veredicto con datos del log y de nvidia-smi.
time.sleep(8)
log = (BASE / 'selkies.log').read_text(errors='replace')
tail = log.split('=== Optimización de captura y stream')[-1]
pattern = re.compile(
    r'nvfbc|nvidia-fbc|dri3|xshm|zero[- ]?copy|capture|backend|libxcb|'
    r'nvenc|encode node|render node|declin|watermark|pixelflux|webcodecs', re.I)
hits = [line.strip() for line in tail.splitlines() if pattern.search(line)]

say('')
say('=' * 20 + ' VEREDICTO ' + '=' * 20)
for line in hits[-12:] or ['(sin líneas de captura aún: conecta el cliente y vuelve a ejecutar)']:
    say('  ' + line[:200])

q = run(['nvidia-smi',
         '--query-gpu=name,driver_version,utilization.gpu,utilization.encoder,'
         'memory.used,encoder.stats.sessionCount,encoder.stats.averageFps',
         '--format=csv,noheader'],
        env={'LD_LIBRARY_PATH': env.get('LD_LIBRARY_PATH', '')}, timeout=30)
say('')
say('NVENC en vivo (utilization.encoder > 0 = codifica el hardware):')
say('  ' + (q.stdout.strip() or q.stderr.strip()))

say('')
say('Resumen aplicado:')
say(f'  libnvidia-fbc.....: {"[OK]" if fbc_dir else "[--]"}')
say(f'  libxcb unificada..: {"[OK]" if APPLY_LIBXCB_UNIFY else "[--]"}')
say(f'  WebCodecs assets..: {len(webcodecs_assets)} archivo(s)')
say(f'  Bitrate...........: {VIDEO_BITRATE_KBPS} kbps CBR')
say(f'  Audio.............: Opus {AUDIO_FRAME_MS} ms, RED={"on" if AUDIO_REDUNDANCY else "off"}')
say(f'  Códec bloqueado...: {ENCODER_LOCK or "(menú abierto)"}')
say('')
say('WebCodecs lo elige el cliente; el servidor solo comprueba que los assets lo traen.')
say('Confirma la decodificación por hardware en chrome://gpu o about:support.')


In [ ]:

# @title Entorno global de juegos (Steam/Proton)
# @markdown Escribe un archivo que el lanzador de Steam carga antes de Proton.
# @markdown Variables globales de DXVK 3.1 y VKD3D 3.0 para Steam/Proton.

import os, pwd
from pathlib import Path

if not isinstance(globals().get('SESSION'), dict):
    raise RuntimeError("Requiere haber ejecutado 'Iniciar PC-Free-T4'.")

user = SESSION['user']
home = Path(SESSION['home'])
account = pwd.getpwnam(user)
config_dir = home / '.config' / 'pcfree'
config_dir.mkdir(parents=True, exist_ok=True)
os.chown(config_dir.parent, account.pw_uid, account.pw_gid)
os.chown(config_dir, account.pw_uid, account.pw_gid)

env_file = config_dir / 'game.env'
env_file.write_text('''# PC-Free: entorno global para Steam/Proton
DXVK_SHADER_CACHE_PATH="$HOME/.cache/pcfree/dxvk"
VKD3D_SHADER_CACHE_PATH="$HOME/.cache/pcfree/vkd3d"
DXVK_LOG_LEVEL=none
PROTON_LOG=0
WINEDEBUG=-all
''')
env_file.chmod(0o644)
os.chown(env_file, account.pw_uid, account.pw_gid)

for name in ('dxvk', 'vkd3d'):
    cache = home / '.cache' / 'pcfree' / name
    cache.mkdir(parents=True, exist_ok=True)
    os.chown(cache.parent, account.pw_uid, account.pw_gid)
    os.chown(cache, account.pw_uid, account.pw_gid)

print('[OK] Entorno global escrito:', env_file)
print('     DXVK_SHADER_CACHE_PATH y VKD3D_SHADER_CACHE_PATH')
print('     Solo variables documentadas de DXVK 3.1 y VKD3D 3.0.')
print('     El lanzador de Steam lo carga al abrir Steam.')


In [ ]:

# @title Verificar Chrome y navegador predeterminado
# @markdown Comprueba versión, perfil y asociación del navegador predeterminado.

import shutil, subprocess
from pathlib import Path

if not isinstance(globals().get('SESSION'), dict):
    raise RuntimeError("Requiere haber ejecutado 'Iniciar PC-Free-T4'.")

user = SESSION['user']
env = SESSION['env']
chrome = shutil.which('google-chrome')
print('Binario Chrome:', chrome or 'NO ENCONTRADO')
if chrome:
    version = subprocess.run([chrome, '--version'], text=True, capture_output=True, timeout=20)
    print('Versión:', (version.stdout or version.stderr).strip())
    wrapper = Path('/usr/local/bin/pcfree-chrome')
    print('Lanzador:', wrapper.is_file())
    if wrapper.is_file():
        text = wrapper.read_text()
        for flag in ('--user-data-dir=', '--renderer-process-limit=2', '--process-per-site',
                     '--disable-background-networking', '--use-gl=egl'):
            print(f'  {flag}:', flag in text)
    profile = Path('/tmp/pcfree-selkies/chrome-profile')
    print('Perfil:', profile.is_dir(), profile)

env = dict(env, PATH='/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin')
runuser = shutil.which('runuser') or '/usr/sbin/runuser'
try:
    check = subprocess.run(
        [runuser, '-u', user, '--', 'xdg-settings', 'get', 'default-web-browser'],
        env=env, text=True, capture_output=True, timeout=20)
    print('Navegador predeterminado:', (check.stdout or check.stderr).strip())
except (FileNotFoundError, subprocess.SubprocessError) as exc:
    print('No se pudo consultar el navegador predeterminado:', exc)


## Steam

`INSTALL_STEAM` controla la instalación. No abre el cliente. Si el kernel no da user namespaces, el fallback `proot-bwrap` queda desactivado salvo que lo actives en la celda.


In [ ]:
# @title Instalar Steam oficial + compatibilidad user namespaces (no lo abre automáticamente)
INSTALL_STEAM = True # @param {type:"boolean"}
STEAM_GAMEPADS = True # @param {type:"boolean"}
INSTALL_GAMEPADS = False # @param {type:"boolean"}
ENABLE_PROOT_BWRAP_FALLBACK = False # @param {type:"boolean"}

import hashlib
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import urllib.parse
import urllib.request
from pathlib import Path

if 'SESSION' not in globals() or not isinstance(SESSION, dict):
    raise RuntimeError("Requiere haber ejecutado 'Iniciar PC-Free-T4 (Selkies + Cloudflare HTTP/2)'.")
if not callable(globals().get('optional_module')):
    raise RuntimeError("No está cargado compat-steam.py; requiere la celda 'Cargar núcleo interno'.")

steam_mod = optional_module('compat-steam.py')
PCFREE_STEAM_BWRAP = None


def _download_https(url, destination, allowed_hosts):
    parsed = urllib.parse.urlparse(url)
    if parsed.scheme != 'https' or parsed.hostname not in allowed_hosts:
        raise RuntimeError(f"URL HTTPS no permitida: {url}")
    request = urllib.request.Request(url, headers={
        'User-Agent': 'PC-Free-T4',
        'Accept': 'application/vnd.github+json',
    })
    with urllib.request.urlopen(request, timeout=90) as response:
        final = urllib.parse.urlparse(response.url)
        if final.scheme != 'https' or final.hostname not in allowed_hosts:
            raise RuntimeError('Redirección a un host no permitido; descarga cancelada.')
        with Path(destination).open('wb') as stream:
            shutil.copyfileobj(response, stream)


def _prepare_proot_bwrap(session, module):
    """Install Selkies' official bubblewrap stand-in privately for this Steam launcher."""
    # Keep this separate from /usr/bin/steam and never run the upstream installer.sh:
    # that script diverts the system Steam launcher and changes system package state.
    if os.geteuid() != 0:
        raise RuntimeError('La preparación de dependencias requiere la celda como root.')

    # Resolve the official repository's current main to an immutable commit for this run.
    api_url = 'https://api.github.com/repos/selkies-project/proot-bwrap/commits/main'
    metadata_path = Path('/tmp/pcfree-selkies/proot-bwrap-commit.json')
    _download_https(api_url, metadata_path, {'api.github.com'})
    metadata = json.loads(metadata_path.read_text())
    commit = metadata.get('sha', '')
    if len(commit) != 40 or any(ch not in '0123456789abcdef' for ch in commit.lower()):
        raise RuntimeError('GitHub no devolvió un SHA completo de proot-bwrap.')

    base = Path('/tmp/pcfree-selkies/proot-bwrap-support')
    base.mkdir(parents=True, exist_ok=True)
    helper = base / 'proot-bwrap'
    proot = base / 'proot'
    stamp = base / 'source-commit.txt'
    valid_cache = (
        helper.is_file() and os.access(helper, os.X_OK)
        and proot.is_file() and os.access(proot, os.X_OK)
        and stamp.is_file() and stamp.read_text().strip() == commit
    )

    log = Path('/tmp/pcfree-selkies/steam-proot-bwrap-install.log')
    module['_run'](['dpkg', '--add-architecture', 'i386'], log=log)
    module['_run'](['apt-get', 'update', '-qq'], log=log, timeout=600)
    if not valid_cache:
        # Bubblewrap's stand-in needs libfakechroot for the browser helper; the
        # pinned modern PRoot backend is used for Steam runtimes and games when possible.
        module['_safe_install']([
            'ca-certificates', 'gcc', 'libc6-dev', 'make', 'pkgconf',
            'libtalloc2', 'libtalloc-dev',
            'libfakechroot', 'libfakechroot:i386',
        ], log)

        # Upstream install.sh pins this PRoot source archive and checks its SHA-256.
        proot_commit = '25dc6a3134891f98a79f57ce1c2c1b23ff15cad1'
        proot_sha256 = '6adb0d808dad47d66f503d6d5a699b455ac1a0a519b905378be4ffc0af92f40e'
        with tempfile.TemporaryDirectory(prefix='pcfree-proot-build-', dir='/tmp') as work:
            work = Path(work)
            archive = work / 'proot.tar.gz'
            _download_https(
                f'https://github.com/proot-me/proot/archive/{proot_commit}.tar.gz',
                archive, {'github.com', 'codeload.github.com'}
            )
            digest = hashlib.sha256(archive.read_bytes()).hexdigest()
            if digest.lower() != proot_sha256:
                raise RuntimeError('SHA-256 de PRoot no coincide con el hash fijado por Selkies; descarga rechazada.')
            with tarfile.open(archive, 'r:gz') as tf:
                members = tf.getmembers()
                expected_prefix = f'proot-{proot_commit}/'
                for item in members:
                    target = Path(item.name)
                    if target.is_absolute() or '..' in target.parts or not item.name.startswith(expected_prefix):
                        raise RuntimeError('Archivo fuente PRoot con rutas inesperadas; extracción cancelada.')
                    if not (item.isfile() or item.isdir() or item.issym() or item.islnk()):
                        raise RuntimeError('Tipo de archivo inesperado en fuente PRoot.')
                tf.extractall(work, members=members, filter='data')
            source_dir = work / f'proot-{proot_commit}' / 'src'
            build = subprocess.run(
                ['make', '-C', str(source_dir), '-j2', 'proot', 'GIT=false', 'WITHOUT_PYTHON=1'],
                text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=600,
            )
            with log.open('a') as stream:
                stream.write('\n$ make pinned PRoot\n' + build.stdout + '\n')
            if build.returncode != 0 or not (source_dir / 'proot').is_file():
                raise RuntimeError('No se pudo compilar PRoot fijado; revisa ' + str(log) + '\n' + build.stdout[-2500:])
            shutil.copy2(source_dir / 'proot', proot)
            proot.chmod(0o755)

        helper_tmp = base / 'proot-bwrap.new'
        _download_https(
            f'https://raw.githubusercontent.com/selkies-project/proot-bwrap/{commit}/proot-bwrap',
            helper_tmp, {'raw.githubusercontent.com'}
        )
        script_text = helper_tmp.read_text(errors='replace')
        if 'stand-in for pressure-vessel' not in script_text or 'PROGRAM = "proot-bwrap"' not in script_text:
            helper_tmp.unlink(missing_ok=True)
            raise RuntimeError('El contenido descargado no parece ser el helper oficial proot-bwrap.')
        helper_tmp.chmod(0o755)
        helper_tmp.replace(helper)
        stamp.write_text(commit + '\n')

    # Quick smoke-test the helper as the actual desktop user, before Steam install.
    user = session.get('user')
    home = session.get('home')
    runtime = session.get('runtime')
    if not all(isinstance(v, str) and Path(v).is_absolute() for v in (user, home, runtime)):
        raise RuntimeError('SESSION no tiene user/home/runtime válidos para probar proot-bwrap.')
    smoke_env = [
        'HOME=' + home,
        'XDG_RUNTIME_DIR=' + runtime,
        'PATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin',
    ]
    test = subprocess.run(
        ['runuser', '-u', user, '--', 'env', *smoke_env,
         str(helper), '--ro-bind', '/', '/', '--proc', '/proc', '--dev', '/dev', '--', '/usr/bin/true'],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=45,
    )
    if test.returncode != 0:
        raise RuntimeError(
            'El smoke-test de proot-bwrap falló; Steam no se instalará con un fallback que no pasó la prueba.\n'
            + test.stdout[-3000:] + '\nRegistro: ' + str(log)
        )
    print('proot-bwrap verificado; commit:', commit)
    print('PRoot compilado con SHA-256 de la fuente validado; prueba aislada: OK.')
    print('Nota: es compatibilidad para contenedores, no una frontera de seguridad independiente.')
    return helper


if INSTALL_STEAM:
    # Test the actual capability; a sysctl value by itself does not prove it works.
    namespace_test = subprocess.run(
        ['unshare', '--user', '--map-root-user', 'true'],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=10,
    ) if shutil.which('unshare') else subprocess.CompletedProcess([], 127, '', 'unshare no disponible')

    steam_options = {'gamepads': STEAM_GAMEPADS}
    if namespace_test.returncode != 0:
        print('El kernel no permite user namespaces sin privilegios:')
        print((namespace_test.stdout or '').strip() or 'unshare falló sin detalle')
        if not ENABLE_PROOT_BWRAP_FALLBACK:
            raise RuntimeError('Steam requiere user namespaces y el fallback proot-bwrap está desactivado.')
        PCFREE_STEAM_BWRAP = _prepare_proot_bwrap(SESSION, steam_mod)
        steam_options['bwrap'] = str(PCFREE_STEAM_BWRAP)
    else:
        print('User namespaces disponibles; usando el sandbox nativo de Steam.')

    SESSION['steam'] = steam_options
    steam_mod['install_steam'](SESSION)

if INSTALL_GAMEPADS and not (INSTALL_STEAM and STEAM_GAMEPADS):
    steam_mod['install_gamepads'](SESSION)

In [ ]:

# @title Instalar Heroic, Wine 9 y Proton 9 con dependencias
INSTALL_HEROIC = True          # @param {type:"boolean"}
INSTALL_GAMING_LIBS = True     # @param {type:"boolean"}
INSTALL_WINE = True            # @param {type:"boolean"}
INSTALL_HEROIC_PROTON = True   # @param {type:"boolean"}

import hashlib
import json
import os
import pwd
import re
import shutil
import subprocess
import tarfile
import tempfile
import urllib.parse
import urllib.request
from pathlib import Path

if not INSTALL_HEROIC:
    print('Heroic omitido.')
elif not isinstance(globals().get('SESSION'), dict):
    raise RuntimeError('Requiere haber iniciado PC-Free-T4.')
elif os.geteuid() != 0:
    raise RuntimeError('Requiere root para instalar paquetes.')
else:
    BASE_HEROIC = Path('/tmp/pcfree-selkies')
    BASE_HEROIC.mkdir(parents=True, exist_ok=True)
    log = BASE_HEROIC / 'install-heroic.log'
    SESSION_HEROIC = SESSION
    HOME = Path(SESSION_HEROIC['home'])
    ACCOUNT = pwd.getpwnam(SESSION_HEROIC['user'])
    HEROIC_CONFIG = HOME / '.config' / 'heroic' / 'config.json'
    HEROIC_TOOLS = HOME / '.config' / 'heroic' / 'tools'
    WINE_BIN = '/usr/bin/wine'
    WINE_SERVER = shutil.which('wineserver') or '/usr/bin/wineserver'
    GE9_VERSION = 'GE-Proton9-27'
    GE9_URL = ('https://github.com/GloriousEggroll/proton-ge-custom/releases/download/'
               f'{GE9_VERSION}/{GE9_VERSION}.tar.gz')
    GE9_SHA512 = ('86a2b2962a2509104201b3532bc829d058d666bc8220417f71bd4af660b6d057'
                  '81e9f684b3982339d695a5fd4babe19e97ec42a82a78311faf99fc1257280623')

    def official_ubuntu_uri(uri):
        parsed = urllib.parse.urlparse(uri)
        if parsed.username or parsed.password or parsed.port not in (None, 80, 443):
            return False
        ubuntu = (parsed.scheme in ('http', 'https') and
                  (parsed.hostname in ('archive.ubuntu.com', 'security.ubuntu.com', 'ports.ubuntu.com')
                   or (parsed.hostname or '').endswith('.archive.ubuntu.com')))
        path = urllib.parse.unquote(parsed.path)
        nvidia = (parsed.scheme == 'https' and parsed.port in (None, 443)
                  and parsed.hostname == 'developer.download.nvidia.com'
                  and path.startswith('/compute/cuda/repos/ubuntu2404/x86_64/')
                  and '..' not in path.split('/'))
        return ubuntu or nvidia

    def safe_install(packages):
        plan = subprocess.check_output(
            ['apt-get', '-s', 'install', '-y', '--no-install-recommends', *packages],
            text=True, stderr=subprocess.STDOUT, timeout=120)
        for line in plan.splitlines():
            if line.startswith('Remv '):
                raise RuntimeError('APT propone eliminar paquetes; operación cancelada.')
            if line.startswith('Inst '):
                pkg = line.split()[1].split(':')[0]
                if 'nvidia' in pkg or pkg.startswith(('linux-image', 'linux-modules', 'linux-headers')):
                    raise RuntimeError(f'APT propone cambiar {pkg}; no se modifica el controlador del host.')
        origins = subprocess.check_output(
            ['apt-get', '--print-uris', '--download-only', 'install', '-y',
             '--no-install-recommends', *packages],
            text=True, stderr=subprocess.STDOUT, timeout=120)
        for uri, _filename in re.findall(r"^'([^']*)'\s+(\S+)", origins, re.MULTILINE):
            if uri.startswith('file:'):
                continue
            if not official_ubuntu_uri(uri):
                raise RuntimeError(f'APT propone un origen no permitido: {uri}')
        with log.open('a') as stream:
            result = subprocess.run(
                ['apt-get', 'install', '-y', '--no-install-recommends', *packages],
                stdout=stream, stderr=subprocess.STDOUT,
                env={**os.environ, 'DEBIAN_FRONTEND': 'noninteractive'},
                timeout=900)
        if result.returncode:
            raise RuntimeError(f'Falló la instalación; revisar {log}')

    def install_group(packages, label):
        try:
            safe_install(packages)
            print(f'{label}: instalado.')
            return
        except RuntimeError as exc:
            print(f'Aviso: {label} se omite en grupo ({exc}); se intentará por paquete.')
        for package in packages:
            try:
                safe_install([package])
                print(f'  [OK] {package}')
            except RuntimeError as exc:
                print(f'  [SKIP] {package}: {exc}')

    subprocess.run(['dpkg', '--add-architecture', 'i386'], check=True)
    subprocess.run(['apt-get', 'update', '-qq'], check=True, timeout=300)

    HEROIC_RUNTIME = [
        'libgtk-3-0t64', 'libnotify4', 'libnss3', 'libxss1', 'libxtst6',
        'libatspi2.0-0t64', 'libuuid1', 'libsecret-1-0',
        'libayatana-appindicator3-1', 'xdg-utils', 'curl', 'ca-certificates',
        'libfuse2t64', 'bubblewrap', 'shared-mime-info',
    ]
    WINE_PACKAGES = [
        'wine', 'wine64', 'wine32:i386', 'libwine', 'libwine:i386',
        'winetricks', 'cabextract', '7zip', 'unzip', 'xz-utils',
        'libgnutls30t64', 'libgnutls30t64:i386', 'libldap2', 'libldap2:i386',
        'libsasl2-2', 'libsasl2-2:i386', 'libkrb5-3', 'libkrb5-3:i386',
        'libcups2t64', 'libcups2t64:i386', 'libxml2', 'libxml2:i386',
        'libxslt1.1', 'libxslt1.1:i386', 'libosmesa6', 'libosmesa6:i386',
        'libpng16-16t64', 'libpng16-16t64:i386', 'libjpeg-turbo8', 'libjpeg-turbo8:i386',
        'libtiff6', 'libtiff6:i386', 'libmpg123-0t64', 'libv4l-0t64', 'libgphoto2-6t64',
        'libsane1', 'libsane1:i386', 'libasound2-plugins', 'libasound2-plugins:i386',
    ]
    GAME_32 = [
        'libc6:i386', 'libstdc++6:i386', 'libgcc-s1:i386', 'libglib2.0-0t64:i386',
        'libgl1:i386', 'libglx0:i386', 'libglvnd0:i386', 'libegl1:i386', 'libgles2:i386',
        'libgbm1:i386', 'libdrm2:i386', 'libvulkan1:i386',
        'libx11-6:i386', 'libx11-xcb1:i386', 'libxext6:i386', 'libxrandr2:i386',
        'libxinerama1:i386', 'libxcursor1:i386', 'libxi6:i386', 'libxss1:i386',
        'libxtst6:i386', 'libxfixes3:i386', 'libxrender1:i386', 'libxcomposite1:i386',
        'libxdamage1:i386', 'libxcb1:i386', 'libxcb-render0:i386', 'libxcb-shm0:i386',
        'libxcb-dri3-0:i386', 'libxkbcommon0:i386', 'libxkbcommon-x11-0:i386',
        'libwayland-client0:i386', 'libudev1:i386', 'libdbus-1-3:i386',
        'libfontconfig1:i386', 'libfreetype6:i386', 'libasound2t64:i386',
        'libpulse0:i386', 'libopenal1:i386', 'libsdl2-2.0-0:i386',
    ]

    install_group(HEROIC_RUNTIME, 'Dependencias de Heroic')
    if INSTALL_GAMING_LIBS:
        install_group(GAME_32, 'Bibliotecas de juego 32 bits')
    if INSTALL_WINE:
        install_group(WINE_PACKAGES, 'Wine 9 y dependencias')

    print('Consultando el último release de Heroic Games Launcher…')
    api_url = 'https://api.github.com/repos/Heroic-Games-Launcher/HeroicGamesLauncher/releases/latest'
    request = urllib.request.Request(api_url, headers={
        'Accept': 'application/vnd.github+json',
        'User-Agent': 'PC-Free-T4',
    })
    with urllib.request.urlopen(request, timeout=45) as response:
        release = json.load(response)
    tag = release['tag_name']
    print(f'Release: {tag}')
    deb_asset = None
    for asset in release.get('assets', []):
        name = asset.get('name', '')
        if name.endswith('.deb') and 'amd64' in name:
            deb_asset = asset
            break
    if deb_asset is None:
        raise RuntimeError(f'No se encontró un .deb amd64 en el release {tag}.')
    deb_url = deb_asset['browser_download_url']
    if not deb_url.startswith('https://github.com/Heroic-Games-Launcher/HeroicGamesLauncher/releases/download/'):
        raise RuntimeError('El enlace del .deb no pertenece al repositorio oficial.')

    deb_path = BASE_HEROIC / deb_asset['name']
    if not deb_path.exists():
        print(f"Descargando: {deb_asset['name']}")
        temporary = deb_path.with_suffix('.deb.part')
        urllib.request.urlretrieve(deb_url, temporary)
        digest = deb_asset.get('digest') or ''
        if digest.startswith('sha256:'):
            actual = hashlib.sha256(temporary.read_bytes()).hexdigest()
            expected = digest.split(':', 1)[1].lower()
            if actual.lower() != expected:
                temporary.unlink(missing_ok=True)
                raise RuntimeError('SHA-256 del .deb de Heroic no coincide.')
            print('SHA-256 verificado.')
        else:
            print('Aviso: GitHub no proporcionó SHA-256; se omite la verificación de hash.')
        temporary.replace(deb_path)
    else:
        print(f'Ya descargado: {deb_path.name}')

    meta_raw = subprocess.check_output(['dpkg-deb', '-f', str(deb_path)], text=True)
    meta = dict(line.split(': ', 1) for line in meta_raw.splitlines() if ': ' in line)
    if meta.get('Package') != 'heroic':
        raise RuntimeError(f"Paquete inesperado: {meta.get('Package')!r}; se esperaba 'heroic'.")
    if meta.get('Architecture') != 'amd64':
        raise RuntimeError(f"Arquitectura inesperada: {meta.get('Architecture')!r}.")
    print(f"Paquete verificado: {meta['Package']} {meta['Version']} ({meta['Architecture']})")

    plan = subprocess.run(['apt-get', '-s', 'install', '-y', str(deb_path)],
                          text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=120)
    for line in plan.stdout.splitlines():
        if line.startswith('Remv '):
            raise RuntimeError('APT propone eliminar paquetes; instalación cancelada.\n' + line)
        if line.startswith('Inst '):
            pkg = line.split()[1].split(':')[0]
            if 'nvidia' in pkg or pkg.startswith(('linux-image', 'linux-modules', 'linux-headers')):
                raise RuntimeError(f'APT propone cambiar {pkg}; instalación cancelada.\n' + line)

    print('Instalando Heroic Games Launcher…')
    with log.open('a') as stream:
        result = subprocess.run(['apt-get', 'install', '-y', str(deb_path)],
                                stdout=stream, stderr=subprocess.STDOUT,
                                env={**os.environ, 'DEBIAN_FRONTEND': 'noninteractive'},
                                timeout=900)
    if result.returncode:
        raise RuntimeError(f'Falló la instalación de Heroic; revisar {log}')
    print('Heroic instalado.')

    def chown_tree(path):
        for directory, directories, files in os.walk(path):
            os.chown(directory, ACCOUNT.pw_uid, ACCOUNT.pw_gid)
            for name in directories + files:
                os.chown(Path(directory) / name, ACCOUNT.pw_uid, ACCOUNT.pw_gid,
                         follow_symlinks=False)

    HEROIC_TOOLS.mkdir(parents=True, exist_ok=True)
    PROTON_DIR = HEROIC_TOOLS / 'proton'
    PROTON_DIR.mkdir(parents=True, exist_ok=True)
    for directory in (HOME / '.config', HOME / '.config' / 'heroic', HEROIC_TOOLS, PROTON_DIR):
        os.chown(directory, ACCOUNT.pw_uid, ACCOUNT.pw_gid)
    chown_tree(HEROIC_TOOLS)

    if INSTALL_HEROIC_PROTON:
        target = PROTON_DIR / GE9_VERSION
        proton = target / 'proton'
        if proton.is_file():
            print(f'{GE9_VERSION}: ya instalado.')
        else:
            cache = HOME / '.cache' / 'pcfree' / 'heroic'
            cache.mkdir(parents=True, exist_ok=True)
            archive = cache / f'{GE9_VERSION}.tar.gz'
            if not archive.is_file():
                print(f'Descargando {GE9_VERSION} (unos 489 MB)…')
                with urllib.request.urlopen(GE9_URL, timeout=90) as response, archive.open('wb') as out:
                    shutil.copyfileobj(response, out)
            digest = hashlib.sha512()
            with archive.open('rb') as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b''):
                    digest.update(block)
            if digest.hexdigest() != GE9_SHA512:
                archive.unlink(missing_ok=True)
                raise RuntimeError('SHA-512 de GE-Proton9-27 no coincide.')
            staging = PROTON_DIR / (GE9_VERSION + '.tmp')
            if staging.exists():
                shutil.rmtree(staging)
            staging.mkdir(parents=True)
            with tarfile.open(archive) as bundle:
                members = bundle.getmembers()
                for member in members:
                    path = Path(member.name)
                    if path.is_absolute() or '..' in path.parts:
                        raise RuntimeError('Entrada no segura en el archivo de Proton.')
                    if member.issym() or member.islnk():
                        link = member.linkname
                        if link.startswith('/'):
                            raise RuntimeError('Enlace no seguro en el archivo de Proton.')
                        if member.issym():
                            resolved = os.path.normpath(os.path.join(os.path.dirname(member.name), link))
                        else:
                            resolved = os.path.normpath(link)
                        if resolved == '..' or resolved.startswith('../') or resolved.startswith('/'):
                            raise RuntimeError('Enlace no seguro en el archivo de Proton.')
                bundle.extractall(staging)
            roots = [item for item in staging.iterdir() if item.is_dir()]
            if len(roots) != 1:
                shutil.rmtree(staging, ignore_errors=True)
                raise RuntimeError('El archivo de Proton no tiene una única carpeta raíz.')
            if target.exists():
                shutil.rmtree(target)
            roots[0].rename(target)
            shutil.rmtree(staging, ignore_errors=True)
            proton.chmod(0o755)
            print(f'{GE9_VERSION}: instalado en {target}')
        chown_tree(PROTON_DIR)

    wine_version = ''
    if Path(WINE_BIN).is_file():
        wine_version = subprocess.check_output([WINE_BIN, '--version'], text=True,
                                                stderr=subprocess.STDOUT).strip()
        print(f'Wine del sistema: {wine_version}')
    else:
        print('Aviso: Wine no quedó instalado; Heroic solo mostrará runners descargados.')

    if HEROIC_CONFIG.is_file():
        try:
            config = json.loads(HEROIC_CONFIG.read_text())
            settings = config.setdefault('defaultSettings', {})
            current = settings.get('wineVersion') or {}
            if not current.get('bin') or 'Not Found' in str(current.get('name', '')):
                settings['wineVersion'] = {
                    'bin': WINE_BIN,
                    'name': wine_version or 'Wine 9',
                    'type': 'wine',
                    'wineserver': WINE_SERVER,
                }
            settings['autoInstallDxvk'] = True
            settings['autoInstallVkd3d'] = True
            settings['autoInstallDxvkNvapi'] = True
            settings['showValveProton'] = True
            HEROIC_CONFIG.parent.mkdir(parents=True, exist_ok=True)
            os.chown(HEROIC_CONFIG.parent, ACCOUNT.pw_uid, ACCOUNT.pw_gid)
            HEROIC_CONFIG.write_text(json.dumps(config, indent=2))
            os.chown(HEROIC_CONFIG, ACCOUNT.pw_uid, ACCOUNT.pw_gid)
            print('Configuración de Heroic actualizada: Wine del sistema y DXVK/VKD3D automáticos.')
        except Exception as exc:
            print(f'Aviso: no se pudo actualizar la configuración de Heroic: {exc}')
    else:
        print('Heroic creará su configuración en el primer inicio y detectará Wine 9.')

    user = SESSION_HEROIC.get('user', 'pcfree-browser')
    home = Path(SESSION_HEROIC.get('home', '/tmp/pcfree-selkies/home'))
    account = pwd.getpwnam(user)
    runner = SESSION_HEROIC.get('run_launcher', '/usr/local/bin/pcfree-run')
    desktop_dir = home / 'Desktop'
    desktop_dir.mkdir(parents=True, exist_ok=True)

    def quote_desktop(value):
        return '"' + str(value).replace('\\', '\\\\').replace('"', '\\"').replace('`', '\\`').replace('$', '\\$') + '"'

    shortcut = desktop_dir / 'Heroic.desktop'
    shortcut.write_text(
        '[Desktop Entry]\n'
        'Type=Application\n'
        'Version=1.0\n'
        'Name=Heroic Games Launcher\n'
        'Comment=Epic Games, GOG y Amazon Prime Gaming\n'
        f'Exec={quote_desktop(runner)} /usr/bin/heroic\n'
        'Icon=heroic\n'
        'Terminal=false\n'
        'Categories=Game;\n'
    )
    shortcut.chmod(0o755)
    os.chown(shortcut, account.pw_uid, account.pw_gid)

    env = dict(SESSION_HEROIC['env'], PATH='/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin')
    try:
        subprocess.run(['gio', 'set', str(shortcut), 'metadata::trusted', 'true'],
                       env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=20)
    except (FileNotFoundError, subprocess.SubprocessError):
        pass

    heroic_bin = shutil.which('heroic') or '(no encontrado)'
    print('\n=== Verificación ===')
    print(f'Binario: {heroic_bin}')
    print(f'Acceso directo: {shortcut}')
    print(f'Versión: {meta["Version"]}')
    print(f'Wine: {wine_version or "no instalado"}')
    print(f'Proton Heroic: {PROTON_DIR / GE9_VERSION if INSTALL_HEROIC_PROTON else "omitido"}')
    print(f'Registro: {log}')
    print('En Heroic: Ajustes > Wine > seleccionar Wine del sistema o GE-Proton9-27.')


## Proton y 7-Zip

Instala `pcfree-nvidia-run` y File Roller. No reinicia el escritorio ni el túnel.

Opción de lanzamiento en Steam:

```text
pcfree-nvidia-run %command%
```


In [ ]:
# @title Instalar compatibilidad conservadora Proton/Vulkan + 7-Zip gráfico
import os, pwd, shutil, subprocess
from pathlib import Path

if not isinstance(globals().get('SESSION'), dict):
    raise RuntimeError("Requiere haber ejecutado 'Iniciar PC-Free-T4 (Selkies + Cloudflare HTTP/2)'.")
if os.geteuid() != 0:
    raise RuntimeError('Instala paquetes del sistema; requiere Colab como root.')

BASE_COMPAT = Path('/tmp/pcfree-selkies')
SESSION_COMPAT = SESSION

# ---- Wrapper Proton/DXVK: conserva las variables del proceso Steam/Proton ----
launcher = Path('/usr/local/bin/pcfree-nvidia-run')
launcher_text = r'''#!/usr/bin/env bash
set -Eeuo pipefail
BASE="/tmp/pcfree-selkies"
if [[ $# -eq 0 ]]; then
  echo "Uso: pcfree-nvidia-run programa [argumentos...]" >&2
  exit 2
fi
if [[ -z "${DISPLAY:-}" ]] && [[ -S /tmp/.X11-unix/X98 ]]; then
  export DISPLAY=:98
fi
if [[ -z "${XAUTHORITY:-}" ]] && [[ -r "$BASE/.Xauthority" ]]; then
  export XAUTHORITY="$BASE/.Xauthority"
fi
if [[ -z "${XDG_RUNTIME_DIR:-}" ]] && [[ -d "$BASE/runtime" ]]; then
  export XDG_RUNTIME_DIR="$BASE/runtime"
fi
# Preferir el manifiesto privado que preparó el núcleo. Respetar cualquier ICD
# que Steam/pressure-vessel ya haya seleccionado en el proceso.
PRIVATE_ICD="$BASE/nvidia-vulkan64.json"
if [[ -z "${VK_DRIVER_FILES:-}" ]] && [[ -r "$PRIVATE_ICD" ]]; then
  export VK_DRIVER_FILES="$PRIVATE_ICD"
fi
if [[ -z "${VK_ICD_FILENAMES:-}" ]]; then
  if [[ -n "${VK_DRIVER_FILES:-}" ]]; then
    export VK_ICD_FILENAMES="$VK_DRIVER_FILES"
  elif [[ -r "$PRIVATE_ICD" ]]; then
    export VK_ICD_FILENAMES="$PRIVATE_ICD"
  fi
fi
export LIBGL_ALWAYS_SOFTWARE=0
# No establecer LD_LIBRARY_PATH o LD_PRELOAD aquí: Proton y Steam conservan los suyos.
exec "$@"
'''
if launcher.exists() and launcher.read_text(errors='replace') != launcher_text:
    backup = Path('/usr/local/bin/pcfree-nvidia-run.bak-before-pcfree-update')
    if not backup.exists():
        shutil.copy2(launcher, backup)
    print('Se hizo copia de seguridad del lanzador anterior:', backup)
launcher.write_text(launcher_text)
launcher.chmod(0o755)
print('Lanzador preparado:', launcher)
print('No se forzó LD_LIBRARY_PATH ni LD_PRELOAD.')

# ---- Instalar 7-Zip + aplicación gráfica sin permitir reemplazos del driver/kernel ----
subprocess.run(['apt-get', 'update', '-qq'], check=True, timeout=600)

def package_available(name):
    p = subprocess.run(['apt-cache', 'show', name], text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, timeout=30)
    return any(line.strip() == f'Package: {name}' for line in p.stdout.splitlines())

archive_package = next((name for name in ('7zip', 'p7zip-full') if package_available(name)), None)
if not archive_package:
    raise RuntimeError('No se encontró el paquete 7zip ni p7zip-full en los repositorios actuales.')
if not package_available('file-roller'):
    raise RuntimeError('No se encontró file-roller en los repositorios actuales.')
packages = [archive_package, 'file-roller']
plan = subprocess.run(['apt-get', '-s', 'install', '-y', '--no-install-recommends', *packages],
                      text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=180)
if plan.returncode:
    raise RuntimeError('APT no pudo simular la instalación segura de 7-Zip/File Roller:\n' + plan.stdout[-3500:])
for line in plan.stdout.splitlines():
    if line.startswith('Remv '):
        raise RuntimeError('APT propone eliminar paquetes; instalación cancelada.\n' + line)
    if line.startswith('Inst '):
        package = line.split()[1].split(':', 1)[0].lower()
        if package.startswith(('nvidia', 'linux-image', 'linux-modules', 'linux-headers')):
            raise RuntimeError('APT propone cambiar componentes NVIDIA/kernel; instalación cancelada.\n' + line)
print('Instalando:', ', '.join(packages))
subprocess.run(['apt-get', 'install', '-y', '--no-install-recommends', *packages], check=True, timeout=900)

# Acceso directo propiedad de la cuenta gráfico; la aplicación se inicia dentro de su entorno.
user = SESSION_COMPAT.get('user', 'pcfree-browser')
home = Path(SESSION_COMPAT.get('home', '/tmp/pcfree-selkies/home'))
account = pwd.getpwnam(user)
desktop_dir = home / 'Desktop'
desktop_dir.mkdir(parents=True, exist_ok=True)
shortcut = desktop_dir / 'Archivos-comprimidos.desktop'
runner = SESSION_COMPAT.get('run_launcher', '/usr/local/bin/pcfree-run')

def quote_desktop(value):
    return '"' + str(value).replace('\\', '\\\\').replace('"', '\\"').replace('`', '\\`').replace('$', '\\$') + '"'

shortcut.write_text(
    '[Desktop Entry]\nType=Application\nVersion=1.0\n'
    'Name=Archivos comprimidos\nComment=Abrir y extraer archivos 7-Zip\n'
    f'Exec={quote_desktop(runner)} {quote_desktop("/usr/bin/file-roller")}\n'
    'Icon=application-x-archive\nTerminal=false\nCategories=Utility;Archiving;\n'
)
shortcut.chmod(0o755)
os.chown(shortcut, account.pw_uid, account.pw_gid)

print('\n=== Verificación ===')
print('7-Zip:', shutil.which('7zz') or shutil.which('7z') or '(no se encontró el binario esperado)')
print('File Roller:', shutil.which('file-roller') or '(no se encontró el ejecutable esperado)')
print('Acceso directo:', shortcut)
if Path('/tmp/pcfree-selkies/nvidia-vulkan64.json').is_file():
    print('ICD NVIDIA privado: encontrado')
else:
    print('Aviso: no se encontró el ICD privado; Vulkan se mantiene como capacidad por verificar.')
print('\nEn Steam usa esta opción de lanzamiento: pcfree-nvidia-run %command%')
print('No se cambió la configuración de lanzamiento de ningún juego ni se inició Steam automáticamente.')

In [ ]:

# @title Instalar GE-Proton11-11 (bajo demanda)
# @markdown Descarga bajo demanda de 564 MB, con SHA-512 oficial, en compatibilitytools.d.

INSTALL_GE_PROTON = False  # @param {type:"boolean"}

if not isinstance(globals().get('SESSION'), dict):
    raise RuntimeError("Requiere haber ejecutado 'Iniciar PC-Free-T4'.")
if not callable(globals().get('optional_module')):
    raise RuntimeError("Falta 'Cargar núcleo interno'.")
if not INSTALL_GE_PROTON:
    print('GE-Proton omitido. Activar INSTALL_GE_PROTON para instalar.')
else:
    module = optional_module('compat-steam.py')
    result = module['install_ge_proton'](SESSION)
    print('Ruta:', result['path'])
    print('Selección global: Steam > Configuración > Compatibilidad > Steam Play para todos los títulos.')


In [ ]:

# @title Pruebas de compatibilidad de Proton (32/64 bits)
# @markdown Verifica OpenGL y Vulkan de 32 y 64 bits y crea prefijos Proton.
# @markdown Requiere GE-Proton instalado.

import os, shutil, subprocess
from pathlib import Path

BASE = Path('/tmp/pcfree-selkies')
if not isinstance(globals().get('SESSION'), dict):
    raise RuntimeError("Requiere haber ejecutado 'Iniciar PC-Free-T4'.")

home = Path(SESSION['home'])
ge = home / '.steam' / 'steam' / 'compatibilitytools.d' / 'GE-Proton11-11'
proton = ge / 'proton'
base = Path('/tmp/pcfree-selkies/probe')
base.mkdir(parents=True, exist_ok=True)

probe_c = r'''#define _GNU_SOURCE
#include <X11/Xlib.h>
#include <GL/gl.h>
#include <GL/glx.h>
#include <dlfcn.h>
#include <stdint.h>
#include <stdio.h>
typedef struct { int sType; const void *pNext; const char *pApplicationName;
  uint32_t applicationVersion; const char *pEngineName; uint32_t engineVersion;
  uint32_t apiVersion; } VkApplicationInfo;
typedef struct { int sType; const void *pNext; uint32_t flags;
  const VkApplicationInfo *pApplicationInfo; uint32_t enabledLayerCount;
  const char *const *ppEnabledLayerNames; uint32_t enabledExtensionCount;
  const char *const *ppEnabledExtensionNames; } VkInstanceCreateInfo;
typedef int (*PFN_vkCreateInstance)(const VkInstanceCreateInfo*, const void*, void**);
static int load(const char *name) { void *h = dlopen(name, RTLD_NOW | RTLD_LOCAL); if (h) dlclose(h); return h != NULL; }
int main(void) {
  int gl = load("libGL.so.1"), egl = load("libEGL.so.1"), vk = load("libvulkan.so.1"), nv = load("libGLX_nvidia.so.0");
  int display = 0, glx = 0; const char *renderer = "none";
  Display *d = XOpenDisplay(NULL);
  if (d) {
    display = 1; int major = 0, minor = 0; glx = glXQueryVersion(d, &major, &minor);
    int attrs[] = {GLX_RGBA, GLX_DOUBLEBUFFER, None};
    XVisualInfo *vi = glXChooseVisual(d, 0, attrs);
    if (vi) {
      GLXContext ctx = glXCreateContext(d, vi, NULL, True);
      Window root = DefaultRootWindow(d);
      if (ctx && glXMakeCurrent(d, root, ctx)) {
        const GLubyte *name = glGetString(GL_RENDERER);
        if (name) renderer = (const char *)name;
      }
    }
  }
  printf("gl=%d egl=%d vulkan=%d nvidia=%d display=%d glx=%d renderer=%s", gl, egl, vk, nv, display, glx, renderer);
  void *v = dlopen("libvulkan.so.1", RTLD_NOW | RTLD_LOCAL);
  if (v) {
    PFN_vkCreateInstance create = (PFN_vkCreateInstance)dlsym(v, "vkCreateInstance");
    if (create) {
      VkApplicationInfo info = {0, 0, "pcfree", 1, "pcfree", 1, 0x00400000};
      VkInstanceCreateInfo ci = {1, 0, 0, &info, 0, 0, 0, 0};
      void *instance = NULL;
      printf(" create=%d", create(&ci, NULL, &instance));
    } else printf(" create=missing");
    dlclose(v);
  }
  printf("
");
  return 0;
}
'''

env = {**os.environ, **SESSION.get('env', {})}
print('=== SONDA 64/32 BITS ===')
for arch, flag in ((64, '-m64'), (32, '-m32')):
    src = base / f'probe{arch}.c'
    exe = base / f'probe{arch}'
    src.write_text(probe_c)
    build = subprocess.run(['gcc', flag, '-O2', '-o', str(exe), str(src), '-lGL', '-lX11', '-ldl'],
                           text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           timeout=60)
    if build.returncode:
        print(f'{arch} bits: build FAIL: {build.stdout.strip()[-300:]}')
        continue
    run = subprocess.run([str(exe)], env=env, text=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, timeout=30)
    print(f'{arch} bits:', (run.stdout or run.stderr).strip())

print('=== PROTON ===')
if not proton.is_file():
    print('GE-Proton no instalado. Ejecuta la celda de GE-Proton antes de esta prueba.')
else:
    for label, extra in (('64', {}), ('32', {'WINEARCH': 'win32'})):
        prefix = base / f'prefix-{label}'
        if prefix.exists():
            shutil.rmtree(prefix)
        prefix.mkdir(parents=True)
        penv = dict(env, **extra)
        penv.update({
            'STEAM_COMPAT_CLIENT_INSTALL_PATH': str(home / '.steam' / 'steam'),
            'STEAM_COMPAT_INSTALL_PATH': str(home / '.steam' / 'steam'),
            'STEAM_COMPAT_DATA_PATH': str(prefix),
            'STEAM_COMPAT_APP_ID': '0', 'SteamAppId': '0', 'SteamGameId': '0',
        })
        run = subprocess.run([str(proton), 'run', 'cmd', '/c', 'ver'], cwd=str(prefix),
                             env=penv, text=True, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, timeout=240)
        tail = ' | '.join((run.stdout or '').strip().splitlines()[-2:])
        print(f'{label} bits: rc={run.returncode} {tail[:300]}')


In [ ]:

# @title Instalador de .deb, .AppImage, .tar y .rpm (manejador predeterminado)
# @markdown Añade un acceso "Descargar aplicación" al escritorio y registra
# @markdown el instalador para .deb, .AppImage, .tar, .tar.gz y .rpm.
# @markdown Acepta una URL HTTPS o un archivo local; crea accesos en el escritorio.

import os, pwd, shutil, subprocess
from pathlib import Path

if not isinstance(globals().get('SESSION'), dict):
    raise RuntimeError("Requiere haber ejecutado 'Iniciar PC-Free-T4'.")

user = SESSION['user']
home = Path(SESSION['home'])
account = pwd.getpwnam(user)
helper = Path('/usr/local/bin/pcfree-app-installer')
helper.parent.mkdir(parents=True, exist_ok=True)
helper.write_text(r'''#!/usr/bin/python3.12
"""Descarga e instala archivos .deb y .AppImage, y crea accesos de escritorio."""
import argparse
import os
import pwd
import re
import shlex
import shutil
import tarfile
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request
from pathlib import Path

USER = pwd.getpwuid(os.getuid()).pw_name
HOME = Path(os.environ.get('HOME', pwd.getpwnam(USER).pw_dir))
DOWNLOADS = HOME / 'Downloads'
APPS = HOME / '.local' / 'bin' / 'apps'
ICONS = HOME / '.local' / 'share' / 'icons' / 'pcfree'
APPLICATIONS = HOME / '.local' / 'share' / 'applications'
DESKTOP = HOME / 'Desktop'
DEB_HELPER = Path('/usr/local/sbin/pcfree-install-deb')
RUNNER = '/usr/local/bin/pcfree-run'
PATH = '/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin'
MIME_TYPE = 'application/x-appimage'
DEB_TYPE = 'application/vnd.debian.binary-package'
HANDLED = (DEB_TYPE, 'application/x-deb', MIME_TYPE, 'application/x-rpm',
           'application/x-tar', 'application/x-compressed-tar', 'application/gzip', 'application/x-xz')


def notify(title, text, kind='info'):
    if shutil.which('zenity'):
        subprocess.run(['zenity', '--' + kind, '--title=' + title, '--text=' + text],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    else:
        print(title + ': ' + text, file=sys.stderr)


def ensure_dirs():
    for directory in (DOWNLOADS, APPS, ICONS, APPLICATIONS, DESKTOP):
        directory.mkdir(parents=True, exist_ok=True)


def quote(value):
    return '"' + str(value).replace('\\', '\\\\').replace('"', '\\"').replace('$', '\\$') + '"'


def ask_url():
    if shutil.which('zenity'):
        result = subprocess.run(['zenity', '--entry', '--title=Descargar aplicación',
                                 '--text=URL HTTPS del archivo .deb, .AppImage, .tar, .tar.gz o .rpm:'],
                                text=True, capture_output=True)
        if result.returncode == 0:
            return result.stdout.strip()
        return ''
    return input('URL HTTPS: ').strip()


def ask_source():
    if not shutil.which('zenity'):
        return input('Archivo local o URL HTTPS: ').strip()
    choice = subprocess.run(['zenity', '--list', '--title=Instalar aplicación',
                             '--column=Acción', 'Descargar por URL', 'Instalar archivo local'],
                            text=True, capture_output=True)
    if choice.returncode:
        return ''
    if 'URL' in choice.stdout:
        return ask_url()
    selected = subprocess.run(['zenity', '--file-selection', '--title=Seleccionar aplicación'],
                              text=True, capture_output=True)
    return selected.stdout.strip() if selected.returncode == 0 else ''


def download(url, target):
    parsed = urllib.parse.urlparse(url)
    if parsed.scheme != 'https':
        raise RuntimeError('La descarga debe usar HTTPS.')
    request = urllib.request.Request(url, headers={'User-Agent': 'PC-Free-T4'})
    temporary = target.with_suffix(target.suffix + '.part')
    progress = None
    try:
        with urllib.request.urlopen(request, timeout=90) as response, temporary.open('wb') as out:
            if urllib.parse.urlparse(response.url).scheme != 'https':
                raise RuntimeError('La redirección debe usar HTTPS.')
            length = int(response.headers.get('Content-Length') or 0)
            if length > 4 * 1024 ** 3:
                raise RuntimeError('El archivo supera 4 GiB.')
            if shutil.which('zenity'):
                command = ['zenity', '--progress', '--auto-close', '--title=Descargando',
                           '--text=' + target.name]
                if not length:
                    command.append('--pulsate')
                progress = subprocess.Popen(command, stdin=subprocess.PIPE,
                                            text=True, stdout=subprocess.DEVNULL,
                                            stderr=subprocess.DEVNULL)
            copied = 0
            while True:
                block = response.read(256 * 1024)
                if not block:
                    break
                out.write(block)
                copied += len(block)
                if progress and length:
                    try:
                        progress.stdin.write(f'{int(copied * 100 / length)}\n')
                        progress.stdin.flush()
                    except (BrokenPipeError, ValueError):
                        progress = None
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)
        if progress:
            try:
                progress.stdin.close()
            except Exception:
                pass
            progress.wait(timeout=10)


def safe_name(url):
    name = os.path.basename(urllib.parse.unquote(urllib.parse.urlparse(url).path))
    name = re.sub(r'[^A-Za-z0-9._-]', '_', name)
    suffixes = ('.deb', '.appimage', '.rpm', '.tar', '.tar.gz', '.tgz', '.tar.xz', '.tar.bz2')
    if not name.lower().endswith(suffixes):
        raise RuntimeError('El archivo debe ser .deb, .AppImage, .tar, .tar.gz, .tgz, .tar.xz, .tar.bz2 o .rpm.')
    return name or 'aplicacion.appimage'


def trusted(path):
    if shutil.which('gio'):
        subprocess.run(['gio', 'set', str(path), 'metadata::trusted', 'true'],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def create_deb_shortcut(package):
    listed = subprocess.run(['dpkg', '-L', package], text=True, capture_output=True)
    candidates = [line for line in (listed.stdout or '').splitlines()
                  if line.endswith('.desktop') and line.startswith('/usr/share/applications/')]
    if not candidates:
        return None
    source = Path(candidates[0])
    text = source.read_text(errors='replace')
    name = re.search(r'^Name=(.+)$', text, re.M)
    executable = re.search(r'^Exec=(.+)$', text, re.M)
    if not executable:
        return None
    parts = [part for part in shlex.split(executable.group(1)) if not part.startswith('%')]
    command = ' '.join(quote(part) for part in [RUNNER, *parts])
    text = re.sub(r'^Exec=.*$', 'Exec=' + command, text, flags=re.M)
    label = name.group(1).strip() if name else package
    destination = DESKTOP / (re.sub(r'[^A-Za-z0-9._-]', '_', label) + '.desktop')
    destination.write_text(text)
    destination.chmod(0o755)
    trusted(destination)
    return destination


def install_deb(path):
    if not DEB_HELPER.is_file():
        raise RuntimeError('Falta el instalador de DEB del escritorio.')
    package = subprocess.check_output(['dpkg-deb', '-f', str(path), 'Package'], text=True).strip()
    result = subprocess.run(['sudo', '-n', str(DEB_HELPER), str(path)],
                            text=True, capture_output=True, timeout=600)
    if result.returncode:
        raise RuntimeError((result.stdout or result.stderr or 'La instalación falló.')[-1500:])
    return create_deb_shortcut(package)


def install_appimage(path):
    slug = re.sub(r'[^A-Za-z0-9._-]', '_', path.stem)
    destination = APPS / (slug + '.AppImage')
    shutil.copy2(path, destination)
    destination.chmod(0o755)
    if path.resolve().parent in (DOWNLOADS.resolve(), DESKTOP.resolve()):
        path.unlink(missing_ok=True)
    cache = HOME / '.cache'
    cache.mkdir(parents=True, exist_ok=True)
    extract = Path(tempfile.mkdtemp(prefix='appimage-', dir=cache))
    name = path.stem
    icon = 'application-x-executable'
    categories = 'Utility;'
    root = extract / 'squashfs-root'
    try:
        environment = dict(os.environ, APPIMAGE_EXTRACT_AND_RUN='1')
        subprocess.run([str(destination), '--appimage-extract'], cwd=extract,
                       env=environment, stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL, timeout=120)
        root = extract / 'squashfs-root'
        desktop = next(root.glob('*.desktop'), None)
        if desktop:
            text = desktop.read_text(errors='replace')
            found = re.search(r'^Name=(.+)$', text, re.M)
            if found:
                name = found.group(1).strip()
            found = re.search(r'^Categories=(.+)$', text, re.M)
            if found:
                categories = found.group(1).strip()
            found = re.search(r'^Icon=(.+)$', text, re.M)
            if found:
                wanted = found.group(1).strip()
                candidates = []
                if Path(wanted).is_absolute() and Path(wanted).is_file():
                    candidates = [Path(wanted)]
                else:
                    wanted = wanted.lstrip('/') if wanted.startswith('/') else wanted
                    candidates = (list(root.rglob(wanted + '.png')) +
                                  list(root.rglob(wanted + '.svg')))
                if candidates:
                    suffix = candidates[0].suffix.lower()
                    copied = ICONS / (slug + suffix)
                    shutil.copy2(candidates[0], copied)
                    icon = str(copied)
    except Exception:
        pass
    finally:
        shutil.rmtree(extract, ignore_errors=True)
    if icon == 'application-x-executable':
        icons = list(root.rglob('*.png')) + list(root.rglob('*.svg'))
        if icons:
            copied = ICONS / (slug + icons[0].suffix.lower())
            shutil.copy2(icons[0], copied)
            icon = str(copied)
    command = RUNNER + ' env APPIMAGE_EXTRACT_AND_RUN=1 ' + quote(destination)
    entry = ('[Desktop Entry]\nType=Application\nName=' + name + '\n'
             'Comment=Aplicación AppImage\nExec=' + command + ' %U\n'
             'Icon=' + icon + '\nTerminal=false\nCategories=' + categories + '\n')
    shortcut = DESKTOP / (re.sub(r'[^A-Za-z0-9._-]', '_', name) + '.desktop')
    shortcut.write_text(entry)
    shortcut.chmod(0o755)
    menu = APPLICATIONS / ('pcfree-' + slug + '.desktop')
    menu.write_text(entry)
    menu.chmod(0o755)
    trusted(shortcut)
    if shutil.which('update-desktop-database'):
        subprocess.run(['update-desktop-database', str(APPLICATIONS)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return shortcut


def archive_slug(path):
    name = path.name
    for suffix in ('.tar.gz', '.tgz', '.tar.xz', '.tar.bz2', '.tar', '.rpm'):
        if name.lower().endswith(suffix):
            return re.sub(r'[^A-Za-z0-9._-]', '_', name[:-len(suffix)])
    return re.sub(r'[^A-Za-z0-9._-]', '_', path.stem)


def create_tree_shortcut(root, slug):
    desktop = next(root.rglob('*.desktop'), None)
    name = slug
    icon = 'application-x-executable'
    categories = 'Utility;'
    executable = None
    if desktop:
        text = desktop.read_text(errors='replace')
        found = re.search(r'^Name=(.+)$', text, re.M)
        if found:
            name = found.group(1).strip()
        found = re.search(r'^Categories=(.+)$', text, re.M)
        if found:
            categories = found.group(1).strip()
        found = re.search(r'^Icon=(.+)$', text, re.M)
        if found:
            wanted = found.group(1).strip()
            candidates = []
            if Path(wanted).is_absolute() and Path(wanted).is_file():
                candidates = [Path(wanted)]
            else:
                candidates = (list(root.rglob(wanted + '.png')) +
                              list(root.rglob(wanted + '.svg')))
            if candidates:
                copied = ICONS / (slug + candidates[0].suffix.lower())
                shutil.copy2(candidates[0], copied)
                icon = str(copied)
        found = re.search(r'^Exec=(.+)$', text, re.M)
        if found:
            parts = [part for part in shlex.split(found.group(1)) if not part.startswith('%')]
            if parts:
                candidate = Path(parts[0])
                if candidate.is_file():
                    executable = candidate
                else:
                    pattern = parts[0].lstrip('/') if parts[0].startswith('/') else parts[0]
                    matches = list(root.rglob(pattern))
                    if matches:
                        executable = matches[0]
    if executable is None:
        candidates = [item for item in root.rglob('*')
                      if item.is_file() and os.access(item, os.X_OK)
                      and not item.name.endswith(('.so', '.sh'))]
        preferred = [item for item in candidates if slug.lower() in item.name.lower()]
        executable = (preferred or candidates)[0] if (preferred or candidates) else None
    if executable is None:
        raise RuntimeError('No se encontró un ejecutable en el archivo.')
    if icon == 'application-x-executable':
        icons = list(root.rglob('*.png')) + list(root.rglob('*.svg'))
        if icons:
            copied = ICONS / (slug + icons[0].suffix.lower())
            shutil.copy2(icons[0], copied)
            icon = str(copied)
    entry = ('[Desktop Entry]\nType=Application\nName=' + name + '\n'
             'Comment=Aplicación instalada\nExec=' + RUNNER + ' ' + quote(executable) +
             ' %U\nIcon=' + icon + '\nTerminal=false\nCategories=' + categories + '\n')
    shortcut = DESKTOP / (re.sub(r'[^A-Za-z0-9._-]', '_', name) + '.desktop')
    shortcut.write_text(entry)
    shortcut.chmod(0o755)
    menu = APPLICATIONS / ('pcfree-' + slug + '.desktop')
    menu.write_text(entry)
    menu.chmod(0o755)
    trusted(shortcut)
    if shutil.which('update-desktop-database'):
        subprocess.run(['update-desktop-database', str(APPLICATIONS)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return shortcut


def install_tar(path):
    slug = archive_slug(path)
    destination = APPS / slug
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    with tarfile.open(path) as source:
        for member in source.getmembers():
            member_path = Path(member.name)
            if member_path.is_absolute() or '..' in member_path.parts:
                raise RuntimeError('Entrada no segura en el archivo tar.')
            if member.issym() or member.islnk():
                link = member.linkname
                if link.startswith('/'):
                    raise RuntimeError('Enlace no seguro en el archivo tar.')
                if member.issym():
                    resolved = os.path.normpath(os.path.join(os.path.dirname(member.name), link))
                else:
                    resolved = os.path.normpath(link)
                if resolved == '..' or resolved.startswith('../') or resolved.startswith('/'):
                    raise RuntimeError('Enlace no seguro en el archivo tar.')
        source.extractall(destination)
    return create_tree_shortcut(destination, slug)


def install_rpm(path):
    rpm2cpio = shutil.which('rpm2cpio')
    cpio = shutil.which('cpio')
    if not rpm2cpio or not cpio:
        raise RuntimeError('Se requieren rpm2cpio y cpio para instalar RPM.')
    slug = archive_slug(path)
    destination = APPS / slug
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    producer = subprocess.Popen([rpm2cpio, str(path)], stdout=subprocess.PIPE,
                                stderr=subprocess.DEVNULL)
    result = subprocess.run([cpio, '-idm', '--quiet'], cwd=destination,
                            stdin=producer.stdout, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT)
    producer.wait(timeout=60)
    if result.returncode:
        raise RuntimeError((result.stdout or b'').decode(errors='replace')[-1200:])
    return create_tree_shortcut(destination, slug)


def register_defaults():
    ensure_dirs()
    mime_dir = HOME / '.local' / 'share' / 'mime'
    packages = mime_dir / 'packages'
    packages.mkdir(parents=True, exist_ok=True)
    (packages / 'pcfree-appimage.xml').write_text("""<?xml version="1.0" encoding="UTF-8"?>
<mime-info xmlns="http://www.freedesktop.org/standards/shared-mime-info">
<mime-type type="application/x-appimage">
<comment>Aplicación AppImage</comment>
<glob pattern="*.AppImage" weight="80"/>
<glob pattern="*.appimage" weight="80"/>
<magic priority="80"><match value="AI" offset="8"/></magic>
</mime-type>
</mime-info>
""")
    environment = dict(os.environ, PATH=PATH, HOME=str(HOME))
    for command in (['update-mime-database', str(mime_dir)],
                    ['update-desktop-database', str(APPLICATIONS)]):
        if shutil.which(command[0]):
            subprocess.run(command, env=environment, stdout=subprocess.DEVNULL,
                           stderr=subprocess.DEVNULL)
    handler = APPLICATIONS / 'pcfree-app-installer.desktop'
    handler.write_text('[Desktop Entry]\nType=Application\nName=Instalar .deb / .AppImage / .tar / .rpm\n'
                       'Comment=Instalar .deb, .AppImage, .tar, .tar.gz y .rpm\n'
                       'Exec=' + RUNNER + ' /usr/local/bin/pcfree-app-installer %f\n'
                       'Icon=system-software-install\nTerminal=false\n'
                       'MimeType=' + ';'.join(HANDLED) + ';\nCategories=System;\n')
    handler.chmod(0o755)
    launcher = DESKTOP / 'Descargar-app.desktop'
    launcher.write_text('[Desktop Entry]\nType=Application\nName=Descargar aplicación\n'
                        'Comment=Instalar .deb, .AppImage, .tar, .tar.gz o .rpm desde URL o archivo local\n'
                        'Exec=' + RUNNER + ' /usr/local/bin/pcfree-app-installer\n'
                        'Icon=system-software-install\nTerminal=false\nCategories=System;\n')
    launcher.chmod(0o755)
    trusted(launcher)
    if shutil.which('xdg-mime'):
        for mime in HANDLED:
            subprocess.run(['xdg-mime', 'default', 'pcfree-app-installer.desktop', mime],
                           env=environment, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return handler


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('path', nargs='?', help='Archivo local o URL HTTPS')
    parser.add_argument('--register', action='store_true')
    args = parser.parse_args()
    ensure_dirs()
    if args.register:
        register_defaults()
        print('Manejador .deb/.AppImage instalado.')
        return
    value = args.path or ask_source()
    if not value:
        return
    if urllib.parse.urlparse(value).scheme == 'https':
        target = DOWNLOADS / safe_name(value)
        if target.exists():
            target = DOWNLOADS / (target.stem + '-' + str(os.getpid()) + target.suffix)
        download(value, target)
        path = target
    else:
        path = Path(value).expanduser().resolve()
    if not path.is_file():
        raise RuntimeError('No existe el archivo indicado.')
    name = path.name.lower()
    if name.endswith('.deb'):
        shortcut = install_deb(path)
    elif name.endswith('.appimage'):
        shortcut = install_appimage(path)
    elif name.endswith(('.tar', '.tar.gz', '.tgz', '.tar.xz', '.tar.bz2')):
        shortcut = install_tar(path)
    elif name.endswith('.rpm'):
        shortcut = install_rpm(path)
    else:
        raise RuntimeError('Formato no admitido.')
    register_defaults()
    notify('Aplicación instalada', str(shortcut) if shortcut else 'Paquete .deb instalado.')


if __name__ == '__main__':
    try:
        main()
    except Exception as exc:
        notify('No se pudo instalar', str(exc)[-1500:], 'error')
        sys.exit(1)
''')
helper.chmod(0o755)

path = '/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin'
env = dict(HOME=str(home), PATH=path, XDG_RUNTIME_DIR=SESSION.get('runtime', ''),
           DISPLAY=SESSION.get('display', ':98'), XAUTHORITY=SESSION.get('xauthority', ''))
runuser = shutil.which('runuser') or '/usr/sbin/runuser'
try:
    result = subprocess.run(
        [runuser, '-u', user, '--', 'env', *[f'{key}={value}' for key, value in env.items()],
         str(helper), '--register'],
        text=True, capture_output=True, timeout=60)
    if result.returncode:
        print('Aviso al registrar el manejador:', (result.stdout or result.stderr).strip())
    else:
        print('Manejador .deb/.AppImage registrado.')
except (FileNotFoundError, subprocess.SubprocessError) as exc:
    print('Aviso: no se pudo registrar el manejador:', exc)
print('Acceso de escritorio: Descargar aplicación')
print('Archivos .deb, .AppImage, .tar, .tar.gz y .rpm: doble clic abre el instalador.')
